In [1]:
from pathlib import Path

import warnings
warnings.filterwarnings(action="ignore")

In [2]:
file_path = Path.cwd().parent / "datasets" / "airfrance_report.pdf"
print(file_path)

c:\Users\user\Desktop\LLM-RAG\datasets\airfrance_report.pdf


## PyMuPDFLoader

In [3]:
from langchain_community.document_loaders import PyMuPDFLoader
# from langchain_classic.document_loaders import PDFMinerLoader

In [ ]:
loader = PyMuPDFLoader(file_path=file_path,
                       extract_tables="markdown",
                       mode="page")

c:\Users\user\Desktop\LLM-RAG\datasets\airfrance_report.pdf


In [5]:
for i in (Path.cwd().parent / "datasets").glob("*.pdf"):
    print(i)

c:\Users\user\Desktop\LLM-RAG\datasets\airfrance_report.pdf


In [6]:
import pprint

docs = loader.load()

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


In [7]:
len(docs)

80

In [ ]:
docs[10].metadata["doc_type"] = "none"

{'producer': 'Wdesk Fidelity Content Translations Version 014.002.055',
 'creator': 'Workiva',
 'creationdate': '2025-07-31T13:03:58+00:00',
 'source': 'c:\\Users\\user\\Desktop\\LLM-RAG\\datasets\\airfrance_report.pdf',
 'file_path': 'c:\\Users\\user\\Desktop\\LLM-RAG\\datasets\\airfrance_report.pdf',
 'total_pages': 80,
 'format': 'PDF 1.4',
 'title': '2025 - RFS - vFR',
 'author': 'anonymous',
 'subject': '',
 'keywords': '',
 'moddate': '2025-07-31T13:03:58+00:00',
 'trapped': '',
 'modDate': "D:20250731130358+00'00'",
 'creationDate': "D:20250731130358+00'00'",
 'page': 10,
 'doc_type': 'none'}

In [8]:
# pprint.pp(docs[9].metadata)

## Docs splitting

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(separators=["\n\n", "\n", "||", "|", " ", ""], chunk_size = 1100, chunk_overlap = 300)
texts = text_splitter.split_text(docs[10].page_content)

In [28]:
len(" AIR FRANCE - KLM – RAPPORT FINANCIER SEMESTRIEL AU 30 JUIN 2025")

64

In [10]:
texts

["1.1.3 \nLa flotte\nAu 30 juin 2025, la flotte du groupe Air France-KLM comprend 590 avions, dont 581 avions en exploitation contre \nrespectivement 574 et 564 avions au 31 décembre 2024.\nLa flotte principale (excluant les avions régionaux) en exploitation comprend 485 avions (467 avions au 31 décembre \n2024). Elle se répartit en 186 avions long-courriers (183 au 31 décembre 2024), 6 avions cargo (6 au 31 décembre 2024) et \n293 avions moyen-courriers (278 au 31 décembre 2024) dont 136 avions dans la flotte du groupe Transavia (125 avions au \n31 décembre 2024).\nLa flotte régionale en exploitation comprend 96 avions (97 avions au 31 décembre 2024).\nAu 30 juin 2025, l'âge moyen de la flotte en exploitation est de 12,0 ans, dont 12,9 ans pour la flotte long-courrier, 11,7 ans \npour la flotte moyen-courrier, 22,3 ans pour la flotte cargo et 10,7 ans pour la flotte régionale contre 12,1 ans au 31 \ndécembre 2024, dont 12,7 ans pour la flotte long-courrier, 12,1 ans pour la flotte moy

## LlamaParser

In [18]:
import os
import nest_asyncio
from IPython.display import display, Markdown

nest_asyncio.apply()

# Masquer les warnings pour y voir clair
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

os.environ["LLAMA_CLOUD_API_KEY"] = "llx-raPDmlgjlxAtXvZ7AO5blh3ftqs4VSoDEtMU8xkOn8gmhGh7"
    # "llx-tfyMbsZUvZot2cNyHUbrZXM2mrryOjWAzqoZc9chmOL0JTB6""

from llama_parse import LlamaParse

parser = LlamaParse(
    result_type="markdown",
    num_workers=4,
    ignore_cache=True,
    premium_mode=True,
    verbose=False,
    merge_tables_across_pages_in_markdown=True,
    adaptive_long_table=True,
    compact_markdown_table=True,
    user_prompt="""Tu es un expert en analyse de documents financiers et en redressement de données mal extraites.
        Ta mission principale est d'extraire l'intégralité du contenu de ce document (texte, paragraphes, titres) de manière fidèle et structurée au format Markdown.

        Attention : Le document contient des tableaux financiers. Le texte qui t'est fourni a été extrait par un outil d'OCR/parsing qui détruit souvent la structure tabulaire originale. Tu ne dois PAS te fier à l'alignement brut fourni.

        Lorsque tu identifies des données tabulaires, tu dois impérativement appliquer les règles de correction suivantes avant de générer le tableau Markdown final :

        1. PIÈGE DE LA CELLULE HAUT-GAUCHE : Très souvent, la toute première cellule du tableau (en haut à gauche) est vide dans le document original. Le parseur tente par erreur de la remplir en aspirant le texte de la ligne d'en-dessous, ce qui détruit tout l'en-tête. Tu dois forcer la création d'une cellule d'en-tête vide (ou la nommer logiquement, ex: "Catégorie" ou " ") pour maintenir l'alignement exact des autres colonnes de l'en-tête.
        2. CORRECTION DES EN-TÊTES : Le parser confond souvent la première ligne de données (les labels des lignes, ex: "Flotte principale") et l'en-tête de colonne. Sépare clairement la colonne des catégories (labels) de la ligne des en-têtes de colonnes (périodes, flux).
        3. RÉALIGNEMENT DES VALEURS (LE PLUS IMPORTANT) : À cause de cellules vides dans le document original, le parser a décalé les nombres de manière anarchique (souvent vers la droite, créant de fausses colonnes vides à la fin). Tu dois réassigner chaque nombre à sa colonne logique.
        4. LOGIQUE FINANCIÈRE : Utilise la cohérence mathématique pour replacer les chiffres. Par exemple : [Valeur début de période] + [Augmentations/Livraisons] - [Diminutions/Annulations] = [Valeur fin de période]. Si un chiffre est sous la mauvaise colonne, déplace-le pour que l'équation soit juste.
        5. NETTOYAGE : Supprime les colonnes fantômes (vides) générées par erreur à l'extrémité droite du tableau.

        Produis un bref résumé des informations clés du tableau juste avant de l'afficher. Restituer la version finale, corrigée, recalculée et parfaitement alignée du tableau au format Markdown. Extrais le reste du document normalement."""
)
# CORRECTION : Chemin relatif corrigé pour sortir du sous-dossier notebooks
file_path = Path.cwd().parent / "datasets" / "airfrance_report.pdf"

# Validation de l'existence du fichier avant de lancer le parser
if not os.path.exists(file_path):
    print(f"❌ Erreur : Le fichier est introuvable au chemin : {os.path.abspath(file_path)}")
else:
    parsed_pages = parser.load_data(file_path)
    print(f"✅ Nombre de pages extraites : {len(parsed_pages)}")

    if len(parsed_pages) > 0:
        # Conversion au format standard LangChain
        langchain_docs = [doc.to_langchain_format() for doc in parsed_pages]
        
        # Extraction de la page index 10
        test_page = langchain_docs[10]
        
        print("\n--- RENDU VISUEL DU TABLEAU EN MARKDOWN ---")
        display(Markdown(test_page.page_content))
    else:
        print("⚠️ Le parser n'a retourné aucune page.")

✅ Nombre de pages extraites : 1


IndexError: list index out of range

In [47]:
parsed_pages

[Document(id_='2d97717b-29fd-454f-b63b-b7773676c8a0', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='# RAPPORT FINANCIER SEMESTRIEL 2025\n\n----\n\n**GROUPE AIR FRANCE-KLM**\n# Sommaire\n\nCe tableau présente la structure du rapport financier semestriel d\'Air France-KLM au 30 juin 2025, divisé en trois sections principales : l\'activité semestrielle, les états financiers et les informations de contrôle.\n\n|Section|Page|\n|-|-|\n|Rapport semestriel d’activité|3|\n|Rapport financier|34|\n|Information et contrôle|77|\n\n\n# Sommaire\n\n# 1.\n\n----\n\nCe tableau présente le sommaire détaillé de la première partie du rapport financier, intitulée "Rapport semestriel d\'activité", incluant les sections sur l\'activité, la flotte, les faits marquants et le gouvernement d\'entreprise, avec leurs numéros de p

In [48]:
langchain_docs

[Document(id='2d97717b-29fd-454f-b63b-b7773676c8a0', metadata={}, page_content='# RAPPORT FINANCIER SEMESTRIEL 2025\n\n----\n\n**GROUPE AIR FRANCE-KLM**\n# Sommaire\n\nCe tableau présente la structure du rapport financier semestriel d\'Air France-KLM au 30 juin 2025, divisé en trois sections principales : l\'activité semestrielle, les états financiers et les informations de contrôle.\n\n|Section|Page|\n|-|-|\n|Rapport semestriel d’activité|3|\n|Rapport financier|34|\n|Information et contrôle|77|\n\n\n# Sommaire\n\n# 1.\n\n----\n\nCe tableau présente le sommaire détaillé de la première partie du rapport financier, intitulée "Rapport semestriel d\'activité", incluant les sections sur l\'activité, la flotte, les faits marquants et le gouvernement d\'entreprise, avec leurs numéros de page respectifs.\n\n|Section|Libellé|Page|\n|-|-|-|\n|1.|RAPPORT SEMESTRIEL D\'ACTIVITÉ|3|\n|1.1|ACTIVITÉ|4|\n|1.1.1|Stratégie|4|\n|1.1.2|Les activités|7|\n|1.1.3|La flotte|11|\n|1.1.4|Faits marquants|13|\n|1.

In [30]:
Markdown(langchain_docs[0].page_content)

# RAPPORT FINANCIER SEMESTRIEL 2025

----

**GROUPE AIR FRANCE-KLM**
# Sommaire

Ce tableau présente la structure du rapport financier semestriel d'Air France-KLM au 30 juin 2025, divisé en trois sections principales : l'activité semestrielle, les états financiers et les informations de contrôle.

|Section|Page|
|-|-|
|Rapport semestriel d’activité|3|
|Rapport financier|34|
|Information et contrôle|77|


# Sommaire

# 1.

----

Ce tableau présente le sommaire détaillé de la première partie du rapport financier, intitulée "Rapport semestriel d'activité", incluant les sections sur l'activité, la flotte, les faits marquants et le gouvernement d'entreprise, avec leurs numéros de page respectifs.

|Section|Libellé|Page|
|-|-|-|
|1.|RAPPORT SEMESTRIEL D'ACTIVITÉ|3|
|1.1|ACTIVITÉ|4|
|1.1.1|Stratégie|4|
|1.1.2|Les activités|7|
|1.1.3|La flotte|11|
|1.1.4|Faits marquants|13|
|1.1.5|Perspectives et évènements post clôture|23|
|1.1.6|Facteurs de risques|26|
|1.1.7|Parties liées|28|
|1.2|GOUVERNEMENT D'ENTREPRISE|29|
|1.2.1|Conseil d'administration|29|
|1.2.2|CEO Committee|31|
|1.2.3|Comité exécutif Groupe|31|
|1.2.4|Bourse et actionnariat|31|


# 1.1 Activité

## 1.1.1 Stratégie

**Environnement économique :**

Ce premier tableau présente les prévisions de croissance du PIB réel pour différentes régions du monde pour les années 2024 et 2025, selon les données du FMI d'avril 2025. On observe un ralentissement généralisé de la croissance mondiale, passant de 3,3 % à 2,8 %.

|Croissance du PIB réel (%)|2024|2025|
|-|-|-|
|Monde|3,3 %|2,8 %|
|Zone euro|0,9 %|0,8 %|
|Dont France|1,1 %|0,6 %|
|Dont Pays-Bas|1,0 %|1,4 %|
|Amérique latine et Caraïbes|2,4 %|2,0 %|
|Afrique subsaharienne|4,0 %|3,8 %|
|États-Unis|2,8 %|1,8 %|
|Chine|5,0 %|4,0 %|
|Japon|0,1 %|0,6 %|


Source<sup>(1)</sup>

Depuis janvier, une série de mesures tarifaires et de contre-mesures ont été annoncées et mises en œuvre par les États-Unis à l'encontre de leurs partenaires commerciaux, ce qui entraine des répercussions sur la croissance mondiale. L'imprévisibilité avec laquelle ces mesures ont été annoncées a également eu une incidence négative sur l'activité économique et les perspectives, rendant plus difficile la production d'hypothèses et de projections cohérentes et fiables.

Selon les prévisions du FMI, la croissance des économies avancées devrait s'établir à 1,4 % en 2025, soit 0,4 point de moins qu'en 2024. La croissance aux États-Unis devrait s'établir à 1,8 %, soit 1 point de moins que l'an dernier et 0,9 point de moins que celle prévue par le FMI en janvier 2025, notamment en raison d'une plus grande incertitude politique, des tensions commerciales et d'un ralentissement de la demande. Pour la zone euro, la croissance devrait s'établir à 0,8 % en 2025, soit -0,1 point en glissement annuel et 0,2 point de moins que les projections du FMI en janvier 2025.

Dans les pays émergents et les économies en développement, la croissance devrait ralentir, passant de 4,3 % en 2024 à 3,7 % en 2025, avec de significatives révisions à la baisse par rapport aux projections du FMI de janvier 2025 pour les pays les plus touchés par les récentes mesures commerciales, comme la Chine.

### Prix du pétrole

Ce second tableau détaille l'évolution trimestrielle des cours moyens du pétrole Brent et du kérosène entre le premier trimestre 2024 et le deuxième trimestre 2025. On note une tendance baissière marquée au T2 2025, avec un Brent tombant à 68,0 $ et le kérosène à 84,0 $.

|Brent (en dollar par baril)|T1 2024|T2 2024|T3 2024|T4 2024|T1 2025|T2 2025|
|-|-|-|-|-|-|-|
|Cours moyen du Brent|83,0|84,6|79,8|75,6|75,8|68,0|
|Cours moyen du Kérosène|110,1|103,3|92,1|87,2|93,5|84,0|


Source<sup>(2)</sup>

Malgré un environnement économique et géopolitique tumultueux et une période marquée par une baisse consécutive du prix moyen du Brent d’environ 5 $ de T2 2024 à T4 2024, le prix est resté stable au T1 2025. Le cours a ensuite enregistré une forte baisse au T2 2025 pour atteindre son plus bas niveau moyen trimestriel depuis T1 2024.

Le prix du kérosène a connu la même tendance au cours des deux derniers trimestres de 2024, mais est revenu au niveau du T3 2024 au début de l’année 2025 avant de diminuer de près de 10 $ pour atteindre 84,0 $ au T2 2025.

***

<sup>(1)</sup> Source: FMI: Perspectives de l’économie mondiale, avril 2025
<sup>(2)</sup> Source: Brent et prix du Jet fuel, EIA, 10 juillet 2024
## Taux de change

Ce tableau présente l'évolution des taux de change moyens de l'euro face aux principales devises (USD, GBP, CHF, CNY, JPY) sur une période allant du premier trimestre 2024 au deuxième trimestre 2025.

|Pour un euro (moyenne)|T1 2024|T2 2024|T3 2024|T4 2024|T1 2025|T2 2025|
|-|-|-|-|-|-|-|
|USD|1,09|1,08|1,10|1,07|1,05|1,13|
|GBP|0,86|0,85|0,85|0,83|0,84|0,85|
|CHF|0,95|0,97|0,95|0,94|0,95|0,94|
|CNY|7,80|7,80|7,87|7,68|7,66|8,20|
|JPY|161,2|167,8|164,0|162,5|160,5|163,8|


Source<sup>(3)</sup>

Les taux de change sont restés stables tout au long de l'année 2024. Au premier trimestre 2025, l’euro a connu son cours le plus bas face au dollar, au yen japonais et au yuan chinois depuis le premier trimestre 2024, tandis qu’il est resté stable face au franc suisse et à la livre sterling. Au deuxième trimestre 2025, et comparé au premier trimestre 2025, l’euro s’est apprécié par rapport au dollar, au yen japonais et au yuan chinois. Tous les autres taux de change sont restés relativement stables.

## Contexte industriel :

### Offre mondiale

Au premier semestre 2025, la capacité mondiale du transport de passagers, mesurée en sièges-kilomètres disponibles (« SKO »), a augmenté de 4,8 % en glissement annuel.

*   Depuis/vers l'Europe, au premier semestre 2025, l’offre a augmenté de 6,1 % comparée à l’an dernier. Tous les flux régionaux ont enregistré une hausse en glissement annuel. La plus forte croissance a été enregistrée sur les routes asiatiques (+12 % en glissement annuel), qui se situent désormais à 95 % des niveaux de 2019. Les flux Europe-Amérique du Nord ont enregistré une croissance de 2 % tandis que les flux intra-européens ont enregistré une croissance de 6,2 % en glissement annuel.
*   Depuis/vers les États-Unis, l'offre a augmenté de 2,0 % sur les routes domestiques et de 3,1 % sur les routes internationales au premier semestre 2025. La plus forte croissance a été enregistrée sur les routes asiatiques (+7 % sur un an), toujours inférieures de 14 % à celles de 2019.
*   Après la réouverture des frontières en 2023, au départ de la Chine, la capacité internationale a augmenté de 17 % en glissement annuel au premier semestre 2025 pour atteindre 79 % de son niveau de 2019. La capacité domestique chinoise est restée stable à -0,3 % en glissement annuel au premier semestre 2025, mais reste à 123 % de son niveau de 2019.

Au cours des 5 premiers mois de 2025, la capacité mondiale de fret aérien, mesurée en tonnes-kilomètres de fret disponibles (« TKO »), a augmenté de 3,3 % en glissement annuel alors que le trafic a augmenté de 3,2 %<sup>(4)</sup>.

### Impact sur l’industrie aéronautique des événements extérieurs

Depuis le début de l'année 2025, et dans la continuité de l'année 2024, le monde a connu plusieurs événements importants ayant un impact sur l'industrie de l'aviation :

**Événements géopolitiques**

D'un point de vue géopolitique, la guerre russo-ukrainienne en cours continue d'impacter le transport aérien, en particulier sur les routes entre l’Europe et l’Asie.

La crise au Moyen-Orient persiste avec de multiples conflits simultanés. En ce moment, plusieurs espaces aériens sont fermés ou évités par certaines compagnies aériennes, qui considèrent ces espaces aériens trop dangereux pour être survolés.

**Tarifs douaniers américains**

Depuis janvier 2025, les États-Unis ont mis en œuvre une série de mesures protectionnistes. Ces mesures ont pris la forme de tarifs commerciaux affectant de nombreux biens importés aux États-Unis en provenance d'autres pays.
Les tarifs imposés par l'administration Trump ont eu des répercussions sur les flux commerciaux et les chaînes d'approvisionnement, affectant le marché du fret aérien. Outre des impacts positifs à court terme sur la demande de fret, en raison du chargement anticipé des marchandises vers les États-Unis, les tarifs ont ajouté de l'incertitude à l'environnement mondial.

Quant au transport aérien de passagers, il a été moins touché à l'échelle mondiale, à l'exception d'un impact indirect sur l'industrie touristique américaine<sup>(5)</sup>.

Les tarifs douaniers ont également eu un impact sur l'industrie de la maintenance, en particulier sur les composants d'avions et les équipements de maintenance. Malgré l'incertitude causée par ces tarifs, les compagnies aériennes adaptent leurs opérations pour faire face aux difficultés.

### Problèmes de chaîne d'approvisionnement

Compte tenu des contraintes de la chaîne d'approvisionnement, les limitations de production risquent également d'être impactées par les droits de douane, tandis que les livraisons d'avions accusent désormais un retard de 30 % par rapport à leur pic de production. Par conséquent, les compagnies aériennes s'adaptent à ce nouvel environnement en maintenant leurs avions en activité plus longtemps que prévu et en recourant à des capacités de maintenance supplémentaires. Par conséquent, la demande de transport aérien a augmenté plus rapidement que la capacité au cours des cinq premiers mois de 2025<sup>(6)</sup>.

Les problèmes de chaîne d'approvisionnement freinent également les progrès en matière de réduction des émissions de CO2, tandis que l'âge moyen des flottes mondiales augmente. À titre d'exemple, l'âge moyen des avions monocouloirs a augmenté de près de 5 mois entre le deuxième trimestre 2024 et le deuxième trimestre 2025, tandis que celui des avions bi-couloirs a augmenté de 7 mois au cours de cette période.
## 1.1.2 Les activités

### Résultat Réseaux

Ce tableau présente la performance financière de l'activité Réseaux pour le deuxième trimestre et le premier semestre 2025. On y observe une croissance du chiffre d'affaires total de +4,6 % au T2, portée par les segments Passage et Cargo. Le résultat d'exploitation progresse significativement (+221 m€ au T2) grâce à la maîtrise des coûts, notamment une baisse de 12,3 % de la facture carburant au deuxième trimestre.

|Réseaux|Deuxième Trimestre<br/>2025|Deuxième Trimestre<br/>variation|Deuxième Trimestre<br/>variation change constant|Premier Semestre<br/>2025|Premier Semestre<br/>variation|Premier Semestre<br/>variation change constant|
|-|-|-|-|-|-|-|
|Chiffre d’affaires Réseaux (m€)|6 671|+4,8%||12 436|+5,8%||
|*Chiffre d’affaires Passage*|*6 197*|*+5,0 %*||*11 441*|*+5,4%*||
|*Chiffre d’affaires Cargo*|*473*|*+2,5 %*||*994*|*+10,1%*||
|Chiffre d’affaires total (m€)|6 937|+4,6 %||12 979|+5,6%||
|Frais de personnel (m€)|-1 738|+3,9 %||-3 431|+4,6%||
|Carburant avions hors ETS (m€)|-1 395|-12,3 %||-2 833|-9,0%||
|Autres charges d’exploitation (m€)|-2 628|+8,5 %||-5 201|+8,0%||
|Dépréciations & Amortissements (m€)|-510|+1,8 %||-1 041|+4,3%||
|Résultat d'exploitation (m€)|666|+221|+190|474|+384|+407|
|Marge d'exploitation (%)|9,6 %|+2,9 pt||3,7 %|+2,9 pt||


Le chiffre d’affaires total a augmenté de +4,6 % par rapport au deuxième trimestre 2024, s’établissant à 6 937 millions d'euros. Le résultat opérationnel s'est élevé à 666 millions d'euros, soit 190 millions d'euros de plus que l'année précédente à taux de change constant, grâce à une baisse du prix du carburant et des revenus plus élevés.

Au total, la marge d’exploitation s’est établie à 9,6 %, en hausse de 2,9 points par rapport à 2024.

### Le réseau Passage affiche une solide performance au deuxième trimestre, portée par le dynamisme des cabines Premium et la progression du yield

Ce tableau détaille les indicateurs opérationnels et financiers du réseau Passage. On note une progression constante du trafic (+2,9 % au T2) et des capacités (+2,8 % au T2), maintenant un coefficient de remplissage élevé à 87,4 %. La recette unitaire au SKO progresse de +2,8 % à change constant au deuxième trimestre, témoignant d'une amélioration du yield.

|Réseaux passage|Deuxième Trimestre<br/>2025|Deuxième Trimestre<br/>variation|Deuxième Trimestre<br/>variation change constant|Premier Semestre<br/>2025|Premier Semestre<br/>variation|Premier Semestre<br/>variation change constant|
|-|-|-|-|-|-|-|
|Passagers (en milliers)|19 752|+3,4%||36 989|+3,4%||
|Capacité (millions de SKO)|70 511|+2,8%||136 421|+2,7%||
|Trafic (millions de PKT)|61 621|+2,9%||118 267|+2,6%||
|Coefficient de remplissage|87,4%|+0,0pt||86,7%|-0,1pt||
|Chiffre d’affaires total (m€)|6 362|+4,6 %|+5,2 %|11 778|+5,4%|+5,5%|
|Chiffre d’affaires passage régulier (m€)|6 197|+5,0 %|+5,7 %|11 441|+5,4%|+5,6%|
|Recette unitaire au SKO (cts €)|8,79|+2,1 %|+2,8 %|8,39|+2,6%|+2,8%|


Au cours du deuxième trimestre 2025, la capacité en Sièges-Kilomètres Offerts (SKO) a augmenté de 2,8 % par rapport à l'année précédente. La croissance du trafic (+2,9 %) a conduit à un coefficient de remplissage globalement stable à 87,4 %. Le yield corrigé des effets de change a affiché une solide performance avec une hausse de 2,8%.

Au cours du deuxième trimestre, les tendances suivantes ont été observées par zone :

**Atlantique Nord**
Malgré une croissance de capacité de 5%, la recette unitaire a progressé de 5%, soutenue par la bonne dynamique du yield dans les cabines premium, tandis que le yield en classe Economy a reculé par rapport à l’an dernier. La performance de juin a été impactée par le conflit au Moyen-Orient.

### Amérique latine
La recette unitaire a progressé, portée par un yield solide (+6,1 %), tandis que le coefficient de remplissage s’est légèrement amélioré pour atteindre 91 % et que la capacité a augmenté de 5,6%.

### Asie et Moyen-Orient
La croissance a été principalement portée par l’Asie, tandis que la capacité vers le Moyen-Orient a été impactée par les tensions géopolitiques. Excellente performance sur le Japon, la Corée et l’Asie du Sud-Est. La recette unitaire dans la région a progressé de 6 %, soutenue par une forte dynamique de yield, tandis que le coefficient d’occupation est resté stable à 89%.

### Caraïbes et Océan Indien
Une forte augmentation de la capacité à l’échelle du secteur (Air France-KLM : +5%) a conduit à un environnement tarifaire plus concurrentiel, entraînant une baisse de 2% de la recette unitaire.

### Afrique
La capacité, le coefficient d’occupation et le yield sont restés globalement stables d’une année sur l’autre.

### Court et moyen-courrier
Globalement, la capacité a augmenté de 5 %, avec un coefficient de remplissage globalement stable à 85 % et un yield inchangé. Les efforts ont porté sur la stimulation du trafic local et des volumes afin de soutenir l’augmentation de capacité.

Au premier semestre, les recettes du segment premium ont progressé de 11% par rapport à l’année précédente, portant leur part dans les recettes de l’activité réseau Passage à 28,7%, contre 27,3% sur la même période en 2024. Cette dynamique positive a été constatée dans toutes les zones géographiques. Le lancement de la nouvelle cabine La Première a renforcé l’offre premium. Sur le réseau transatlantique, la part des recettes premium est restée stable, au-delà de 41 %.

Par ailleurs, les cabines Premium Confort ont confirmé leur fort développement avec une croissance de 27 % en un an, portant leur contribution aux recettes réseau Passage à 8,1%, contre 6,7% au premier semestre 2024.

### Cargo : Performance robuste

Ce tableau présente les indicateurs de performance de l'activité Cargo pour le deuxième trimestre et le premier semestre 2025. On y observe une croissance généralisée du chiffre d'affaires et des recettes unitaires, malgré une légère baisse du tonnage au T2. Les variations sont présentées en données réelles et à change constant.

|Cargo|Deuxième Trimestre<br/>2025|Deuxième Trimestre<br/>variation|Deuxième Trimestre<br/>variation change constant|Premier Semestre<br/>2025|Premier Semestre<br/>variation|Premier Semestre<br/>variation change constant|
|-|-|-|-|-|-|-|
|Tonnage (milliers)|218|-0,2%||442|+1,9 %||
|Capacité (millions de TKO)|3 614|+1,4%||7 077|+0,8%||
|Trafic (millions de TKT)|1 644|+1,1%||3 340|+2,8 %||
|Coefficient de remplissage|45,5%|-0,1pt||47,2%|+0,9pt||
|Chiffre d’affaires total (m€)|565|+3,6%|+5,4%|1 188|+7,2%|+7,3%|
|Chiffre d’affaires transport de fret (m€)|473|+2,5%|+4,2%|994|+10,1%|+10,3%|
|Recette unitaire par TKO (cts€)|13,10|+1,0%|+2,6%|14,05|+9,1%|+9,3%|


Au cours du deuxième trimestre 2025, la capacité en Tonnes-Kilomètres Offerts (TKO) a augmenté de +1,4 % par rapport à l'année précédente, la capacité des avions cargo ayant été impactée négativement par des opérations de maintenance plus longues que prévues.

Le trafic a augmenté de +1,1 %, légèrement en deçà de la croissance de la capacité, maintenant ainsi le coefficient d’occupation quasiment stable à 45,5 %. Associé à une hausse de 3% du yield, la recette unitaire par TKO a progressé de +2,6 % à change constant. En juin, KLM a finalisé la bascule des anciens systèmes IT vers le nouveau système, Air France ayant réalisé cette opération l’année précédente.

Lors des World Air Cargo Awards (WACA) 2025, Air France-KLM Cargo a été désignée Meilleure compagnie aérienne européenne. Cette distinction récompense les compagnies aériennes ayant fait preuve d’une performance exceptionnelle, offrant un service d’excellence de manière constante, démontrant un leadership fort et contribuant au développement du secteur du fret aérien, à l’échelle mondiale ou régionale.
## Transavia : La croissance des recettes et l'amélioration du yield soutiennent les résultats du deuxième trimestre malgré des pressions sur les coûts

Ce tableau présente les indicateurs opérationnels et financiers de Transavia pour le deuxième trimestre et le premier semestre 2025. On y observe une croissance à deux chiffres du trafic et du chiffre d'affaires, contrebalancée par une hausse des coûts unitaires et des charges d'exploitation, menant à un résultat d'exploitation négatif sur le semestre.

|Transavia|Deuxième Trimestre<br/>2025|Deuxième Trimestre<br/>variation|Premier Semestre<br/>2025|Premier Semestre<br/>variation|
|-|-|-|-|-|
|Passagers (en milliers)|7 506|+12,9%|12 078|+11,3%|
|Capacité (millions de SKO)|14 266|+11,4%|23 873|+12,3%|
|Trafic (millions de PKT)|12 776|+11,2%|21 082|+11,0%|
|Coefficient d'occupation|89,6%|-0,1pt|88,3%|-1,0pt|
|Recette unitaire au SKO (cts €)|6,86|+2,9%|6,31|+1,8%|
|Coût unitaire au SKO (cts €)|6,77|+4,9 %|7,12|+3,9 %|
|Chiffre d'affaires total (m€)|946|+12,2%|1 472|+12,8%|
|Frais de personnel (m€)|-212|+13,2 %|-404|+17,0%|
|Carburant avions hors ETS (m€)|-204|-7,2 %|-358|-3,3%|
|Autres charges d'exploitation (m€)|-406|+21,6 %|-706|+20,7%|
|Dépréciations & Amortissements (m€)|-113|+48,6 %|-199|+37,3%|
|Résultat d'exploitation (m€)|12|-15|-193|-54|
|Marge d'exploitation (%)|1,3 %|-1,9pt|-13,1 %|-2,5pt|


La capacité de Transavia en sièges-kilomètres offerts a augmenté de 11,4 %, tandis que le trafic a progressé de 11,2 %, conduisant ainsi à un coefficient de remplissage globalement stable. La recette unitaire a progressé de +2,9 %, soutenue par une évolution positive du yield tant chez Transavia France que chez Transavia Pays-Bas. Cependant, aux Pays-Bas Transavia a fait face à une concurrence accrue, en partie liée à une redirection de capacité depuis le Moyen-Orient vers d'autres destinations européennes, ce qui a exercé une pression sur les recettes unitaires. Par ailleurs, l'augmentation des tarifs de Schiphol combinée à la hausse de la taxe sur les billets l'an dernier, entraînant une augmentation du prix des billets, a poussé certains voyageurs vers des aéroports allemands. En France, la performance a été affectée par une grève ayant conduit à d'importantes compensations clients. Globalement, le coût unitaire a augmenté de 4,9 % malgré la baisse des prix du carburant, principalement en raison d'une hausse de l'activité de leasing avec équipage pour Transavia Pays-Bas.

## Activité de maintenance : croissance à deux chiffres et amélioration de la marge d'exploitation

Ce tableau détaille les performances de l'activité Maintenance. On note une solide progression du chiffre d'affaires externe (+19,3% au T2) et une amélioration significative de la rentabilité, avec une marge d'exploitation atteignant 5,1% au deuxième trimestre.

|Maintenance|Deuxième Trimestre<br/>2025|Deuxième Trimestre<br/>variation|Premier Semestre<br/>2025|Premier Semestre<br/>variation|
|-|-|-|-|-|
|Chiffre d'affaires total (m€)|1 378|+14,6%|2 789|+15,0%|
|Dont Chiffre d'affaires externe (m€)|562|+19,3%|1 153|+15,2%|
|Dépenses externes (m€)|-885|+11,3%|-1 813|+13,1%|
|Frais de personnel (m€)|-320|+7,8%|-638|+8,0%|
|Dépréciations & Amortissements (m€)|-103|+40,7%|-203|+22,1%|
|Résultat d'exploitation (m€)|70|+33|135|+69|
|Marge d'exploitation (%)|5,1 %|+2,0pt|4,8 %|+2,1pt|


L'activité Maintenance a poursuivi sa forte croissance du chiffre d'affaires externe au premier trimestre 2025 avec une augmentation à deux chiffres de +19,3 %, montrant notamment une forte reprise sur le segment moteur, tandis que le chiffre d'affaires total a progressé de 14,6 %. Le résultat d'exploitation a augmenté de 33 millions d'euros et la marge d'exploitation s'est établie à 5,1 %, représentant une hausse de 2,0 point par rapport à 2024.

Le 17 juin, AFI KLM E&M, la branche MRO d'Air France-KLM, et AerCap ont annoncé être entrés en négociations exclusives en vue de la création d'une coentreprise dédiée à la location de moteurs LEAP. Les deux parties envisagent
de détenir et de gérer conjointement une flotte de moteurs CFMI LEAP-1A et LEAP-1B, permettant d'assurer la continuité des opérations des flottes Airbus A320neo et Boeing 737 MAX, pendant que les moteurs sont envoyés en visite atelier (quick-turn ou restauration de performance) au sein du réseau MRO d'AFI KLM E&M.

La création de cette coentreprise, encore soumise aux approbations nécessaires, viendrait renforcer la position d'Air France-KLM sur le marché du MRO, en s'appuyant sur l'expertise combinée et complémentaire des deux partenaires en matière de location de moteurs, de gestion d'actifs et de services MRO, afin de proposer une offre de support complète à ses clients dans le monde entier.

Au cours du deuxième trimestre 2025, AFI KLM E&M a également finalisé et annoncé plusieurs contrats MRO majeurs de long terme, notamment :

*   un contrat de 13 ans avec le groupe Saudia pour la maintenance des moteurs GE90 de la flotte Boeing 777 de Saudia,
*   un accord avec Salam Air pour des opérations de maintenance rapide (quick-turn) sur moteurs LEAP-1A,
*   un contrat de 3 ans avec Kuwait Airways portant sur les groupes auxiliaires de puissance (APU) de sa flotte Boeing 777,
*   et une extension du support actuel de maintenance moteurs pour les Boeing 777 long-courriers d'Air Austral.

Grâce à ces nouveaux contrats, l'activité MRO d'Air France-KLM renforce encore sa position sur le marché et étoffe son carnet de commandes sur des segments stratégiques clés.
### 1.1.3 La flotte

Au 30 juin 2025, la flotte du groupe Air France-KLM comprend 590 avions, dont 581 avions en exploitation contre respectivement 574 et 564 avions au 31 décembre 2024.

La flotte principale (excluant les avions régionaux) en exploitation comprend 485 avions (467 avions au 31 décembre 2024). Elle se répartit en 186 avions long-courriers (183 au 31 décembre 2024), 6 avions cargo (6 au 31 décembre 2024) et 293 avions moyen-courriers (278 au 31 décembre 2024) dont 136 avions dans la flotte du groupe Transavia (125 avions au 31 décembre 2024).

La flotte régionale en exploitation comprend 96 avions (97 avions au 31 décembre 2024).

Au 30 juin 2025, l'âge moyen de la flotte en exploitation est de 12,0 ans, dont 12,9 ans pour la flotte long-courrier, 11,7 ans pour la flotte moyen-courrier, 22,3 ans pour la flotte cargo et 10,7 ans pour la flotte régionale contre 12,1 ans au 31 décembre 2024, dont 12,7 ans pour la flotte long-courrier, 12,1 ans pour la flotte moyen-courrier, 21,8 ans pour la flotte cargo et 10,3 ans pour la flotte régionale.

Au 30 juin 2025, la flotte totale du Groupe est détenue à hauteur de 34,4 % en pleine propriété (contre 34,7 % au 31 décembre 2024), de 13,1 % en crédit-bail (contre 13,8 % au 31 décembre 2024) et de 52,5 % en location opérationnelle (contre 51,6 % au 31 décembre 2024).

Le nombre d'avions en commandes fermes au 30 juin 2025, hors locations opérationnelles, s'élève à 211 appareils, après la livraison de 10 appareils en pleine propriété du Groupe et la conversion de 30 options en commandes fermes. Le nombre d'options s'élève à 136 appareils (contre 166 au 31 décembre 2024).

La flotte du Groupe comporte 177 avions de nouvelle génération, soit 30,3 % de sa flotte.

**Résumé des mouvements de commandes :** Au premier semestre 2025, le portefeuille de commandes fermes est passé de 191 à 211 unités, porté par la conversion de 30 options malgré 10 livraisons effectuées.

|Évolution du portefeuille de commandes du groupe Air France-KLM ⁽¹⁾|31 décembre 2024|Livraisons au cours de la période|Nouvelles commandes|Conversion d’options|30 juin 2025|
|-|-|-|-|-|-|
|Flotte principale|191|10||30|211|
|Flotte régionale||||||
|TOTAL|191|10||30|211|


*(1) Hors locations opérationnelles.*

**Résumé des mouvements d'options :** Le stock d'options a diminué, passant de 166 à 136, suite à l'exercice (conversion en commandes fermes) de 30 options sur la flotte principale.

|Évolution du portefeuille d’options du groupe Air France-KLM ⁽¹⁾|31 décembre 2024|Exercice au cours de la période|Options annulées ou expirées|Nouvelles options|30 juin 2025|
|-|-|-|-|-|-|
|Flotte principale|156|30|||126|
|Flotte régionale|10||||10|
|TOTAL|166|30|||136|


*(1) Hors locations opérationnelles. Droits d’acquisition inclus*

### Gestion de flotte

Air France-KLM poursuit une politique active de renouvellement et de modernisation de sa flotte, participant ainsi à l’amélioration de son efficacité énergétique et à la réduction de son empreinte environnementale.

Ainsi, au cours du premier semestre 2025, le groupe Air France a procédé :
* pour le long courrier, à la livraison de trois A350-900 neufs et à la sortie de deux A330-200,
* pour le moyen courrier, à la livraison de trois A220-300 neufs et à la sortie de trois avions de la famille A320,
* pour HOP! à la réception de deux E190,
* pour Transavia France, à la livraison de neufs A320neo.

Le groupe KLM s’est séparé de trois E190 tandis que sept A321neo ont intégré sa flotte ainsi qu’un B787-10 et un E195-E2.

La modernisation de la flotte s’exprimera par la poursuite de la croissance de la flotte d’A350 au sein du groupe Air France-KLM et celle des B787-10 chez KLM. La poursuite de la croissance des flottes, pour Air France d’A220-300 et, pour KLM, d’E195-E2, participe également à cet important effort de modernisation de la flotte et de réduction des émissions, tout comme l’arrivée des A320neo/A321neo au sein des flottes de Transavia et KLM.
La flotte d'Air France-KLM au 30 juin 2025 :

Ce tableau présente la répartition détaillée de la flotte du groupe Air France-KLM au 30 juin 2025, ventilée par compagnie (Air France, KLM, Transavia France et Transavia Pays-Bas) et par mode de détention (Propriété, Crédit-bail, Location d'exploitation). Au total, le groupe exploite 590 appareils, dont 186 long-courriers, 296 moyen-courriers, 102 avions régionaux et 6 avions cargo.

||AF (incl. HOP!)|(incl. KLC & Martinair)|Transavia France|Transavia Pays-Bas|Total|Propriété|Crédit-bail|Location d'exploitation|
|-|-|-|-|-|-|-|-|-|
|Long-courrier|119|67|-|-|186|73|41|72|
|B777-300|43|16|-|-|59|24|11|24|
|B777-200|18|15|-|-|33|29|1|3|
|A350-900|38|-|-|-|38|4|12|22|
|B787-10|-|12|-|-|12|1|10|1|
|B787-9|10|13|-|-|23|4|7|12|
|A330-300|-|5|-|-|5|-|-|5|
|A330-200|10|6|-|-|16|11|-|5|
|Moyen-courrier|108|50|88|50|296|96|18|182|
|B737-900|-|5|-|-|5|5|-|-|
|B737-800|-|31|69|40|140|36|8|96|
|B737-700|-|6|-|-|6|6|-|-|
|A321|14|-|-|-|14|7|-|7|
|A320|36|-|-|-|36|4|3|29|
|A319|8|-|-|-|8|6|-|2|
|A318|6|-|-|-|6|5|-|1|
|A220-300|44|-|-|-|44|22|5|17|
|A320neo|-|-|19|-|19|-|1|18|
|A321neo|-|8|-|10|18|5|1|12|
|Régional|38|64|-|-|102|30|18|54|
|Embraer 190|25|24|-|-|49|17|4|28|
|Embraer 175|-|17|-|-|17|3|14|-|
|Embraer 170|13|-|-|-|13|10|-|3|
|Embraer 195 E2|-|23|-|-|23|-|-|23|
|Cargo|2|4|-|-|6|4|-|2|
|B747-400BCF|-|1|-|-|1|1|-|-|
|B747-400ERF|-|3|-|-|3|3|-|-|
|B777-F|2|-|-|-|2|-|-|2|
|TOTAL AF-KLM|267|185|88|50|590|203|77|310|


## 1.1.4 Faits marquants

### Le 29 avril 2025

**Florence Parly est nommée Présidente du Conseil d’Administration du Groupe Air France-KLM**

Sur proposition du Comité de nomination et de gouvernance, le Conseil d’Administration d’Air France-KLM a décidé, lors de sa réunion du 29 avril 2025, de nommer Florence Parly en tant que Présidente du Conseil d’Administration du Groupe Air France-KLM.

Cette nomination a pris effet à l’issue de l’Assemblée générale 2025 statuant sur les comptes de l’année 2024, qui s’est tenue le 4 juin prochain.

Florence Parly a succédé à Anne-Marie Couderc, Présidente du Conseil d’Administration du Groupe depuis mai 2018, dont le mandat arrivait à échéance.

Florence Parly avait rejoint le Conseil d’Administration du Groupe en décembre 2023 en tant qu’administratrice.

Commentant la nomination de Florence Parly, Anne-Marie Couderc a déclaré : « Je suis très heureuse de la décision prise par le Conseil d’administration de nommer Florence Parly pour me succéder à la présidence du Conseil. Sa très grande expérience du monde des affaires et des institutions, à des postes de direction dans le domaine des transports et en tant que ministre, sont des atouts remarquables pour la réussite d’Air France-KLM.

Arrivée à la tête du Groupe il y a 7 ans, je suis fière du chemin parcouru avec Benjamin Smith. Malgré une situation financière qui reste à consolider, notre Groupe est ressorti plus fort des crises qui se sont succédé. Je suis confiante pour l’avenir d'Air France-KLM et remercie particulièrement l’ensemble des collaborateurs pour les moments d’exception partagés avec eux ces dernières années. »

### Le 1<sup>er</sup> juin 2025

**IndiGo, Delta Air Lines, Air France-KLM et Virgin Atlantic annoncent un partenariat visant à relier l’Inde et son économie en pleine croissance à l'Amérique du Nord et à l'Europe**

Les transporteurs ont signé un protocole d'accord prévoyant la mise en place d'un partenariat de premier plan entre l'Amérique du Nord, le Royaume-Uni, l'Europe, l'Inde, et au-delà.

IndiGo, Delta Air Lines, Air France-KLM et Virgin Atlantic ont dévoilé aujourd'hui leur intention de mettre en place un partenariat de premier plan reliant l'Inde à l'Europe et à l'Amérique du Nord, et pouvant à terme être étendu au niveau mondial.

L'Inde, l'un des marchés aériens les plus dynamiques au monde, est au cœur de cette collaboration. En connectant le vaste réseau domestique d'IndiGo à la puissance de Delta en Amérique du Nord et sur les liaisons transatlantiques, à la couverture étendue d'Air France-KLM en Europe et en Amérique du Nord, ainsi qu'à la présence de Virgin Atlantic au Royaume-Uni et sur les liaisons transatlantiques, ce partenariat offrira aux voyageurs une plus grande connectivité, des trajets plus fluides et une expérience de voyage harmonisée à travers les continents. Reliant chacune des dizaines de villes aux États-Unis, au Canada, en Europe et en Inde, les compagnies aériennes visent à répondre à la demande croissante en matière de voyages internationaux tout en établissant de nouveaux standards de connectivité et de coopération mondiale dans le secteur aérien.

Pieter Elbers, Président-Directeur général d'IndiGo, a déclaré : « IndiGo s'est engagée dans le projet ambitieux de devenir une compagnie aérienne mondiale d'ici 2030. Le partenariat annoncé aujourd'hui marque une nouvelle étape importante dans la mise en œuvre de synergies commerciales, dans la recherche de l'excellence opérationnelle et en matière d'innovation. Cette annonce renforce notre relation avec Air France-KLM et Virgin Atlantic, mais marque également le début d'un nouveau chapitre passionnant avec l'arrivée de Delta Air Lines comme partenaire privilégié. Nous sommes particulièrement enthousiastes à l'idée de commencer notre développement sur les vols long-courriers dès cet été et de nous connecter aux réseaux de nos partenaires pour offrir un meilleur accès à l'Europe et à l'Amérique du Nord. Ce partenariat stratégique nous permettra de développer une offre attractive, avec une connectivité intercontinentale complète, une expérience fluide et des bénéfices de grande valeur en matière de fidélisation. Il pose également les bases pour un échange de bonnes pratiques dans les domaines des nouvelles technologies, de l’excellence opérationnelle et de la qualité de service. »

Ed Bastian, Président-Directeur Général de Delta Air Lines, a déclaré : « Cet accord illustre notre engagement à rendre le voyage plus connecté, plus inclusif et plus accessible. En combinant nos forces à celles d'IndiGo, d'Air France-KLM et de Virgin Atlantic, nous offrirons une connectivité et une fluidité inégalées, garantissant à nos clients les plus hauts standards de service et de fiabilité à travers le monde. En parallèle, nous sommes impatients de reprendre prochainement nos vols directs entre les États-Unis et l'Inde. »
Benjamin Smith, Directeur Général d'Air France-KLM, a déclaré : « Nous nous réjouissons d’étendre notre partenariat existant avec IndiGo et de le faire aux côtés de nos partenaires Delta et Virgin Atlantic. L'Inde est un marché stratégique pour Air France-KLM, où nous avons une présence forte et historique qui va bientôt s'accroître. Nous sommes impatients d'accueillir les clients d'IndiGo sur nos vols et de jouer un rôle actif dans la connectivité du pays. »

Shai Weiss, PDG de Virgin Atlantic, a déclaré : « Alors que nous fêtons les 25 ans de nos liaisons vers l'Inde, nous sommes ravis de pouvoir tirer parti de l’expérience de notre joint-venture avec Delta et Air France-KLM et de notre accord de partage de code avec IndiGo, pour développer plus encore ce partenariat. Avec le lancement des vols long-courrier d’IndiGo nous relierons quatre des plus grandes économies mondiales, tout en améliorant l'expérience de voyage de nos clients, grâce à une meilleure connectivité, des horaires de vol optimisés et des avantages liés à la fidélité. »

### Des réseaux aériens connectés depuis 2022

IndiGo est partenaire d’Air France-KLM et de Virgin Atlantic depuis 2022, permettant aux clients d'Air France, de KLM et de Virgin Atlantic d’accéder à plus de 30 destinations IndiGo en Inde.

Plus récemment, IndiGo a annoncé le lancement de vols vers l'Europe, ce qui ouvre à la fois la voie à une extension du partenariat existant avec Air France-KLM et Virgin Atlantic pour les vols transatlantiques, et offre de nouvelles perspectives de partenariat entre IndiGo et Delta, permettant aux clients d'IndiGo de bénéficier du vaste réseau transatlantique de la compagnie américaine. Une fois les contrats commerciaux signés et les procédures réglementaires accomplies, IndiGo pourra vendre les vols de ses partenaires comme s'il s'agissait des siens, en y apposant son code marketing 6E*. Les clients d'IndiGo pourront ainsi réserver des trajets en correspondance sur certains vols opérés par ses partenaires internationaux, ce qui lui permettra d’offrir davantage de destinations en Europe et en Amérique du Nord. Ceci concerne notamment la commercialisation par IndiGo :

* de vols KLM au départ d'Amsterdam vers 30 destinations en Europe
* de vols Delta et KLM au départ d'Amsterdam vers les États-Unis et le Canada
* de vols Virgin Atlantic au départ de Manchester vers les États-Unis

En complément, la nouvelle liaison récemment annoncée par KLM entre Amsterdam à Hyderabad offrira de nouvelles opportunités de coopération avec IndiGo. Dès le lancement de cette liaison en septembre 2025, Air France-KLM vendra des vols IndiGo vers 24 destinations en continuation d'Hyderabad.

IndiGo a précédemment annoncé qu'elle comptait renforcer sa flotte de Boeing 787 avec l’ajout de six appareils en damp lease (location avec équipage) d'ici la fin de l'année. La compagnie a également passé une commande ferme à long terme portant sur 30 Airbus A350-900, avec des options pour 70 appareils supplémentaires. À mesure que ces appareils rejoindront la flotte d'IndiGo, l'expansion de la compagnie aérienne sur long-courrier permettra une coopération plus étroite et plus approfondie entre IndiGo et Delta, Air France-KLM et Virgin Atlantic.

Delta, quant à elle, prévoit de reprendre ses liaisons vers l'Inde avec des vols sans escale entre Atlanta et Delhi, sous réserve de l’obtention des autorisations gouvernementales requises. La date de lancement sera annoncée ultérieurement.

### Collaboration future

Le protocole d'accord établit également un cadre pour une collaboration plus étroite entre les transports aériens sur une base bilatérale et multilatérale. Il ouvre la voie à une collaboration commerciale, notamment en matière de réseau, de fidélisation, de fret et de ventes (sous réserve de l’obtention des autorisations réglementaires requises). Dans les domaines non-commerciaux, la coopération pourrait notamment porter sur les activités de maintenance aéronautique, le développement durable, la formation et l'assistance en escale. Afin de servir efficacement leurs clients communs, les partenaires prévoient enfin de développer une collaboration avancée grâce à l'utilisation de la technologie.

### Le 2 juin 2025

#### Riyadh Air et Air France-KLM signent un protocole d’accord en vue d’établir une coopération stratégique et d’étendre la connectivité aérienne mondiale

L’accord porte sur la connexion des réseaux aériens, la mise en place d’accords de partage de codes et la reconnaissance mutuelle des programmes de fidélisation, afin d’offrir aux clients une expérience fluide lors de leurs déplacements.

Les partenaires exploreront également les opportunités de coopération dans les domaines de la maintenance aéronautique, du développement durable, de la transformation numérique et du transport de marchandises.

Riyadh Air - la nouvelle compagnie aérienne nationale d’Arabie Saoudite, et le Groupe Air France-KLM ont signé un protocole d’accord (MoU) visant à établir une coopération stratégique. Cet accord représente une avancée significative pour l’amélioration de la connectivité aérienne mondiale offerte aux passagers voyageant entre Riyad, Paris, Amsterdam, et effectuant des correspondances vers d’autres destinations.

L’accord a été signé lors de l’assemblée générale 2025 de l’Association Internationale du Transport Aérien (IATA), par Tony Douglas, le PDG de Riyadh Air, et Benjamin Smith, Directeur général du groupe Air France-KLM. Il pose les bases d’un partenariat dynamique. Sous réserve de l’obtention des autorisations réglementaires requises, cette coopération
vise à introduire progressivement un large éventail d’avantages destinés aux clients, et ouvrira de nouvelles opportunités en Europe occidentale, en Amérique du Nord et du Sud, au Moyen-Orient, en Asie et au Royaume d’Arabie Saoudite.

Cette alliance stratégique vise en priorité à renforcer la connectivité du réseau aérien. Les deux partenaires uniront leurs forces pour maximiser les opportunités de correspondance futures via le hub de Riyadh Air à Riyad d’une part, et via ceux d’Air France-KLM à Paris-Charles de Gaulle et Amsterdam-Schiphol d’autre part. L’ouverture récente par Air France d’une liaison directe entre Paris-Charles de Gaulle et Riyad, en complément de la liaison Amsterdam-Riyad déjà offerte par KLM, reflète ainsi l’engagement des partenaires à élargir les options de voyage de et vers l’Arabie Saoudite.

Au-delà de la connexion de leurs réseaux aériens, Riyadh Air et Air France-KLM souhaitent coopérer en vue d’améliorer l’expérience de leurs clients respectifs. Les partenaires exploreront les opportunités de mise en place d’une réciprocité des avantages liés aux programmes de fidélisation ; de soutien opérationnel et d’accès aux salons dans les aéroports. Le partenariat couvre également la maintenance aéronautique, la transformation numérique, le développement durable et le transport de marchandises, dans une démarche tournée vers la création de valeur.

Tony Douglas, Président-Directeur général de Riyadh Air, a déclaré : « Nous mettons tout en œuvre pour relier Riyad au reste du monde et notre partenariat avec Air France-KLM, un leader mondial du transport aérien, nous permettra d’atteindre cet objectif plus rapidement. Nous souhaitons offrir des expériences exceptionnelles et concrétiser notre ambition à long terme : redéfinir le transport aérien grâce à l'innovation, à l'excellence opérationnelle et à des services qui placent le client au centre de toutes les attentions. Ce partenariat viendra à la fois renforcer notre réseau international, soutenir la confiance dans notre trajectoire de croissance et confirmer notre rôle en soutien des objectifs de diversification économique de l'Arabie saoudite dans le cadre de la Vision 2030. »

Benjamin Smith, Directeur général d’Air France-KLM, a ajouté : « L'Arabie Saoudite s’impose rapidement comme un marché clé pour l'aviation. Nos trois marques, Air France, KLM et Transavia, desservent désormais le pays, et grâce à ce nouvel accord avec Riyadh Air, nous souhaitons continuer à renforcer notre présence dans la région. Nous sommes heureux de combiner nos réseaux et notre expertise à ceux de Riyadh Air afin d'offrir à nos clients communs davantage de choix et des voyages toujours plus fluides. Cette coopération est complémentaire avec celle déjà en place avec notre partenaire au sein de l’alliance SkyTeam présent dans la région. »

Alors que Riyadh Air entend devenir l'un des principaux transporteurs aériens mondiaux, ce partenariat avec Air France-KLM vient souligner son engagement en faveur de l'excellence, de l'innovation et d'une expérience client exceptionnelle.

### Le 05 juin 2025

**Flying Blue, le programme de fidélité du Groupe Air France-KLM célèbre 20 ans d'innovation au service du voyage et poursuit sa mue pour s’intégrer toujours davantage dans le quotidien de ses membres**

En juin 2025, Flying Blue a célébré son vingtième anniversaire. Deux décennies de fidélité, de relation privilégiée avec ses membres, d'innovation et d'excellence au-delà de l’expérience de voyage. Né en 2005 de la fusion des programmes Fréquence Plus d’Air France et Flying Dutchman de KLM, Flying Blue a su évoluer pour devenir l'un des programmes de fidélité les plus appréciés du secteur, avec près de 30 millions de membres à travers le monde.

**De plus en plus de membres et une reconnaissance internationale**

Initialement destiné aux clients d’Air France, de KLM et d’Aircalin, Flying Blue a élargi sa portée en accueillant la compagnie Tarom en 2010 et Transavia en 2013. Aujourd'hui, Flying Blue rassemble près de 30 millions de membres dans le monde et comptabilise près de 40 compagnies aériennes partenaires. Plus de 100 partenaires commerciaux sont également connectés au programme, pour permettre aux membres de gagner et d’utiliser leurs miles lors de leurs voyages et au quotidien.

Récemment élu « meilleur programme de fidélité aérien mondial » par le site spécialiste des programmes de fidélité Point.me, et « meilleur programme élite en Europe et Afrique » par les Freddie Awards (prix prestigieux dans le secteur), Flying Blue continue de se distinguer par ses nombreux avantages, la qualité de son service client et sa facilité d'utilisation.

**Un programme de fidélité toujours plus proche du quotidien de ses membres**

Depuis sa création, Flying Blue a régulièrement développé des produits et services marquants : la carte Air France et American Express lancée en 1998, remplacée par la carte cobrandée Flying Blue x American Express en 2010, l’offre de location de voiture avec Hertz, les offres Cash & Miles et Flying Blue Shop for Miles en 2018 et le partenariat All (le programme de fidélité du groupe Accor) x Flying Blue lancé en 2019.

En 2022, l'offre Flying Blue Family a été introduite, permettant le partage de Miles au sein des membres d’une même famille. Des partenariats avec Amazon, Uber, Hertz et aussi la SNCF pour acheter des bons d’achats de train en France ont enrichi l'écosystème entre 2023 et 2025. L’objectif à chaque fois : augmenter les opportunités de gain de Miles pour permettre aux membres d’accéder rapidement à des primes ou à des avantages. Initialement pensé comme un programme destiné aux voyageurs fréquents, Flying Blue est ainsi devenu un programme de fidélité à part entière, ne limitant plus l’accumulation et la dépense de miles au seul secteur du voyage.
Avec sa nouvelle application mobile Flying Blue+ disponible pour l’instant au Royaume-Uni et aux Etats-Unis, le programme s’inscrit résolument dans cette démarche, permettant à ses membres d’accumuler des miles lors de leurs achats auprès d’enseignes comme Macy’s, Burberry et Apple. Cette application sera progressivement disponible dans d’autres pays prochainement.

Flying Blue permet par ailleurs à ses membres d’agir pour l’environnement ou de soutenir des associations. Ils peuvent ainsi utiliser leurs miles pour acheter du carburant d'aviation durable et en faire don à des ONG comme La Croix Rouge, l’Unicef ou encore Ocean Clean up. En 2024, 450 millions de Miles ont ainsi été donnés par les membres Flying Blue, permettant notamment de financer les déplacements et l’action de ces structures.

En 20 ans, Flying Blue a su s'adapter aux transformations technologiques, économiques et sociétales de l'industrie du voyage, le tout en rester fidèle à une volonté : celle de créer des expériences mémorables pour ses membres.

Pour marquer cet anniversaire, Flying Blue lance la campagne publicitaire mondiale et innovante « Turning Miles into Memories, for 20 years », mettant en lumière l'évolution du programme et les souvenirs des voyageurs au fil des années. Cette campagne illustre le rôle de Flying Blue dans l'expérience de voyage de ses membres, les accompagnant à chaque étape de leur vie, en créant des souvenirs inoubliables tout en s'adaptant à l'évolution de leurs besoins et aux évolutions de la société.

Depuis le 6 juin 2025, la campagne est visible dans les aéroports de Paris Charles de Gaulle, d’Orly, d’Amsterdam Schiphol et de New York John F. Kennedy ainsi que sur les réseaux sociaux Air France-KLM et Flying Blue dont Facebook, Instagram et le service de musique Spotify.

Enfin, une opération spéciale, « Miles of Memories », offrira à quelques heureux membres de plus de 30 pays tirés au sort l'opportunité de revisiter des destinations emblématiques du réseau Air France-KLM et de créer de nouveaux souvenirs.

### Le 06 juin 2025
### Air France-KLM : principales décisions prises lors de l’Assemblée Générale du 4 juin 2025

L’Assemblée générale des actionnaires d’Air France-KLM s’est réunie le mercredi 4 juin 2025 à 14h30 à l’hôtel Van der Valk Paris CDG Airport, 351 avenue du Bois de la Pie, 95700 Roissy-en-France.

Elle a été diffusée en direct via webcast sur le site internet d’Air France-KLM. Il est également possible de la visionner à tout moment en différé via le lien suivant : https://voda.akamaized.net/airfrance/ag-2025-fr/

Au cours de cette Assemblée générale mixte, où plus de 6527 actionnaires étaient présents ou représentés, l’ensemble des résolutions proposées ont été adoptées. Outre l’approbation des comptes de l’exercice clos le 31 décembre 2024 et l'affectation du résultat, l’Assemblée Générale a notamment pris les décisions suivantes :

**Nominations/renouvellements :**

*   Mme Gwenaëlle Avice-Huet a été renouvelée en qualité d’administratrice pour une durée de deux ans (résolution n°6) ;
*   Mme Leni Boeren a été renouvelée en qualité d’administratrice pour une durée de quatre ans (résolution n°7) ;
*   Delta Air Lines, Inc. a été renouvelé en qualité d’administrateur pour une durée de quatre ans (résolution n°8) ;
*   Mme Isabelle Guichot a été nommée en qualité d’administratrice pour une durée de quatre ans (résolution n°9) ;
*   Mme Anne-Marie Idrac a été renouvelée en qualité d'administratrice pour une durée de deux ans (résolution n°10) ;
*   Mme Véronique Penchienati-Bosetta a été nommée en qualité d’administratrice pour une durée de quatre ans (résolution n°11) ;
*   M. Qingchao Wan a été nommé en qualité d’administrateur pour une durée de quatre ans (résolution n°12) ;

**Approbation des conventions et engagements réglementés relatifs :**

*   à un nouvel accord de joint-venture entre Air France-KLM, Air France, KLM et China Eastern Airlines (résolution 4) ;
*   à la coopération commerciale entre Air France-KLM, Delta Air Lines Inc. et Virgin Atlantic Airways Ltd (résolution n°5) ;

**Rémunération :**

*   Approbation des informations sur la rémunération 2024 de chacun des mandataires sociaux requises par l’article L.22-10-9 I du Code de commerce (résolution n°13) ;
*   Approbation des éléments de rémunération versés ou attribués au cours de l’exercice 2024 à la Présidente du Conseil d’administration et au Directeur général (résolutions n°14 et 15) ;
*   Approbation des politiques de rémunération 2025 des mandataires sociaux non dirigeants, de la Présidente du Conseil d’administration et du Directeur général (résolutions n°16 à 18).

**Autorisations/délégations financières :**
*   Autorisation donnée au Conseil d'administration, avec faculté de subdélégation, de mettre en oeuvre le programme de rachat d'actions dans les limites et conditions fixées dans la résolution y afférant (résolution n°19) ;
*   L'ensemble des délégations financières ont été adoptées. Le détail de ces délégations figure en page 37 et suivantes de la brochure de convocation (résolutions 22 à 28) ;
*   Adoption des délégations autorisant des augmentations de capital réservées aux salariés du Groupe dans la limite de 3% du capital social (résolutions n°29 et 30) ;
*   Autorisation donnée au Conseil d'administration de réduire le capital par annulation des actions auto-détenues (résolution n°31).

**Modification statutaire :**

*   Modification de l'article 2 des statuts relatif à l'objet social de la Société afin de mieux refléter la réalité de son activité (résolution 20).
*   Modification de l'article 20 des statuts relatif aux délibérations du Conseil d'administration afin de tenir compte des nouvelles dispositions de la loi n°2024-537 du 13 juin 2024 visant à accroître le financement des entreprises et l'attractivité de la France (la « loi attractivité »), ayant notamment simplifié les modalités de tenue des réunions du Conseil d'administration (résolution n°21).

Le résultat détaillé des votes ainsi que l'ensemble des documents relatifs à l'Assemblée générale sont disponibles sur le site internet de la Société (https://www.airfranceklm.com/fr/finance/actionnaires/assemblee-generale).

### Le 20 juin 2025

#### « Connect France » : Air France-KLM et Groupe ADP font drapeau commun, avec le soutien de l'Etat français, pour mieux connecter et faire gagner la France

Les deux groupes établissent un partenariat inédit qui repose sur une feuille de route commune pour la compétitivité, le rayonnement et la décarbonation du hub d'Air France de Paris-Charles de Gaulle à l'échelle mondiale. Cette démarche ambitieuse vise à promouvoir la souveraineté nationale et européenne ainsi qu'à positionner le hub d'Air France à Charles de Gaulle en leader mondial.

**Connect France**

A l'occasion de la 55ème édition du Salon International de l'Aéronautique et de l'Espace (SIAE) et en présence de M. Emmanuel Macron, Président de la République, Benjamin Smith, Directeur général du Groupe Air France-KLM et Philippe Pascal, Président-Directeur général du Groupe ADP, ont annoncé aujourd'hui le lancement de « Connect France », un partenariat d'actions ambitieux destiné à renforcer la coopération entre les deux groupes.

**Connect France : faire gagner la Team France**

L'initiative Connect France est née d'un constat commun : dans un contexte géopolitique complexe et un environnement de concurrence exacerbée, un renforcement de leur coopération existante est indispensable. Aussi, Connect France a pour ambition de mieux aligner la compagnie aérienne nationale et le premier aéroport européen, pour faire du hub d'Air France à l'aéroport Paris-Charles de Gaulle une référence mondiale.

Air France-KLM et Groupe ADP sont des actifs stratégiques, essentiels à la souveraineté et l'attractivité de la France. Représentant plus de 870 000 emplois directs, indirects et induits<sup>[1]</sup> en France, ils soutiennent l'emploi et l'économie du pays tout entier, avec une forte empreinte territoriale. Air France-KLM est le premier employeur privé en Ile-de-France et 11% de la valeur des exportations de la France passe par l'aéroport Paris-Charles de Gaulle. En unissant leurs forces, les deux entreprises se donnent les moyens de renforcer la position du hub, pour faire gagner la Team France.

Si la France reste la première destination touristique mondiale, sa desserte passe progressivement aux mains d'acteurs extra-européens bénéficiant d'un cadre réglementaire et fiscal plus attractif. Des hubs aux frontières de l'Europe détournent un trafic dont une partie transitait précédemment par les plateformes européennes, dans une forme de délocalisation silencieuse. Un sursaut est nécessaire pour que la France reste maîtresse de sa connectivité et qu'elle continue à disposer d'un hub mondial puissant, directement connecté à l'ensemble des centres économiques mondiaux.

Ce contexte appelle une mobilisation collective de tous les acteurs et à un soutien de l'Etat. Outre une démarche commune de sensibilisation des autorités françaises et européennes aux enjeux de compétitivité du transport aérien, Connect France a pour objet de porter des initiatives concrètes. Les deux groupes communiqueront à échéance régulière sur l'avancée des travaux réalisés dans le cadre de ce partenariat.

**Faire du hub d'Air France à l'aéroport Paris-Charles de Gaulle une référence mondiale**

A l'image de duos existants à travers le monde unissant un aéroport et la compagnie y étant basée, Connect France vise à renforcer les liens entre Air France et l'aéroport Paris-Charles de Gaulle, première porte d'entrée du pays, pour positionner ce duo au plus haut niveau de l'industrie et en faire une référence en matière d'expérience client, de performance opérationnelle et de décarbonation.
Les deux partenaires ont déjà démontré leur capacité à relever ensemble des défis considérables, notamment lors des Jeux Olympiques et Paralympiques de Paris 2024. Connect France permettra de renforcer ce travail collaboratif et de capitaliser sur les progrès réalisés ces dernières années.

« Le contexte impose un plus grand alignement stratégique entre tous les acteurs de l’écosystème du transport aérien en France, à la hauteur de l'enjeu », a déclaré Benjamin Smith, Directeur général du Groupe Air France-KLM. « Avec Connect France, nous formalisons cette démarche absolument nécessaire pour mieux faire face à la concurrence d’acteurs extra-européens qui ont de longue date pris la mesure de l’importance du transport aérien pour le rayonnement et l’économie d’une nation. Je me réjouis que cette initiative soit partagée avec l’Etat français. Elle se concentrera également sur l’amélioration du service rendu à nos clients, pour que la France se donne les moyens de rester une place forte de l’aviation, avec une connectivité renforcée, créatrice d’emploi local ».

« L'aéroport de Paris-Charles de Gaulle est un instrument de souveraineté essentiel de la France pour sa connectivité, son tourisme, et son économie. Face à une compétition féroce, la France peut rayonner par le développement de sa connexion au monde et par une expérience pour le passager attractive et différenciante. Nous avons donc décidé d'allier les forces de nos deux groupes pour faire de la France une puissance aérienne mondiale compétitive et unique en son genre » a déclaré Philippe Pascal, Président-Directeur général du Groupe ADP. « J'ai fait de cette démarche ma priorité dès le début de mon mandat. Connect France porte une méthode nouvelle dans la relation entre Groupe ADP et Air France-KLM au travers de réalisations concrètes et rapides que nous engageons pour la performance, la différenciation et la décarbonation du hub, au service des passagers ».

**Agir rapidement. Ensemble.**

Les actions rapides et prioritaires sont issues des remontées de terrain des passagers, des partenaires et des collaborateurs, afin d'améliorer et enrichir l'expérience client.

**Fluidité**

Mise en place d'un accès dédié pour fluidifier le parcours des passagers en correspondances courtes, "short connection pass". Il sera proposé dès l'enregistrement aux passagers qui rencontrent sur leur voyage une correspondance de moins d'une heure, pour les prioriser aux contrôles grâce à un cheminement dédié sur la base d'informations actualisées en temps réel.

→ Objectif été 2025 : mise en place du dispositif

Augmenter la part des embarquements directs en passerelle pour réduire au maximum les trajets en bus, dans les infrastructures existantes, en particulier pour les passagers internationaux long-courrier et les passagers en correspondance. Le taux de contact pour les gros porteurs du hub d'Air France se situe aujourd'hui en moyenne autour de 95%. Grâce à une optimisation des opérations, au double tractage, et à la création de nouveaux postes avions gros porteurs au 2E hall K, le taux de contact s'améliorera progressivement, en particulier sur les périodes de pointe du hub d'Air France. Le double tractage consiste à libérer un poste avion le temps de la préparation d'un autre avion, pour ensuite le ramener rapidement au contact.

→ Objectif 2026 : croissance progressive du taux de contact gros porteurs (hors travaux) avec l'objectif d'atteindre le meilleur taux d'Europe et du Moyen-Orient.

**Lisibilité**

Renommer les terminaux de l'aéroport pour plus de lisibilité pour les passagers. L’objectif est de faciliter les parcours en aérogare, notamment en correspondance, rejoignant ainsi les meilleurs standards internationaux. Cette nouvelle dénomination facilitera également la gestion des opérations.

→ Objectif fin 2025 : annonce du nouveau plan de dénomination pour un déploiement d'ici 2026.

**Attractivité**

Lancer une offre commune différenciante de stop-over – escale touristique de correspondance – pour proposer le meilleur de la destination Paris-Ile-de-France. L'objectif de cette offre, de quelques heures jusqu'à plusieurs nuitées, est de créer la différence et la préférence face à d'autres grands hubs du monde, en capitalisant sur la force de la destination Paris-Ile-de-France (loisirs, culture, patrimoine, divertissement, gastronomie, etc.)

→ Objectif fin 2025 : lancement d'une nouvelle offre de stop-over.

Faire du 2E Hall K, le flagship France, le plus beau terminal au monde, avec le meilleur du savoir-faire national : gastronomie avec de grand-chefs, mode, joaillerie, beauté, patrimoine culturel. Ce niveau d'excellence française sera proposé tant dans la zone commerciale, que la salle d'embarquement, le Salon Business et le Salon Première d'Air France.

→ Objectif fin 2025 : dévoilement du projet flagship France avec une première étape d'ouverture dès 2026 (zone centrale).

**Durabilité**

Structurer un soutien des deux groupes au développement de la filière des carburants d’aviation durables. En matière de décarbonation, les deux groupes se constituent en leaders : Air France est l'un des premiers acheteurs mondiaux de carburant d’aviation durable et Groupe ADP se positionne comme facilitateur de l'approvisionnement en énergie bas carbone sur l'ensemble de ses aéroports.
→ Objectif fin 2025 : dans la continuité de CarbAero[2], soutien mutualisé des deux groupes aux acteurs européens de production de SAF.

**Annexe :**
Les 10 initiatives conjointes du pacte d’actions Connect France entre Air France-KLM et le Groupe ADP

### #1 Infrastructures de l’aéroport pour le hub d’Air France
**Penser ensemble les futures infrastructures du Hub**

Le plan d'aménagement de l’aéroport Paris-Charles de Gaulle à l'horizon 2035 et 2050, porté à la concertation d'avril à juillet 2025, se traduira par des investissements structurants, qui engageront l'avenir de la plateforme sur les 10 à 15 prochaines années. Afin que ces projets répondent au mieux aux besoins du Hub, les deux entreprises mettent en place des équipes projet conjointes pour clarifier et préciser ensemble leurs besoins fonctionnels.

### #2 Performance opérationnelle
**Déployer une série d'initiatives conjointes pour la performance opérationnelle du Hub**

Cette initiative vise à accentuer et systématiser les travaux menés entre Air France et l'aéroport de Paris-Charles de Gaulle pour l'amélioration de la gestion des capacités du Hub d'Air France, notamment en pointe, ainsi que sur le plan de robustesse bagage et trieurs avec l'objectif de réduire les temps de livraison bagages, d'améliorer la ponctualité en particulier sur les premiers vols matinaux, ou encore sur le pilotage de la performance des touchées avion.

### #3 Expérience des passagers en correspondance
**Transformer l'expérience des passagers en correspondance heureuse**

Les deux entreprises s'engagent à mobiliser leurs compétences pour faciliter et simplifier les parcours des passagers en correspondance (meilleure information aux passagers, dénomination des terminaux, simplification et fluidification des parcours, etc.). Par ailleurs, elles s'engagent à offrir en salle d'embarquement ainsi que dans les salons des expériences singulières mettant l'accent sur le savoir-faire national et la singularité de Paris, pour marquer leur différence vis-à-vis des Hubs concurrents.

### #4 Intermodalité
**Transformer l'offre d'intermodalité voyageurs et concevoir une gare mieux adaptée aux parcours en correspondance train / avion**

Le développement de l’intermodalité est au cœur de la vision d’aménagement de l’aéroport Paris-Charles de Gaulle. Dès lors, les deux entreprises renforcent leur collaboration pour transformer l'expérience des passagers sur les correspondances train / avion entre la gare TGV et les terminaux, afin d’améliorer l’offre existante et préparer la vision cible.

### #5 Offre de stop-over à Paris
**Etudier le développement d'une offre différenciante qui mise sur la destination France**

Parmi les axes de différenciation vis-à-vis des hubs concurrents, l’aéroport Paris-Charles de Gaulle dispose d'un atout non négligeable : Paris. L'ambition est d'offrir aux passagers en correspondance un moment d'art de vivre à la française, qui pourra s'exprimer dans l'aéroport, mais également dans Paris via la mise en place d'une offre de stop-overs dans la capitale et/ou en Ile-de-France.

### #6 Visibilité des marques
**Faire rayonner l’aéroport en capitalisant sur la force des marques des deux Groupes**

Il s'agira de capitaliser sur la force des marques des deux Groupes, tant à travers la communication visuelle en aérogares que via des campagnes communes de communication, à l'instar de ce qui est fait par d’autres duos aéroport/ compagnie basée à travers le monde. Par ailleurs, le Groupe ADP et Air France-KLM ambitionnent de développer leur coopération commerciale.

### #7 Carburants durables d'aviation (SAF)
**Coopérer pour accélérer le déploiement des carburants d'aviation durables à l’aéroport Paris-Charles de Gaulle**

Air France et le Groupe ADP disposent de tous les atouts pour faire du Hub d’Air France à l’aéroport Paris-Charles de Gaulle un leader mondial de développement des carburants d'aviation durables (SAF). Par-delà la valorisation de leurs efforts conjoints et coopérations dans le cadre de plateformes industrielles françaises et européennes, les deux entreprises étudieront l'opportunité d'investissements communs dans la chaîne de valeur SAF.

### #8 Enjeux énergétiques et environnementaux
**Faire ensemble du Hub d’Air France à l’aéroport Paris-Charles de Gaulle une référence mondiale en matière environnementale**

Air France et le Groupe ADP partagent une même ambition de verdissement de l’aéroport Paris-Charles de Gaulle à horizons 2035 et 2050 et poursuivent le développement de synergies sur l'électrification des usages, les investissements dans la chaleur renouvelable, l'optimisation des opérations, la lutte contre les pollutions ou encore l'adaptation au changement climatique.
### #9 Frontière (initiative conduite avec la DNPAF)

**Soutenir des solutions innovantes pour améliorer la fluidité à la frontière**

Dans le cadre du déploiement progressif de l'Entry Exit System (EES) à compter de l'automne 2025, Air France-KLM et le Groupe ADP, en lien étroit avec la Direction Nationale de la Police aux Frontières (DNPAF), mettront tout en œuvre pour préserver la fluidité de parcours aux frontières. L'initiative vise par ailleurs à plaider conjointement en faveur de nouveaux leviers de fluidification de la frontière, à accompagner la montée en puissance des ressources (PARAFE notamment) et à planifier de nouvelles infrastructures aux frontières.

### #10 Data et innovation

**Accroître le partage de données et innover ensemble pour renforcer les opérations**

Dans le strict respect de la réglementation en vigueur, Air France-KLM et le Groupe ADP souhaitent améliorer le partage de données opérationnelles et clients autour du Hub pour gagner en performance, mais aussi d'être davantage en mesure de communiquer et proposer de nouveaux services aux passagers. Elles soutiendront les innovations soutenant la performance opérationnelle, l’expérience client et la fluidité de leur parcours.

[1] Impact socio-économique du Groupe Air France-KLM en France : 45 000 emplois directs et 506 000 emplois indirects et induits (étude Castéran, juin 2024). Impact socio-économique de Paris-CDG : 90 000 emplois directs et 230 emplois indirects (étude BDO, 2023)

[2] Appel à projets pour le développement d’une filière de production française de carburants aéronautiques durables, annoncé le 23 avril 2025 par Philippe TABAROT, ministre chargé des transports et Marc FERRACCI, ministre chargé de l’Industrie et de l’Energie. L’Assemblée générale des actionnaires d’Air France-KLM s’est réunie le mercredi 5 juin 2024 à 14h30 à l’Hôtel Hilton Paris Charles de Gaulle, 8 rue de Rome, 93290 Tremblay-en-France. Elle a été diffusée en direct via webcast sur le site internet d’Air France-KLM. Il est également possible de la visionner à tout moment en différé via le lien suivant : https://www.yuca.tv/fr/air-france-klm/ag-2024-air-france-klm.

### Le 20 juin 2025

#### Air France-KLM et Airbus signent un accord permettant aux salariés d’Airbus de réduire l’impact de leurs déplacements professionnels en soutenant le développement du carburant d’aviation durable

Air France-KLM et Airbus poursuivent leurs efforts en faveur de la décarbonisation du transport aérien.

Dans le cadre d’un accord entré en vigueur début 2025, les salariés d’Airbus peuvent désormais réserver des "SAF Bundles" pour leurs voyages d'affaires, soit des tarifs incluant directement dans le billet d’avion une contribution volontaire à l’achat de carburant d’aviation durable (SAF). Cette initiative innovante permet aux collaborateurs d’Airbus de réduire l’impact de leurs déplacements. Cette option est disponible directement via leur outil de réservation.

A la suite du succès d'un projet pilote en 2023, Air France-KLM et Airbus renforcent leur collaboration avec un accord tarifaire spécifique pour les SAF. Airbus s'engage désormais à acheter des options SAF pour les déplacements professionnels de ses employés en intégrant pour la troisième année consécutive le programme « SAF Corporate » d’Air France-KLM qui permet le financement et l’achat de SAF.

Ainsi, depuis début 2025, Air France-KLM propose aux salariés d’Airbus la possibilité de financer un pourcentage de SAF lors de la réservation sur une sélection d’itinéraires très utilisés par le Groupe, tels que Toulouse-Paris et Toulouse-Montréal, où Airbus dispose d’un site de production. Cette option, visible dès la réservation, vise à sensibiliser individuellement les salariés aux efforts de réduction des émissions de CO2 liés aux voyages d'affaires.

Depuis son engagement dans le programme SAF en novembre 2023, Airbus a réduit de plus de 2 000 tonnes ses émissions de CO2 grâce à l'achat de plus de 670 tonnes de SAF.

Sébastien Guyot, Directeur général des Ventes Globales chez Air France-KLM, a déclaré : « Réduire nos émissions de CO2 est une priorité pour notre groupe. Les carburants d'aviation durable sont essentiels pour atteindre nos ambitions. La contribution de nos clients corporate, à l'instar d'Airbus, est cruciale. »

Raphaël Duflos, Vice President Corporate Services Procurement chez Airbus a déclaré en retour : « Nous sommes fiers d'être pionniers dans l'adoption de la toute nouvelle offre "SAF bundle" suite au déploiement du Nouveau Canal de Distribution (NDC) avec Air France. Cela illustre l'importance de notre partenariat stratégique avec Air France sur ce sujet clé autour de la décarbonation du voyage d’affaires, une collaboration qui ne cesse d'évoluer et de se développer depuis 2023. »

Le Groupe Air France-KLM est pleinement engagé dans la réduction des émissions de CO2 et active tous les leviers disponibles pour atteindre ses ambitions de décarbonation notamment le renouvellement de la flotte, les mesures opérationnelles comme l’éco-pilotage, l’intermodalité qui permet de combiner l’avion à des modes de transport bas carbone comme le train, et bien sûr l’incorporation de carburant d’aviation durable. Les SAF sélectionnés par Air France-KLM permettent une réduction d'au moins 65% des émissions de CO2 sur l'ensemble du cycle de vie du carburant, en comparaison à leur équivalent fossile. Le groupe a mis en place une politique d'achat très stricte et ne se fournit qu’avec des carburants d’aviation durable qui ne concurrencent pas la production alimentaire humaine, ne contribuent pas à la déforestation et ne sont pas produits à partir d'huile de palme.
### Le 23 juin 2025

#### Air France-KLM lance un rafraichissement de la marque de sa filiale low-cost Transavia

Transavia, la filiale low-cost du Groupe Air France-KLM, entame un nouveau chapitre de son histoire. Devenue un nom familier pour les voyageurs à travers l'Europe, Transavia est aujourd'hui une marque à part entière. L'année dernière, 23 millions de passagers ont voyagé avec Transavia, en hausse de 8,1% par rapport à 2023.

Alors que Transavia poursuit sa croissance, en ligne avec l'expansion de sa flotte et de son réseau, Air France-KLM a décidé de rafraîchir sa marque de manière subtile. Le logo évoluera pour devenir plus moderne et statutaire, tout en incarnant l'audace qui caractérise la marque. La couleur verte sera conservée, dans une nouvelle nuance plus douce.

Ces éléments seront progressivement déployés sur tous les points de contact de la compagnie : un nouveau site internet, une nouvelle signalétique dans les aéroports, ainsi qu'une nouvelle livrée.

Un premier aperçu des changements à venir est visible sur un Airbus A320neo en cours d'assemblage à Hambourg (Allemagne), avion qui sera livré à Transavia France dans le courant de l'année. L'empennage de l'appareil est désormais visible, et la livrée complète sera dévoilée dans les prochaines semaines.

Transavia exploite actuellement 131 appareils et a amorcé une transition progressive vers une flotte composée à 100 % d'avions de la famille Airbus A320neo. Elle opère près de 400 lignes à travers l'Europe, l'Afrique du Nord et le Moyen-Orient, leurs efforts en faveur de la décarbonisation du transport aérien.

### Le 25 juin 2025

#### Le Groupe Air France-KLM au Salon du Bourget 2025 : innovation, coopération et ambition collective

Du 16 au 22 juin 2025, le Groupe Air France-KLM était présent à la 55e édition du Salon International de l'Aéronautique et de l'Espace (SIAE), au Bourget. Une semaine intense et stratégique, marquée par des échanges riches avec clients, partenaires, institutions, médias et futurs talents.

L'occasion de mettre en lumière l'expertise industrielle du Groupe, de réaffirmer sa détermination à accélérer la décarbonation du transport aérien, et de rappeler le rôle central de ses activités dans les économies française, néerlandaise et européenne.

**« Connect France » : une nouvelle alliance stratégique pour renforcer la connectivité française**

L'un des temps forts de cette édition fut le lancement de « Connect France » en présence de M. Emmanuel Macron, Président de la République française. Cette collaboration inédite entre Air France-KLM et Groupe ADP vise, avec le soutien de l'État français, à renforcer la connectivité de la France, à améliorer l'expérience client et à accélérer la transition écologique du transport aérien.

Benjamin Smith, Directeur général du Groupe Air France-KLM – aux côtés du Président de la République et de Philippe Pascal, Président-Directeur générale du Groupe ADP – a déclaré : « Avec Connect France, nous formalisons cette démarche absolument nécessaire pour mieux faire face à la concurrence d'acteurs extra-européens. Elle se concentrera également sur l'amélioration du service rendu à nos clients, pour que la France reste une place forte de l'aviation, créatrice d'emplois locaux. »

Fluidification des correspondances, simplification des noms des terminaux, possibilité de proposer conjointement des escales touristiques à Paris et en Île-de-France... cliquez ici pour découvrir les 10 mesures phares du pacte d'actions « Connect France ».

**Développement durable : une vision claire et des actions concrètes**

Engagé de longue date dans la décarbonation du transport aérien, le Groupe Air France-KLM a multiplié au salon du Bourget les prises de parole et les initiatives.

Lors d'une présentation keynote au Paris Air Lab, Anne Rigail, Directrice générale d'Air France, a rappelé les quatre piliers de la stratégie de décarbonation de la compagnie. Elle a également insisté sur l'urgence d'une politique européenne ambitieuse sur les carburants durables.

Le Groupe Air France-KLM a également conclu un accord avec Airbus, permettant aux collaborateurs du constructeur de réduire l'empreinte carbone de leurs déplacements professionnels en soutenant la production de carburant d'aviation durable (SAF). Cet engagement volontaire s'inscrit dans le cadre du programme « SAF Corporate » d'Air France-KLM, et vise à développer une offre transparente, traçable et contrôlée.

En parallèle, Air France a signé le volet SAF du contrat de filière des Nouveaux Systèmes Énergétiques avec l'État et les industriels. En la présence de M. Eric Lombard (ministre de l'Économie, des Finances et de la Souveraineté industrielle et numérique), de M. Marc Ferracci (ministre chargé de l'industrie et de l'énergie) et de M. Philippe Tabarot (ministre chargé des transports), les signataires ont identifié quatre objectifs principaux :

*   Fixer des cibles nationales de production et consommation de SAF d'ici 2030 et au-delà;
* Développer des modèles de financement compétitifs à l’échelle française et européenne
* Soutenir l’industrialisation des projets SAF et sécuriser leur rentabilité
* Lancer les premiers projets de production de bioSAF avancés et eSAF avant 2030.
* Une activité de maintenance aéronautique qui rayonne à l’international.

Sur le stand Air France Industries KLM Engineering & Maintenance, la semaine a été rythmée par des annonces et des signatures d’accords commerciaux. Parmi les principales actualités de la semaine :

* AFI KLM E&M et Saudia Group signent un accord stratégique pour la maintenance de moteurs CFM56-5B, illustrant une confiance renouvelée entre les deux groupes.
* EPCOR (une filiale à part entière d’AFI KLM E&M) et Kuwait Airways prolongent leur coopération pour le support APU des Boeing 777, renforçant un partenariat historique.
* SalamAir confie à AFI KLM E&M le support de ses moteurs LEAP-1A, consolidant la position du Groupe au Moyen-Orient.

AFI KLM E&M a finalisé l’industrialisation du moteur LEAP-1A sur ses bancs d’essai modernisés à Paris et Amsterdam, désormais pleinement opérationnels. Cette avancée renforce son leadership dans la maintenance des moteurs de nouvelle génération et prépare l’arrivée du LEAP-1B d’ici la fin de l’été.

Toujours sur stand d’AFI KLM E&M, le MRO Lab a mis à l’honneur ses expertises reconnues et ses innovations concrètes : maintenance prédictive, impression 3D, réalité augmentée… autant de solutions à la pointe de la technologie, au service des 200 compagnies aériennes clientes à travers le monde.

### Attractivité et recrutement : cap sur les talents de demain

La dernière partie du Salon du Bourget, ouverte au grand public du 20 au 22 juin, a été une véritable vitrine pour les métiers du Groupe.

Elle a offert une occasion unique de faire découvrir la diversité des savoir-faire d’Air France-KLM, d’échanger avec des collaborateurs passionnés et de susciter des vocations. Les équipes de ressources humaines d’Air France et de Transavia étaient présentes pour accueillir les visiteurs, répondre à leurs questions et les informer sur les nombreuses opportunités de carrière au sein du Groupe.
### 1.1.5 Perspectives et évènements post clôture

#### Perspectives 2025 reconfirmées

Pour 2025, le Groupe garde une approche agile compte tenu de l'incertitude actuelle et prévoit :

*   Une **capacité** en sièges kilomètres offerts pour le groupe Air France-KLM, y compris Transavia, en hausse de 4 à 5% en 2025 par rapport à 2024
*   Une augmentation du **coût unitaire**<sup>7</sup> à un chiffre bas par rapport à 2024
*   Des **dépenses d'investissements nettes** comprises entre 3,2 et 3,4 milliards d'euros
*   Un **levier d'endettement** (dette nette/EBITDA) compris entre 1,5x et 2,0x

#### Événements post clôture

**Le 04 juillet 2025**

**Air France-KLM entame le processus en vue d'une prise de participation majoritaire dans le capital de SAS**

*   Air France-KLM prévoit de porter sa participation au capital de SAS de 19,9 % actuellement à 60,5 %, en acquérant l'intégralité des parts détenues par Castlelake et Lind Invest.
*   Ce projet de transaction reflète le succès de la restructuration de SAS et les résultats positifs de la coopération commerciale initiée en 2024. L'opération permettrait à Air France-KLM et SAS d'exploiter pleinement leur potentiel de synergies, de confirmer l'expansion du Groupe sur le marché scandinave, et de créer un potentiel supplémentaire de création de valeur pour les actionnaires.
*   Sous réserve de l'obtention des autorisations réglementaires requises et de la levée des conditions suspensives, l'objectif est de finaliser l'opération au deuxième semestre 2026.

Le Groupe Air France-KLM annonce aujourd'hui qu'il va entamer un processus visant à prendre une participation majoritaire dans le capital de SAS. Le Groupe détient actuellement 19,9 % du capital de la compagnie scandinave et a initié à l'été 2024 une coopération commerciale entre Air France, KLM et SAS, reposant sur des accords élargis de partage de codes et de commercialisation interline. Cette coopération a été renforcée par l'entrée de SAS dans l'alliance SkyTeam.

Sous réserve de la satisfaction de l'ensemble des conditions requises, Air France-KLM procéderait à l'acquisition de l'ensemble des parts détenues par Castlelake et Lind Invest, portant ainsi sa propre participation dans SAS à 60,5 %. L'État danois conserverait sa participation de 26,4 % ainsi que ses sièges au sein du Conseil d'administration.

La valeur de l'investissement envisagé par Air France-KLM serait déterminée au moment de la finalisation de l'opération, sur la base des dernières performances financières de SAS – incluant l'EBITDA et la dette nette. Cette opération serait alignée avec les perspectives financières du Groupe à moyen terme.

Sous réserve de l'obtention des autorisations réglementaires requises et de la levée des conditions suspensives, l'ambition du Groupe serait de finaliser l'opération au deuxième semestre 2026.

L'acquisition de cette participation majoritaire permettrait à Air France-KLM de prendre le contrôle de SAS, qui deviendrait alors une filiale du Groupe. Cette nouvelle étape permettrait à Air France-KLM et SAS d'exploiter pleinement leur potentiel de synergies grâce à une intégration dans l'ensemble des domaines d'activité, y compris les programmes de fidélité, et au-delà des seules fonctions commerciales. Le Groupe détiendrait la majorité des sièges au sein du Conseil d'administration de la compagnie aérienne.

L'intérêt d'Air France-KLM pour cette opération repose sur l'amélioration significative des performances financières et opérationnelles de SAS, sur le succès de la coopération commerciale actuelle, ainsi que sur la confiance du Groupe dans le potentiel à long terme de la compagnie scandinave.

« Nous nous réjouissons à l'idée d'accueillir SAS en tant que membre à part entière de la famille Air France-KLM » a déclaré Benjamin Smith, Directeur général d'Air France-KLM. « SAS affiche d'excellentes performances à la suite de sa restructuration réussie, et nous sommes convaincus que son potentiel continuera de croître grâce à une intégration plus poussée au sein du Groupe. Cette opération serait bénéfique pour nos clients et pour les voyageurs scandinaves, qui profiteraient d'une meilleure connectivité, ainsi que pour les équipes de SAS, dont l'engagement a été déterminant pour redonner à leur compagnie la place qu'elle mérite. SAS rejoindrait un groupe de compagnies aériennes unies par

***

<sup>7</sup> à prix de carburant et change constants et hors ETS
une même ambition d’excellence et par un engagement en faveur d’un transport aérien plus durable. Nous avons hâte d’ouvrir ensemble ce nouveau chapitre. »

**Annexe - informations clés concernant SAS à fin juin 2025**

Ce tableau présente les indicateurs opérationnels et financiers de la compagnie SAS pour l'exercice 2024 et sa situation à fin juin 2025, incluant sa flotte, son trafic, ses effectifs, son chiffre d'affaires ainsi que la répartition de son actionnariat.

|Indicateur|Valeur||
|-|-|-|
|Avions en opération :|138||
|Destinations desservies :|>130||
|Passagers transportés :|>25 millions (2024)||
|Cargo transporté|60 000 tonnes (2024)||
|Membres du programme Eurobonus :|>8 millions||
|Employés :|10500||
|Chiffre d'affaires :|4,1 milliards d'euros (exercice 2024)||
|Structure du capital :|||
||Castlelake (32,0%)||
||Etat danois (26,4%)||
||Air France-KLM (19,9%)||
||Lind Invest (8,6%)||
||Autres (13,1%)||
|Alliance :|SkyTeam (depuis Septembre 2024)||


### Le 10 juillet 2025

#### <mark>Air France-KLM et Qantas étendent leur partenariat historique pour améliorer la connectivité entre l'Europe et l'Australie</mark>

Air France-KLM et Qantas, la compagnie aérienne nationale australienne, ont annoncé aujourd'hui l’extension de leur accord de partage de code visant à renforcer la connectivité et à améliorer l'expérience de voyage entre l'Europe et l'Australie. Ce partenariat renforcé entre Air France, KLM et Qantas offrira plus de flexibilité et d’avantages liés à la fidélité.

Dans le cadre de l’extension de ce partenariat, Air France apposera son code AF sur le vol direct de Qantas reliant Paris-Charles de Gaulle (CDG) à Perth (PER). Ce mois-ci marque le premier anniversaire de l’opération du vol direct Qantas entre l’Australie et la France. Ce service, le premier et unique vol passager sans escale entre les deux pays, réduit d’environ trois heures le temps de trajet le plus rapide entre les deux villes.

De plus, Air France est ainsi la seule compagnie aérienne à proposer un accord de partage de code à ses clients sur une liaison directe d’Europe vers l’Australie.

Par ailleurs, et depuis le 1er juillet, Air France a ajouté un nouveau point de connexion où ses vols rejoignent ceux de Qantas : l'aéroport de Tokyo Haneda (HND). Ce nouvel hub vient compléter les points de correspondance déjà disponibles, à savoir l'aéroport de Singapour Changi (SIN) ainsi que l'aéroport international de Hong Kong (HKG). Les clients en transit via ces trois points de connexion peuvent effectuer une correspondance facilement vers plusieurs villes australiennes : Sydney (SYD), Brisbane (BNE), Melbourne (MEL), Darwin (DRW) et Perth (PER) à bord des vols opérés par Qantas. Par ailleurs, Air France et KLM ajoutent également leurs codes aux les vols de Qantas entre Singapour (SIN) et Darwin (DRW).

Cette extension permet également aux clients de cumuler et de dépenser des Points Qantas lors de leurs voyages avec Air France et KLM. D’autres avantages liés à la fidélité seront introduits ultérieurement.

Pour la première fois, les membres du programme Flying Blue d'Air France-KLM ont la possibilité d'utiliser leurs Miles pour voyager en cabine First de Qantas. De plus, les membres Flying Blue ayant le statut "Ultimate" bénéficient désormais d'un accès exclusif aux salons de Première classe de Qantas.

Pour les voyageurs australiens, ce partenariat donne accès à un plus grand nombre de destinations en France et en Europe, grâce aux vastes réseaux domestiques et européens d'Air France et de KLM.

Cette coopération renforcée entre Air France, KLM et Qantas marque le début d'une nouvelle ère pour les voyages internationaux, offrant une meilleure connectivité, une meilleure flexibilité et un confort amélioré pour les clients.
voyageant entre les deux continents. Air France-KLM salue la décision rendue ce jour par la Commission européenne confirmant que les mesures de soutien en liquidités accordées aux compagnies du Groupe par les États français et néerlandais pendant la crise du Covid-19 étaient conformes aux règles européennes en matière d’aides d’État. Pour rappel, ces mesures d’aides avaient été accordées en 2020 sous la forme de Prêts Garantis par l’État (PGE) et de prêts directs d’État. Les aides néerlandaises et françaises ont été intégralement remboursées en juin 2022 et avril 2023 respectivement.

### Le 11 juillet 2025

#### Angus Clarke quitte Air France-KLM, avec effet le 31 août 2025

Angus Clarke, Directeur général adjoint et Directeur commercial d'Air France-KLM, a informé la compagnie qu'il démissionnera de son poste à compter du 31 août 2025.

Il quitte ses fonctions dans l'intention de prendre de nouvelles responsabilités plus tard dans l'année, en dehors de la compagnie.

Benjamin Smith, Directeur général d'Air France-KLM, a déclaré : "Angus m'a informé de sa volonté de relever désormais de nouveaux défis. Le travail qu’il a accompli dans son champ de responsabilités au cours des six dernières années permet au Groupe d’envisager avec sérénité la transition avec le Directeur commercial qui prendra sa succession.

Parmi les nombreux projets menés à bien, je voudrais remercier personnellement Angus pour sa contribution significative à la transformation de la flotte d'Air France-KLM, en particulier pour son implication directe dans l'engagement pris par le Groupe de commander jusqu'à 99 avions long-courriers et près de 200 avions court et moyen-courriers de nouvelle génération. Ces commandes constituent la colonne vertébrale des futurs plans de flotte d'Air France, de KLM et de Transavia pour une aviation plus décarbonée.

Angus a également été très actif sur de nombreux autres projets, notamment le réaménagement des cabines d’avions, la planification du réseau, la modernisation de la co-entreprise Blue Skies et, plus récemment, le choix d'un nouveau système de ventes de services aux passagers : Amadeus Nevio.

Je souhaite à Angus beaucoup de succès dans ses projets futurs et je le remercie pour les services qu'il a rendus à Air France-KLM".

Angus Clarke a déclaré : "Ce fut un plaisir et un honneur de travailler pour l'un des plus grands groupes aériens au monde et de siéger à son comité exécutif. Je tiens également à remercier personnellement Ben pour l'opportunité qu'il m'a donnée de travailler à ses côtés chez Air France-KLM au cours des six dernières années."

### Le 28 juillet 2025

#### La force du collectif : comment la coopération opérationnelle fait avancer le Groupe Air France-KLM

Dans un environnement opérationnel de plus en plus complexe et souvent imprévisible, l’adaptabilité et la collaboration sont devenues essentielles. Au sein d’Air France-KLM, les compagnies aériennes du Groupe unissent plus que jamais leurs forces pour relever les défis opérationnels, garantir la continuité des activités et renforcer leur résilience à long terme. Du partage des ressources de pilotage au déploiement d’outils communs, les initiatives récentes montrent que la coopération entre entités n’est plus seulement une opportunité — c’est désormais un véritable levier stratégique qui renforce chacune des compagnies et le Groupe dans son ensemble.

**Des pilotes Air France sur des vols KLM : une première historique**

Cette année, et pour la première fois dans l’histoire du Groupe, des pilotes Air France assurent des vols KLM entre Amsterdam et New York. Ces vols, programmés du 16 juillet au 26 octobre 2025, sont opérés sur des Boeing 777 de KLM, avec des personnels navigants commerciaux KLM à bord. Cette solution ciblée a été mise en place pour aider KLM à faire face à une pénurie temporaire de pilotes qualifiés sur cet appareil durant la haute saison estivale.

L’initiative a été développée en seulement trois mois par la Direction de la Transformation Groupe, en collaboration étroite avec les équipes des deux compagnies. Environ 200 collaborateurs issus de nombreux métiers — opérations aériennes, sécurité, maintenance, escale, informatique, ressources humaines, affaires juridiques, etc. — ont mis ensemble leurs expertises pour faire aboutir ce projet en un temps record. Le dispositif a nécessité l’obtention d’autorisations réglementaires des autorités françaises et néerlandaises, l’implication des organisations professionnelles de pilotes et PNC, l’adaptation de la documentation opérationnelle, l’intégration des systèmes informatiques, la définition de protocoles communs, et la mise en place de formations spécifiques. La rapidité et la précision d’exécution illustrent la capacité du Groupe à mettre en œuvre des solutions agiles, transverses et efficaces.

**Une réponse coordonnée aux défis communs**
Cette logique de coopération s'étend à d'autres domaines opérationnels. Pour préserver la continuité de son programme de vols et sécuriser certains créneaux aéroportuaires, KLM Cityhopper va s'appuyer sur Hop! pour l'exploitation de trois Embraer E190 durant trois saisons, à partir de l'hiver 2025. Hop! opère déjà la ligne Nantes–Amsterdam, s'inscrivant ainsi dans un modèle opérationnel éprouvé, prêt à être élargi selon les besoins. En parallèle, alors que KLM s'apprête à intégrer ses premiers Airbus A350 à partir de 2026, une solution conjointe de formation des pilotes a été mise en place. Dès septembre 2025, des pilotes KLM ayant une expérience préalable sur Airbus commenceront à voler sur des A350 opérés par Air France, encadrés par des instructeurs de la compagnie française. Ce dispositif leur permet d'acquérir les qualifications requises tout en participant activement aux opérations du Groupe.

### Des outils partagés au service de l'efficacité et de l'harmonisation

Au-delà des ressources humaines, le Groupe mise également sur des outils numériques et des systèmes communs pour accroître son efficacité. Une nouvelle solution commune de planification et de régulation des vols est en cours de déploiement chez Air France, KLM et Transavia, accompagnée d'une plateforme d'information partagée sur les programmes de vol et les opérations. Ces outils favorisent une meilleure cohérence, évitent les doublons, améliorent la réactivité en situation perturbée — tout en générant des économies d'échelle significatives.

En parallèle, l'outil d'analyse de données de vol SARA FDM (Flight Data Monitoring) – utilisé par Air France, KLM, Transavia et Hop! – permet de centraliser les données de sécurité des vols et de générer des tableaux de bord partagés. Cette harmonisation renforce l'analyse des risques et les capacités de prévention, contribuant à un environnement de vol plus sûr pour toutes les compagnies du Groupe.

### Un modèle de résilience collective

Ensemble, ces projets illustrent une approche pragmatique et orientée résultats : concevoir des solutions efficientes, reproductibles, et parfaitement adaptées aux réalités opérationnelles. Ils témoignent également d'une dynamique plus large au sein du Groupe, qui repose sur la complémentarité des expertises internes et un sens partagé des responsabilités.

Dans un secteur en constante évolution, ces synergies permettent à Air France-KLM d'agir avec agilité et confiance. En mutualisant les compétences, en réaffectant les ressources lorsque nécessaire, et en investissant dans des outils communs, le Groupe renforce chacune de ses compagnies tout en consolidant sa performance collective. À tous les niveaux des opérations, ces initiatives confirment une conviction simple : en travaillant ensemble, le Groupe Air France-KLM est plus fort que la somme de ses parties.

## 1.1.6 Facteurs de risques
Les facteurs de risques auxquels est exposé le groupe Air France-KLM sont décrits dans le Document d'Enregistrement Universel 2024 déposé le 03 avril 2025 sous le numéro D.25-0226 auprès de l'AMF (Chapitre 3 "Facteurs de risque "). A l'exception de ce qui suit, la nature de ces risques n'a pas connu d'évolution significative au cours du premier semestre de l'exercice 2025 :

### 3.1.2.9 Engagements vis-à-vis de la Commission Européenne

#### Description du risque

**a) Contrôle des concentrations et accords de coopération**

En 2004, un certain nombre d'engagements ont été pris par Air France et KLM afin de s'assurer de la conformité de la décision de la Commission européenne qui avait autorisé le rapprochement d'Air France et de KLM et, notamment, la possibilité de mettre des créneaux de décollage et d'atterrissage à la disposition des concurrents sur certains aéroports.

**b) Aides d'État**

En 2020, la mise en œuvre des mesures de renforcement de la liquidité du Groupe (à savoir (i) la garantie par l'État français d'un prêt bancaire de 4 milliards d'euros (le PGE), le prêt direct de l'État français de 3 milliards d'euros, ainsi que (ii) la garantie par l'État néerlandais d'un crédit bancaire renouvelable de 2,4 milliards d'euros et le prêt direct de l'État néerlandais de 1 milliard d'euros) a été approuvée par la Commission Européenne en vertu des règles relatives aux aides d'État dans le cadre de la crise du Covid-19 (Décisions respectivement du 4 mai 2020 et du 13 juillet 2020, cette dernière décision ayant été remplacée, après annulation pour défaut de motivation, par une décision du 16 juillet 2021).

Le 6 avril 2021, le Groupe a annoncé la première partie de son plan global de recapitalisation. Certaines mesures de ce plan contenaient des aides d'État dites « de recapitalisation Covid-19 » qui ont été notifiées par les autorités françaises à la Commission européenne, cette dernière les ayant approuvées dans sa décision du 5 avril 2021. Cette approbation a été accordée sous réserve d'un certain nombre d'engagements pris par l'État français conduisant, notamment, Air France à mettre des créneaux d'atterrissage et de décollage sur l'aéroport d'Orly à la disposition d'un transporteur tiers désigné. À l'instar de la plupart des décisions relatives aux compagnies aériennes bénéficiaires d'aides d'État dans le cadre de la crise du Covid-19, les décisions de la Commission européenne approuvant les mesures de soutien à Air France et à KLM ont fait l'objet de recours en annulation de la part de Ryanair. Le 20 décembre 2023 et le 7 février 2024, le Tribunal de l'Union européenne a annulé les décisions de la Commission européenne qui avaient approuvé les mesures de soutien susmentionnées, respectivement à hauteur de (i) 7 milliards d'euros de mesures de liquidité accordées par l'État français à Air France en mai 2020 et 3,6 milliards d'euros de mesures de recapitalisation accordées par l'État français à Air France et Air France-KLM S. A. en 2021, et (ii) 0,9 milliard d'euros de mesures de liquidité accordées à KLM par l'État néerlandais en 2020. Ces annulations se sont faites sur l'unique motif d'une détermination erronée du bénéficiaire de ces aides, celui-ci devant être, d'après le Tribunal, le Groupe en lui-même.

Comme expliqué infra, Air France-KLM, Air France et KLM, tout comme la Commission européenne, ont contesté ces trois arrêts d'annulation devant la Cour de justice de l'Union européenne. Sans attendre l'issue de ces pourvois, la Commission européenne a décidé d'approuver les mesures françaises et néerlandaises de renforcement de la liquidité du Groupe, dans une décision unique du 10 juillet 2024 confirmant leur compatibilité avec le droit de l'Union (même en appliquant la détermination du bénéficiaire par le Tribunal de l'Union européenne). Cette décision a fait l'objet d'un nouveau recours en annulation devant le Tribunal de l'Union européenne de la part de Ryanair le 14 avril 2025. Air France-KLM, Air France et KLM interviendront, aux côtés des États français et néerlandais, au soutien de la défense de la Commission européenne.

En janvier 2025, le Groupe a été informé du dépôt par Ryanair d'un recours devant le Tribunal administratif de Paris contre l'État français à la suite des arrêts d'annulation précités du Tribunal de l'Union européenne. La demande de Ryanair vise à ce que l'État récupère toute aide d'État dont il est allégué qu'elle n'aurait pas encore été remboursée. Le Groupe et Air France se sont constitués parties, le 3 juillet 2025, pour répondre en défense à ce recours aux côtés de l'État français. La procédure est en cours.

Par ailleurs, Air France-KLM a également été informée, en avril 2025, d'une procédure similaire (mais de nature administrative dans un premier temps) diligentée par Ryanair contre les autorités néerlandaises s'agissant de la décision relative à l'aide accordée à KLM en 2020. La procédure est en cours.

Si la Cour de justice annule les arrêts précités du Tribunal de l'Union européenne, ces recours de Ryanair deviendraient sans objet.

#### Impact

**a) Contrôle des concentrations et accords de coopération**

Le non-respect des engagements pris dans le cadre du contrôle des concentrations et des accords de coopération comporte un risque financier, réputationnel et structurel.

**b) Aides d'État**

L'incertitude demeure quant aux conséquences juridiques et financières de l'annulation des décisions d'approbation des aides d'État jusqu'à l'obtention d'un arrêt en dernier ressort des juridictions européennes. Toutefois, toutes les aides accordées ont déjà été remboursées, en 2023, en pleine conformité avec toutes les contraintes liées
(engagements, mesures comportementales, application d’intérêts) au cadre juridique applicable.

Néanmoins, l’annulation des décisions d’approbation de ces aides pourrait (en attendant les arrêts de la Cour de justice) entraîner le paiement des seuls « intérêts d’illégalité » sur les mesures de liquidité réapprouvées, (à nouveau contestées devant le Tribunal de l’Union européenne, comme indiqué supra), et une récupération d’un montant à déterminer sur les mesures de recapitalisation qui s’ajouterait aux montants ayant déjà fait l’objet d’un remboursement.

### Plan d'atténuation

#### a) Contrôle des concentrations et accords de coopération
Le groupe Air France-KLM s’est assuré que les conséquences éventuelles de la mise à disposition de créneaux horaires au titre des mesures correctives de 2004 restent admissibles et ne modifient pas l’économie des lignes considérées. Par ailleurs, Air France-KLM contacte régulièrement la Commission européenne afin de discuter de la nécessité du maintien de ces engagements adoptés il y a plus de seize ans. À cet égard, la Commission européenne a levé le 24 février 2023 les engagements offerts par Air France et KLM en 2004 sur la route Paris-Amsterdam.

#### b) Aides d’État
Le Groupe a procédé au cours des exercices 2022 et 2023 au remboursement de l’intégralité des aides d’État à la liquidité et à la recapitalisation Covid-19 susmentionnées et qui étaient grevées des engagements et contraintes précitées. En conséquence, Air France-KLM, Air France et KLM sont totalement libérées depuis lors des engagements et contraintes précitées liées aux aides de recapitalisation Covid-19.

Air France-KLM, Air France et KLM ont formé trois pourvois en annulation devant la Cour de justice de l’Union européenne contre les trois arrêts du Tribunal annulant les décisions mentionnées ci-dessus rendues en décembre 2023 et février 2024, au seul motif d’une mauvaise détermination du bénéficiaire de ces mesures d’aides. La Cour de justice de l’Union européenne doit encore se prononcer sur ces pourvois.

Si la Cour de justice annulait les arrêts du Tribunal, les décisions d’approbation de 2020 et 2021 seraient à nouveau pleinement effectives, étant censées n’avoir jamais été annulées. Dans ce cas, les risques liés au paiement d’intérêts d’illégalité ou à une récupération ne seraient plus matérialisés.

Par ailleurs, à la suite du recours précité de Ryanair du 14 avril 2025, le Tribunal de l’Union européenne doit à nouveau se prononcer sur la validité de l’approbation des aides de liquidité, réadoptée le 10 juillet 2024. Air France-KLM, Air France et KLM interviendront au soutien de la défense de la Commission européenne.

Enfin, comme elle l'a fait dans des cas similaires, la Commission européenne peut également décider, le cas échéant, d'entamer une procédure formelle d'examen des mesures de recapitalisation au cours de laquelle le Groupe veillera à défendre au mieux ses intérêts. Une telle procédure formelle d’examen peut aboutir à une nouvelle approbation formelle de ces mesures, rendant sans objet toute demande de récupération de leur montant nominal.

### 1.1.7 Parties liées

Les informations concernant les parties liées font l’objet de la Note 26 aux comptes consolidés.
## 1.2 Gouvernement d'entreprise

### 1.2.1 Conseil d'administration

Au 30 juin 2025, le Conseil d’administration comprend dix-neuf membres dont :
* seize administrateurs nommés par l’Assemblée générale<sup>8</sup>,
* un représentant de l’État français nommé par arrêté ministériel<sup>9</sup> et
* deux représentants des salariés dont un nommé par le Comité de Groupe Français et l’autre par le Comité d’entreprise européen<sup>10</sup>.

Au cours du premier semestre 2025, la composition du Conseil d’administration a évolué comme suit :

Ce tableau récapitule les mouvements (départs, nominations et renouvellements) au sein du Conseil d'administration survenus lors de l'Assemblée générale du 4 juin 2025.

|Départ|Nomination|Renouvellement|
|-|-|-|
|Anne-Marie Couderc (1)<br/>4 juin 2025|Isabelle Guichot (5)<br/>4 juin 2025|Gwenaelle Avice-Huet (9)<br/>4 juin 2025|
|Isabelle Bouillot (2)<br/>4 juin 2025|Véronique Penchienati-Bosetta (6)<br/>4 juin 2025|Leni Boeren (10)<br/>4 juin 2025|
|James Wang (3)<br/>4 juin 2025|Qingchao Wan (7)<br/>4 juin 2025|Delta Air Lines (représentée par Alain Bellemare) (11)<br/>4 juin 2025|
|Didier Dague (4)<br/>4 juin 2025|Pierre Lichon (8)<br/>4 juin 2025|Anne-Marie Idrac (12)<br/>4 juin 2025|
|||Terence Tilgenkamp (13)<br/>4 juin 2025|


<sup>(1)</sup> Le mandat d’administratrice de Mme Anne-Marie Couderc a pris fin à l’issue de l’Assemblée générale du 4 juin 2025.
<sup>(2)</sup> Le mandat d’administratrice de Mme Isabelle Bouillot a pris fin à l’issue de l’Assemblée générale du 4 juin 2025.
<sup>(3)</sup> Le mandat d’administrateur de M. James Wang a pris fin à l’issue de l’Assemblée générale du 4 juin 2025.
<sup>(4)</sup> Le mandat d’administrateur représentant les salariés de M. Didier Dague a pris fin à l’issue de l’Assemblée générale du 4 juin 2025.
<sup>(5)</sup> Mme Isabelle Guichot a été nommée en qualité d’administratrice par l’Assemblée générale d’Air France-KLM le 4 juin 2025 pour une durée de quatre ans.
<sup>(6)</sup> Mme Véronique Penchienati-Bosetta a été nommée en qualité d’administratrice par l’Assemblée générale d’Air France-KLM le 4 juin 2025 pour une durée de quatre ans.
<sup>(7)</sup> M. Qingchao Wan a été nommé en qualité d’administrateur par l’Assemblée générale d’Air France-KLM le 4 juin 2025 pour une durée de 4 ans.
<sup>(8)</sup> Le Comité de Groupe Français a nommé M. Pierre Lichon, le 10 avril 2025, en qualité d’administrateur représentant les salariés d’Air France – KLM, à compter de l’issue de l’Assemblée générale du 4 juin 2025, pour une durée de deux ans.
<sup>(9)</sup> Le mandat d’administratrice de Mme Gwenaelle Avice-Huet a été renouvelé par l’Assemblée générale d’Air France-KLM du 4 juin 2025 pour une durée de deux ans.
<sup>(10)</sup> Le mandat d’administratrice de Mme Leni Boeren a été renouvelé par l’Assemblée générale d’Air France-KLM du 4 juin 2025 pour une durée de quatre ans.
<sup>(11)</sup> Le mandat de Delta Air Lines, représentée par M. Alain Bellemare, a été renouvelé par l’Assemblée générale d’Air France-KLM du 4 juin 2025 pour une durée de quatre ans.
<sup>(12)</sup> Le mandat d’administratrice de Mme Anne-Marie Idrac a été renouvelé par l’Assemblée générale d’Air France-KLM du 4 juin 2025 pour une durée de deux ans.
<sup>(13)</sup> Le mandat de M. Terence Tilgenkamp en qualité d’administrateur représentant les salariés a été renouvelé par le Comité d’entreprise européen d’Air France – KLM le 27 mars 2025 à compter de l’issue de l’Assemblée générale du 4 juin 2025, pour une durée de deux ans.

### Évolution de la gouvernance : nomination de Mme Florence Parly à la Présidence du Conseil d’administration d’Air France-KLM

Conformément à la décision du Conseil d’administration d’Air France-KLM lors de sa réunion du 29 avril 2025, et sur proposition du Comité de nomination et de gouvernance, Mme Florence Parly a été nommée Présidente du Conseil d’administration d’Air France-KLM. Elle succède ainsi à Mme Anne-Marie Couderc, Présidente du Conseil d’administration d’Air France-KLM depuis mai 2018 dont le mandat arrivait à échéance.

Cette nomination a pris effet à l’issue de l’Assemblée générale du 4 juin 2025.

***

<sup>8</sup> Dont deux administrateurs nommés sur proposition de l'État français, un administrateur nommé sur proposition de l’État néerlandais, un administrateur nommé sur proposition de China Eastern Airlines et deux administrateurs représentant les salariés actionnaires.
<sup>9</sup> Conformément à l’article 4 de l’ordonnance n° 2014 - 948 du 20 août 2014 relative à la gouvernance et aux opérations sur le capital des sociétés à participation publique.
<sup>10</sup> En application des dispositions des articles L. 22-10-7 et L. 225-27-1 du Code de commerce et de l’article 17-3 des statuts.
# Composition du Conseil d'administration au 30 juin 2025

Ce tableau présente la composition détaillée du Conseil d'administration d'Air France-KLM au 30 juin 2025. Il regroupe les informations personnelles (genre, nationalité, âge, actions détenues), l'expérience (nombre de mandats), les détails du mandat au sein du conseil (dates d'entrée et d'échéance, ancienneté) ainsi que la participation aux différents comités (audit, rémunération, nomination et gouvernance, développement durable et conformité). Les administrateurs sont classés par mode de nomination, et les administrateurs indépendants sont signalés par un symbole spécifique.

|AdministrateursADMINISTRATEURS ÉLUS PAR L'ASSEMBLÉE GÉNÉRALE|Informations personnelles<br/>Genre<br/>ADMINISTRATEURS ÉLUS PAR L'ASSEMBLÉE GÉNÉRALE|Informations personnelles<br/>Nationalité<br/>ADMINISTRATEURS ÉLUS PAR L'ASSEMBLÉE GÉNÉRALE|Informations personnelles<br/>Âge<br/>ADMINISTRATEURS ÉLUS PAR L'ASSEMBLÉE GÉNÉRALE|Informations personnelles<br/>Nombre d'actions détenues<br/>ADMINISTRATEURS ÉLUS PAR L'ASSEMBLÉE GÉNÉRALE|Expérience<br/>Nombre de mandat dans des sociétés cotées<br/>ADMINISTRATEURS ÉLUS PAR L'ASSEMBLÉE GÉNÉRALE|Position au sein du conseil<br/>Date d'entrée<br/>ADMINISTRATEURS ÉLUS PAR L'ASSEMBLÉE GÉNÉRALE|Position au sein du conseil<br/>Date d'échéance du mandat<br/>ADMINISTRATEURS ÉLUS PAR L'ASSEMBLÉE GÉNÉRALE|Position au sein du conseil<br/>Ancienneté au Conseil<br/>ADMINISTRATEURS ÉLUS PAR L'ASSEMBLÉE GÉNÉRALE|Participation à des Comités<br/>Comité d'audit<br/>ADMINISTRATEURS ÉLUS PAR L'ASSEMBLÉE GÉNÉRALE|Participation à des Comités<br/>Comité de rémunération<br/>ADMINISTRATEURS ÉLUS PAR L'ASSEMBLÉE GÉNÉRALE|Participation à des Comités<br/>Comité de nomination et de gouvernance<br/>ADMINISTRATEURS ÉLUS PAR L'ASSEMBLÉE GÉNÉRALE|Participation à des Comités<br/>Comité de développement durable et de conformité<br/>ADMINISTRATEURS ÉLUS PAR L'ASSEMBLÉE GÉNÉRALE|
|-|-|-|-|-|-|-|-|-|-|-|-|-|
|° Florence Parly|Femme|Française|62|110|3|07/12/2023|AG 2026|1 an|||▲ (Présidente)||
|Benjamin Smith|Homme|Canadienne|53|56 891|1|05/12/2018|AG 2027|6 ans|||||
|° Gwenaëlle Avice-Huet|Femme|Française|45|350|2|26/05/2021|AG 2027|4 ans|▲|▲|||
|° Leni M.T. Boeren|Femme|Néerlandaise|61|1 600|1|16/05/2017|AG 2029|8 ans|▲|||▲|
|° Isabelle Guichot|Femme|Française|60|100|2|04/06/2025|AG 2029|26 jours|▲ (Présidente)|▲|||
|Delta Air Lines, Inc. (représentée par Alain Bellemare)||Américaine||7 340 118|2|03/10/2017|AG 2029|7 ans||▲|||
|Wiebe Draijer|Homme|Néerlandaise|59|110|1|05/06/2024|AG 2028|1 an||||▲|
|Dirk Jan van den Berg|Homme|Néerlandaise|71|400|1|26/05/2020|AG 2028|5 ans||||▲|
|° Anne-Marie Idrac|Femme|Française|72|100|1|02/11/2017|AG 2027|7 ans||||▲ (Présidente)|
|° Véronique Penchienati-Bosetta|Femme|Française|58|500|1|04/06/2025|AG 2029|26 jours|||||
|Qingchao Wan|Homme|Chinoise|53|110|2|04/06/2025|AG 2029|26 jours|||||
|° Alexander R. Wynaendts|Homme|Néerlandaise|64|100|2|19/05/2016|AG 2028|9 ans||▲ (Président)|▲||
|ADMINISTRATEURS ÉLUS PAR L'ASSEMBLÉE GÉNÉRALE SUR PROPOSITION DE L'ÉTAT|||||||||||||
|Yann Leriche|Homme|Française|52|N/A|2|07/06/2023|AG 2027|2 ans|||||
|Pascal Bouchiat|Homme|Française|65|N/A|2|03/04/2022|AG 2027|3 ans||||▲|
|ADMINISTRATEURS ÉLUS PAR L'ASSEMBLÉE GÉNÉRALE REPRÉSENTANTS DES SALARIÉS ACTIONNAIRES|||||||||||||
|Nicolas Foretz|Homme|Française|46|319|1|27/07/2023|AG 2026|2 ans|▲||||
|Michel Delli-Zotti|Homme|Française|60|777|1|24/05/2022|AG 2026|2 ans|▲||||
|ADMINISTRATEUR REPRÉSENTANT DE L'ÉTAT NOMMÉ PAR ARRÊTÉ MINISTÉRIEL|||||||||||||
|Céline Fornaro|Femme|Française|47|N/A|4|09/10/2023|AG 2027|2 ans|▲|▲|▲||
|ADMINISTRATEUR REPRÉSENTANT LES SALARIÉS NOMMÉ PAR LE COMITÉ DE GROUPE FRANÇAIS|||||||||||||
|Pierre Lichon|Homme|Française||N/A|1|04/06/2025|AG 2027|26 jours|||||
|ADMINISTRATEUR REPRÉSENTANT LES SALARIÉS NOMMÉ PAR LE COMITÉ D'ENTREPRISE EUROPÉEN|||||||||||||
|Terence Tilgenkamp|Homme|Néerlandaise|42|N/A|1|01/12/2021|AG 2027|3 ans||▲|||


° Administrateurs indépendants.
### 1.2.2 CEO Committee

Le CEO Committee est dirigé par le Directeur général d’Air France-KLM, M. Benjamin Smith et comprend trois autres membres qui sont directement rattachés à M. Smith :
* Mme Anne Rigail, Directrice générale d’Air France,
* Mme Marjan Rintel, Directrice générale et Présidente du Directoire de KLM et
* M. Steven Zaat, Directeur général adjoint Économie et Finances d’Air France-KLM.

Le CEO Committee est chargé de déterminer l’orientation stratégique de l’ensemble des compagnies aériennes et unités opérationnelles du Groupe.

### 1.2.3 Comité exécutif Groupe

Présidé par le Directeur général d’Air France-KLM, le Comité exécutif Groupe est composé de onze (11) membres et d’un secrétaire du Comité exécutif Groupe.

**Composition du Comité exécutif du Groupe au 30 juin 2025 :**

Ce tableau présente les membres du Comité exécutif au 30 juin 2025, en précisant leur âge ainsi que leur expérience professionnelle (secteur et nombre d'années) en lien avec leur fonction actuelle.

|Membres au 30 juin 2025|Âge au 30 juin 2025|Expérience professionnelleen lien avec la fonction<br/>Secteur|Expérience professionnelleen lien avec la fonction<br/>Expérience|
|-|-|-|-|
|**Benjamin Smith**<br/>Directeur général d’Air France-KLM|53 ans|Transport aérien|34 ans|
|**Marjan Rintel**<br/>Directrice générale et Présidente du Directoire de KLM|58 ans|Transport aérien|25 ans|
|**Anne Rigail**<br/>Directrice générale d’Air France|56 ans|Transport aérien|33 ans|
|**Steven Zaat**<br/>Directeur général adjoint Finances, Air France-KLM|55 ans|Transport aérien|22 ans|
|**Alexandre Boissy**<br/>Secrétaire général d’Air France-KLM|51 ans|Transport aérien|21 ans|
|**Anne Brachet**<br/>Directrice générale adjointe Engineering & Maintenance, Air France-KLM|61 ans|Transport aérien|29 ans|
|**Oltion Carkaxhija**<br/>Directeur général adjoint Stratégie et Transformation, Air France-KLM|48 ans|Transport aérien|18 ans|
|**Angus Clarke** (1)<br/>Directeur général adjoint et Chief Commercial Officer, Air France-KLM|50 ans|Transport aérien|23 ans|
|**Adriaan Den Heijer**<br/>Directeur général adjoint Cargo, Air France-KLM|55 ans|Transport aérien|30 ans|
|**Pierre-Olivier Bandet**<br/>Directeur général adjoint Systèmes d’Information, Air France-KLM|57 ans|Informatique<br/>Transport aérien|6 ans<br/>28 ans|
|**Henri de Peyrelongue**<br/>Directeur général adjoint Marketing, Air France-KLM|61 ans|Transport aérien|34 ans|


(1) Comme annoncé dans un communiqué de presse du 11 juillet 2025, M. Angus Clarke, Directeur général adjoint et Chief Commercial Officer d’Air France-KLM, quittera ses fonctions au sein du Groupe à compter du 31 août 2025.

Le secrétariat du Comité exécutif Groupe est assuré par le Directeur de cabinet du Directeur général d’Air France-KLM.

### 1.2.4 Bourse et actionnariat

#### Le capital social

Les actions sont entièrement libérées sous forme nominative ou au porteur au choix du titulaire. Depuis le 3 avril 2016, en application de la loi, les actionnaires détenant leurs actions au nominatif depuis plus de deux ans bénéficient d’un droit de vote double. Il n’existe pas de droits particuliers attachés aux actions. Par ailleurs, il n’existe pas de titres non représentatifs de capital.

Au 30 juin 2025, le capital d’Air France-KLM se compose de 262 769 869 actions d’une valeur nominale d’un euro.

|Exercice clos le|Montant du capital (en euros)|Nombre d'actions|Nombre de droits de vote théoriques|Nombre de droits de vote exerçables|
|-|-|-|-|-|
|30 juin 2024|262 769 869|262 769 869|360 723 054|360 482 418|
|30 juin 2025|262 769 869|262 769 869|370 049 931|369 790 085|


## L’actionnariat d’Air France-KLM

Le tableau ci-dessous présente l’évolution de l’actionnariat de la Société au 30 juin 2025 par rapport au 31 décembre 2024 :

### Résumé de l'actionnariat
Ce tableau détaille la répartition du capital et des droits de vote (exerçables et théoriques) entre les principaux actionnaires (État français, État néerlandais, CMA CGM, China Eastern Airlines, etc.) au 30 juin 2025 comparé au 31 décembre 2024. On note une stabilité globale du capital, avec l'État français restant le premier actionnaire à 28,0 %.

||% du capital<br/>30/06/2025|% du capital<br/>31/12/2024|% des droits de votes exerçables (1)<br/>30/06/2025|% des droits de votes exerçables (1)<br/>31/12/2024|% des droits de votes théoriques (2)<br/>30/06/2025|% des droits de votes théoriques (2)<br/>31/12/2024|
|-|-|-|-|-|-|-|
|Nombre d'actions ou droits de vote|262 769 869|262 769 869|369 790 085|360 482 263|370 049 931|360 705 547|
|État français|28,0 %|28,0 %|29,3 %|27,5 %|29,3 %|27,5 %|
|État néerlandais|9,1 %|9,1 %|13,0 %|13,3 %|13,0 %|13,3 %|
|CMA CGM|8,8 %|8,8 %|12,5 %|12,8 %|12,5 %|12,8 %|
|China Eastern Airlines (3)|4,6 %|4,6 %|6,5 %|6,7 %|6,5 %|6,7 %|
|Salariés (FCPE)|3,0 %|3,1 %|2,9 %|3,0 %|2,9 %|3,0 %|
|Delta Air Lines, Inc (4)|2,8 %|2,8 %|4,0 %|4,1 %|4,0 %|4,1 %|
|Spaak|0,9 %|0,9 %|1,2 %|1,2 %|1,2 %|1,2 %|
|Auto contrôle (5)|— %|— %|— %|— %|— %|— %|
|Autres|42,8 %|42,7 %|30,6 %|31,4 %|30,7 %|31,4 %|


(1) Les droits de vote exerçables ne comprennent pas les droits de vote attachés aux actions auto-détenues et autocontrôlées ou privées de droits de vote du fait d’une déclaration de franchissement de seuil tardive notamment.
(2) Le calcul des droits de vote théoriques prend en compte l’ensemble des droits de vote y compris les droits de vote doubles.
(3) Par l’intermédiaire de Eastern Airlines Industry Investment (Luxembourg) Company Limited.
(4) Agissant en qualité de general partner du partnership de droit néerlandais DAL Foreign Holdings, C.V. La société Delta Air Lines, Inc. contrôle DAL Foreign Holdings, C.V. et, en tant que general partner de DAL Foreign Holdings, C.V., est le porteur légal des actions Air France–KLM.
(5) Dont 128 994 actions auto-détenues au 30 juin 2024.

## Évolution boursière

L’action Air France–KLM est cotée à la bourse de Paris et d’Amsterdam (Euronext Paris et Amsterdam) sous le code ISIN FR001400J770. Elle figure dans le SBF 120. Depuis février 2008, le programme d’ADR (American Deposit Receipt) d’Air France–KLM est sur le marché hors cote OTC Pink Marketplace où il apparaît sous le code AFLYV. Le code Reuters du titre est AIRF.PA ou AIRF.AS et le code Bloomberg est AF FP.

Conformément à l’article 222-1 du Règlement général de l’Autorité des marchés financiers (l’AMF), Air France–KLM ayant son siège situé en France, son État membre d’origine, au sens de la Directive 2004/109/CE du 15 décembre 2004 telle que modifiée (la Directive Transparence), est la France et, en conséquence, l’autorité compétente pour le contrôle du respect de ses obligations en matière d’information réglementée est l’AMF.
Sur le premier semestre 2025, l'action Air France – KLM a augmenté de 14% :

Ce tableau présente les indicateurs boursiers de l'action Air France - KLM pour le premier semestre 2025 comparés au premier semestre 2024. Il détaille les cours extrêmes (haut et bas), le volume de titres en circulation ainsi que la capitalisation boursière totale en fin de période.

||Janvier-juin 2025|Janvier-juin 2024|
|-|-|-|
|Cours le plus haut (en euros)|12,16|13,48|
|Cours le plus bas (en euros)|7,05|8,23|
|Nombre de titres en circulation|262 769 869|262 769 869|
|Capitalisation boursière à la fin de la période (en milliards d'euros)|2,4|2,2|


# Sommaire

# 2.

Ce tableau présente le sommaire détaillé de la section 2 "RAPPORT FINANCIER" du document, incluant les commentaires sur la situation financière, les états financiers consolidés et les notes annexes, avec leurs numéros de page respectifs.

|Section|Titre|Page|
|-|-|-|
|2.|RAPPORT FINANCIER|34|
|2.1|COMMENTAIRES SUR LA SITUATION FINANCIÈRE|35|
|2.1.1|Résultats consolidés semestriels au 30 juin 2025|35|
|2.1.2|Investissements|38|
|2.1.3|Financement|39|
|2.1.4|Structure et profil de remboursement de la dette|40|
|2.1.5|Principaux ratios financiers du Groupe|40|
|2.1.6|Capitaux propres consolidés au 30 juin 2025|43|
|2.1.7|Résultats sociaux de la société Air France-KLM|43|
|2.2|ETATS FINANCIERS CONSOLIDÉS|44|
|2.2.1|Compte de résultat consolidé|44|
|2.2.2|Etat du résultat global consolidé|45|
|2.2.3|Bilan consolidé|46|
|2.2.4|Variation des capitaux propres consolidés|48|
|2.2.5|Tableau des flux de trésorerie consolidé|49|
|2.3|NOTES AUX ÉTATS FINANCIERS CONSOLIDÉS|50|


# 2.1 COMMENTAIRES SUR LA SITUATION FINANCIÈRE

## 2.1.1 Résultats consolidés au 30 juin 2025

### Périmètre au 30 juin 2025
Au 30 juin 2025, le périmètre comprend 85 sociétés consolidées, 22 sociétés mises en équivalence et une activité conjointe. Air France et KLM, les deux principales filiales, représentent 88 % du chiffre d’affaires et 65 % du bilan. Les autres filiales exercent principalement des activités de transport aérien (HOP!, KLM Cityhopper), de maintenance ou de transport loisirs (Transavia).

**Résumé des indicateurs clés :**
Le tableau suivant présente les principaux indicateurs financiers du Groupe Air France-KLM pour le premier semestre 2025 comparés au premier semestre 2024. On observe une croissance généralisée, avec un chiffre d'affaires en hausse de 1 005 millions d'euros (+6,9 %) et un retour à un résultat net positif de 401 millions d'euros contre une perte l'année précédente.

|(en millions d'euros)|30 juin 2025|30 juin 2024|Variation|
|-|-|-|-|
|Chiffres d'affaires|15 608|14 603|1 005|
|EBITDA courant|1 866|1 345|521|
|Résultat d'exploitation courant|409|24|385|
|Résultat des activités opérationnelles|397|(79)|476|
|Résultat net|401|(314)|715|
|Résultat net – part Groupe|314|(400)|714|
|Résultat net – part du Groupe par action de base (en euros)|1,09|(1,63)|2,72|


### Chiffre d’affaires
Le chiffre d’affaires consolidé de la période s’élève à 15,6 milliards d’euros (14,6 milliards d’euros en 2024), en hausse de 6,9 % comparé à l’année dernière. Le chiffre d’affaires est en hausse sur l’ensemble des activités. En effet, le chiffre d’affaires de l’activité Réseau a augmenté de 5,6 %, celui de Transavia de 12,8 % et enfin celui de l’activité Maintenance de 15,0 %.

En termes d’activité Air France-KLM a augmenté sa capacité globale de 4,0 % (SKO). Ainsi, le Groupe a augmenté ses capacités en termes de transport passagers de 2,7 % (SKO) et ses capacités en termes de transport de fret de 0,9 % (TKO). Les capacités de Transavia ont augmenté de 12,3 % (SKO).

Le premier semestre 2025 a été marqué par une augmentation de la recette unitaire hors change de 2,6% expliquée par une performance positive sur l’ensemble des Business: La performance du passage est à 2,8%, l’activité Transavia s’élève à 1,8 %, et enfin l’activité Cargo s’élève à 9,3%.

### Charges d’exploitation
Les charges d’exploitation ont augmenté de 4,3 % à 15,2 milliards d’euros, tandis que la capacité (SKO) a augmenté de 4,0 %.

Les charges externes ont augmenté de 2,4 % et s’établissent au premier semestre 2025 à (9,6) milliards d’euros contre (9,4) milliards d’euros l’année 2024.
Les charges externes se répartissent de la façon suivante :

### Résumé du tableau des charges externes
Ce tableau détaille la répartition des charges externes du Groupe au 30 juin 2025 comparé au 30 juin 2024. On observe une augmentation globale des charges de 2,4 % (passant de 9 385 M€ à 9 608 M€). Les postes les plus importants en volume sont le carburant (3 073 M€, en baisse), l'entretien aéronautique (1 824 M€, en forte hausse de 14,1 %) et les redevances aéronautiques (1 116 M€). La variation à change constant est globalement alignée avec la variation nominale (2,3 % contre 2,4 %).

|(en millions d'euros)|30 juin 2025|30 juin 2024|Variation (en %)|Variation à change constant (en %)|
|-|-|-|-|-|
|Carburant avions|(3 073)|(3 380)|(9,1)%|(7,4)%|
|SAF|(118)|(106)|11,3 %||
|Quotas de CO₂|(151)|(125)|20,8 %||
|Affrètements aéronautiques|(232)|(247)|(6,1)%|(6,5)%|
|Redevances aéronautiques|(1 116)|(976)|14,3 %|14,8 %|
|Commissariat|(471)|(434)|8,5 %|9,0 %|
|Achat d'assistance en escale|(1 041)|(974)|6,9 %|7,1 %|
|Achats et consommations d'entretien aéronautique|(1 824)|(1 598)|14,1 %|13,4 %|
|Frais commerciaux et de distribution|(568)|(553)|2,7 %|2,8 %|
|Autres frais|(1 014)|(992)|2,2 %|2,1 %|
|TOTAL|(9 608)|(9 385)|2,4 %|2,3 %|


Les principales variations sont les suivantes :
* **carburant aéronautique** : les charges de carburant de l'année ont reculé de (9,1) % par rapport à 2024, ce qui représente une diminution de (7,4 %) à change constant expliqué par une baisse du prix du pétrole ;
* **SAF** : ces charges correspondent aux SAF achetés notamment dans le cadre du EU SAF mandat français d'incorporation de SAF et des contrats de SAF proposés aux clients Entreprises pour leur trafic passage et cargo. L'augmentation résulte principalement du nouveau mandat UE ;
* **quotas de CO₂** : ces charges correspondent aux achats de quotas d'émission de CO₂ et sont en augmentation en raison de la hausse des capacités ;
* **affrètements aéronautiques** : les coûts engagés pour louer des capacités à d'autres compagnies aériennes ont diminué en 2025 ((6,1) %) ;
* **redevances aéronautiques** : les redevances aéronautiques sont versées dans le cadre de l'utilisation des espaces aériens et de l'utilisation des aéroports. Leur augmentation en 2025 (+14,3 %) est légèrement supérieure à celle de la capacité produite par le groupe en raison des hausses de tarifs appliquées dans certains aéroports ;
* **commissariat** : les dépenses de commissariat correspondent aux prestations fournies à bord des avions du groupe Air France-KLM pour son propre compte. Elles ont augmenté de +8,5 % par rapport à l'année dernière en raison des hausses de capacités et des pressions inflationnistes ;
* **achats d'assistance en escale** : les achats d'assistance en escale correspondent principalement aux frais d'assistance des avions au sol et à la prise en charge des passagers pour le Groupe et pour une faible part, pour le compte de clients tiers. La hausse de ce poste (+6,9 %) s'explique principalement par la hausse de capacité et par la tension inflationniste ;
* **achats d'entretien** : ils comprennent les achats et consommations d'entretien aéronautique, pour les avions du Groupe et pour l'activité tiers ; leur hausse est en lien avec l'augmentation de l'activité en interne et pour le compte de clients tiers ;
* **coûts commerciaux et de distribution** : ces coûts ont augmenté de +2,7 %, ce qui est inférieur à la capacité ;
* **autres charges** : les autres charges comprennent principalement les charges locatives, les frais de télécommunication, les charges d'assurances et charges d'honoraires.

**Les salaires et charges associées** sont en hausse de 5,9 % à 4,9 milliards d'euros, contre 4,6 milliards d'euros en 2024. Cette hausse est principalement expliquée par la capacité ainsi que par la hausse des salaires.

**Les impôts et taxes** se sont élevés à (102) millions d'euros en 2025 contre (96) millions en 2024, soit une hausse de 6,3 %.
Les autres produits et charges courants représentent un produit net de 835 millions d'euros au 30 juin 2025 contre 819 millions au 30 juin 2024. Ils comprennent :
* la production capitalisée pour un montant de 755 millions d'euros en 2025 contre 728 millions d'euros en 2024 ;
* le résultat de l'exploitation conjointe de lignes pour -19 millions d'euros en 2025 contre 3 millions d'euros en 2024 ;
* les couvertures de change, pour 5 millions d'euros en 2025 contre 25 millions en 2024.

## EBITDA courant

L'EBITDA courant s'élève à 1 866 millions d'euros au 30 juin 2025 (contre 1 345 millions d'euros au 30 juin 2024).

La contribution à l'EBITDA courant par secteur d'activité est la suivante :

**Résumé du tableau :** Ce tableau présente l'EBITDA courant par secteur d'activité (Réseau, Maintenance, Transavia, Autres) au 30 juin 2025 comparé au 30 juin 2024, ainsi que la variation en pourcentage. On note une forte progression globale de 38,7 %.

|(en millions d'euros)|30 juin 2025|30 juin 2024|% ch.|
|-|-|-|-|
|Réseau|1 513|1 088|39,1 %|
|Maintenance|338|232|45,7 %|
|Transavia|5|6|(16,7 %)|
|Autres|10|19|(47,4 %)|
|TOTAL|1 866|1 345|38,7 %|


## Amortissements, dépréciations et provisions

Les amortissements, dépréciations et provisions ressortent à (1 457) millions d'euros en 2025 contre (1 321) millions d'euros en 2024.

## Résultat d'exploitation courant

Le résultat d'exploitation courant représente un gain de 409 millions d'euros au 30 juin 2025 (contre 24 millions d'euros au 30 juin 2024).

La contribution au chiffre d'affaires et au résultat d'exploitation courant par secteur d'activité est la suivante :

**Résumé du tableau :** Ce tableau détaille le chiffre d'affaires et le résultat d'exploitation courant par secteur d'activité pour les périodes closes au 30 juin 2025 et 30 juin 2024. Le résultat d'exploitation total passe de 24 millions à 409 millions d'euros.

|(en millions d'euros)|30 juin 2025<br/>Chiffre d'affaires autres|30 juin 2025<br/>Résultat d'exploitation courant|30 juin 2024<br/>Chiffre d'affaires autres|30 juin 2024<br/>Résultat d'exploitation courant|
|-|-|-|-|-|
|Réseau|12 978|473|12 295|90|
|Maintenance|2 789|135|2 425|66|
|Transavia|1 473|(193)|1 305|(139)|
|Autres|149|(6)|144|7|
|TOTAL|15 607|409|14 603|24|


## Résultat des activités opérationnelles

Le résultat des activités opérationnelles est un produit de 397 millions d'euros au 30 juin 2025 contre une charge de (79) millions d'euros au 30 juin 2024.

En 2025, le résultat des activités opérationnelles inclut notamment l'impact des cessions-bail (« *Sales and Leaseback* ») pour 2 millions d'euros.

En 2024, le résultat des activités opérationnelles incluait notamment :
* l'impact des cessions-bail (« *Sales and Leaseback* ») pour (2) millions d'euros ;
* l'impact des autres cessions aéronautiques lié au refinancement d'un B777 chez KLM ayant généré un produit de 16 millions d'euros ;
* une indemnité de 115 millions d'euros à payer à Virgin dans le cadre de la renégociation d'un contrat.

Ces opérations sont décrites dans la Note 10 « Cession de matériels aéronautiques et autres produits et charges non courants » de l'annexe aux états financiers consolidés.
### Coût de l’endettement financier net
Le coût de l’endettement financier net a augmenté pour s’établir à (207) millions d’euros au 30 juin 2025 contre (144) millions d’euros au 30 juin 2024. Cette variation s’explique notamment par la diminution des produits de la trésorerie et équivalents de trésorerie pour 68 millions d’euros.

### Autres produits et charges financières
Les autres produits et charges financières nets représentent un produit de 398 millions d’euros en 2025 contre une charge de (213) millions d’euros en 2024 et correspondent essentiellement à un gain de change pour un montant de 555 millions d’euros en 2025 contre une perte de change de (47) millions en 2024 et à l’impact du taux utilisé pour désactualiser les passifs et provisions de restitution pour avions loués.

### Résultat net – part du Groupe
L’impôt représente une charge de (176) millions d’euros en 2025 contre un produit de 119 millions d’euros en 2024.

La part dans les résultats des sociétés mises en équivalence représente une charge de (11) millions d’euros en 2025 contre un produit 3 millions d’euros en 2024. Il s’agit principalement du résultat de SAS AB ainsi que du groupe Servair, des partenariats dans l’activité Maintenance et dans l’activité Autres du Groupe.

Le résultat net consolidé – part du Groupe est un gain de 314 millions d’euros en 2025 contre une perte de (400) millions d’euros en 2024.

La contribution au résultat net consolidé – part du Groupe par trimestre est respectivement de (292) millions d’euros au 31 mars 2025 et de 606 millions d’euros au 30 juin 2025.

Le résultat net de base par action s’élève à 1,09 euro au 30 juin 2025 contre (1,63) euro au 30 juin 2024.

## 2.1.2 Investissements

Ce tableau détaille les flux de trésorerie liés aux opérations d'investissement pour le premier semestre 2025 comparé au premier semestre 2024. On y note une augmentation des investissements corporels et incorporels (2 315 M€ contre 2 067 M€) et une hausse significative des produits de cessions d'immobilisations (573 M€ contre 373 M€). Le flux net total d'investissement s'établit à -1 642 M€ pour la période.

|(en millions d'euros)|30 juin 2025|30 juin 2024|
|-|-|-|
|Acquisition de filiales et participations avec prise de contrôle, achats de parts dans les sociétés non contrôlées|(11)|(3)|
|Investissements corporels et incorporels|(2 315)|(2 067)|
|Produits liés à la perte de contrôle de filiales ou à la cession de titres de sociétés non contrôlées|-|8|
|Produits de cessions d'immobilisations corporelles ou incorporelles|573|373|
|Intérêts reçus|88|156|
|Dividendes reçus|9|1|
|Diminution (augmentation) nette des placements de plus de 3 mois|14|131|
|FLUX DE TRÉSORERIE LIÉS AUX OPÉRATIONS D'INVESTISSEMENT|(1 642)|(1 401)|


Les investissements aéronautiques incluent les acomptes et soldes à la livraison des achats d’avion, les modifications capitalisables réalisées sur les avions, l’achat de pièces détachées et les coûts de maintenance capitalisables. Les investissements incorporels sont des achats de logiciels informatiques et la capitalisation des développements informatiques. Les autres investissements corporels incluent principalement les achats d’équipements industriels pour les opérations aériennes, la maintenance et l’informatique.

Au cours des six premiers mois de 2025, les investissements corporels et incorporels du groupe Air France-KLM se sont élevés à (2 315) millions d’euros et les produits de cession à 573 millions d’euros.
## 2.1.3 Financement

Ce tableau présente les flux de trésorerie liés aux activités de financement pour le premier semestre 2025 comparé au premier semestre 2024. On y observe notamment l'émission de titres subordonnés (obligations hybrides) pour 494 millions d'euros et un flux net de financement négatif de 1 364 millions d'euros.

|(en millions d'euros)|30 juin 2025|30 juin 2024|
|-|-|-|
|Paiement pour acquérir des actions d'autocontrôle|(1)|–|
|Émission de titres subordonnés|494|–|
|Coupons sur titres subordonnés|(65)|(62)|
|Émission de nouveaux emprunts|314|936|
|Remboursement d'emprunts|(1 152)|(1 260)|
|Paiements de dettes de loyers|(487)|(442)|
|Nouveaux prêts|(146)|(11)|
|Remboursement des prêts|87|56|
|Intérêts payés|(407)|(386)|
|Dividendes distribués|(1)|–|
|FLUX DE TRÉSORERIE LIÉS AUX ACTIVITÉS DE FINANCEMENT|(1 364)|(1 170)|


La principale opération suivante a eu lieu au cours du premier semestre 2025 impactant les flux de trésorerie liés aux activités de financement :

### Emission d'obligations hybrides pour un montant de 500 millions d'euros

Air France-KLM a placé le 15 mai 2025 500 millions d'euros d'obligations hybrides, au coupon fixe annuel de 5,75% (taux d'intérêt effectif de 5,875%) jusqu'à la première date de reset.

Elles sont comptabilisées en fonds propres dans les états financiers consolidés (voir Note 18.2 "Titres subordonnés à durée indéterminée" des notes aux états financiers consolidés).

Ainsi, au 30 juin 2025, les liquidités nettes du Groupe s'élèvent à 6,9 milliards d'euros, dont 1,0 milliard d'euros de valeurs de placement immobilisés ayant une maturité supérieure à trois mois et 1,1 milliard d'euros en obligations. En outre, le Groupe dispose de lignes de crédit à hauteur de 2,5 milliards d'euros disponibles au 30 juin 2025 (voir également la note 21 de l'annexe aux comptes consolidés).

La dette nette s'établit à 7,1 milliards d'euros (7,3 milliards d'euros au 31 décembre 2024). Le détail du calcul de la dette nette se trouve à la Note 22.3 de l'annexe aux états financiers consolidés.
### 2.1.4 Structure et profil de remboursement de la dette

#### Structure de la dette
Les dettes financières brutes du Groupe s'élèvent à 8,5 milliards d'euros au 30 juin 2025. La structure de la dette est la suivante :
* financements de marché (emprunts obligataires et emprunt subordonné perpétuel) : 3,1 milliards d'euros ;
* emprunts de location financement avec option d'achat : 3,9 milliards d'euros ;
* autres emprunts dont emprunts bancaires et intérêts courus non échus : 1,5 milliard d'euros.

#### Profil de remboursement de la dette et des titres subordonnés en millions d'euros <sup>(1)</sup>

Ce graphique présente l'échéancier de remboursement de la dette du Groupe par année, ventilé par type d'instrument financier (Obligations senior, Autres dettes long terme, et Obligations liées au développement durable).

|Année|Air France-KLM obligations senior|Autres dettes à long terme|Obligations liées au développement durable|Total|
|-|-|-|-|-|
|2025|500|400|0|900|
|2026|400|600|500|1500|
|2027|0|700|0|700|
|2028|0|550|500|1050|
|2029 et au-delà|650|3000|0|3650|


**Détails des instruments :**
* **Air France-KLM obligations senior**
    * juillet 2025 : AF titres Apollo (497 M€)
    * juillet 2026 : AFKL 3,875 % (282 M€)
    * décembre 2026 : AFKL 4,35 % 145 M$
    * mai 2029 : AFKL 4,625 % (650 M€)
* **Autres dettes à long terme émises par Air France et KLM**
    * Principalement sécurisées par leurs actifs
* **Obligations liées au développement durable**
    * 2026 : 7,250 % (500 M€)
    * 2028 : 8,125 % (500 M€)

<sup>(1) Excluant les paiements de dettes des locations opérationnelles, les emprunts perpétuels de KLM, les quasi-fonds propres perpétuels du Groupe Air France-KLM, les intérêts courus et accords de vente et de rachat de quotas de CO<sub>2</sub>.</sup>

### 2.1.5 Principaux ratios financiers du Groupe

#### Recette unitaire au SKO
Pour suivre la performance de chaque activité de transport en termes de recette, le Groupe divise la recette de cette activité par les capacités produites, exprimées en SKO pour le passage ou Transavia, et en TKO pour le cargo. Pour suivre la performance globale en termes de chiffre d'affaires au transport, le Groupe utilise la recette unitaire au SKO. Ce ratio est obtenu en divisant le chiffre d'affaires au transport par les capacités produites exprimées en siège-kilomètre offert (SKO). La capacité produite par les deux activités de transport passagers est calculée en sommant les capacités de l'activité passage (en SKO) et les capacités de Transavia (en SKO).

Ce tableau présente l'évolution de la recette unitaire au SKO entre juin 2024 et juin 2025, montrant une progression de l'indicateur de 8,49 à 8,70.

|||30 juin 2025|30 juin 2024|
|-|-|-|-|
|Chiffre d’affaires au transport (en M€)|A|13 942|13 077|
|Capacités produites exprimées en SKO|B|160 297|154 092|
|*RECETTE UNITAIRE AU SKO*|*C=A/B*|*8,70*|*8,49*|


#### Coût net au SKO
Pour suivre la performance de chaque activité de transport en termes de coût, le Groupe divise le coût net de cette activité par les capacités produites, exprimées en SKO pour le passage ou Transavia, et en TKO pour le cargo. Pour suivre la performance globale de l'entreprise en termes de coûts, le Groupe utilise le coût net au SKO. Ce coût net est obtenu en divisant le coût net total par les capacités produites exprimées en siège-kilomètre offert (SKO). Le coût net est calculé en retirant des coûts d'exploitation totaux le chiffre d'affaires autre que celui réalisé dans les trois activités de transport (passage, cargo, Transavia). La capacité produite par les deux activités de transport passagers est calculée en sommant les capacités de l'activité passage (en SKO) et les capacités de Transavia (en SKO).
Ce document présente l'analyse du coût net au SKO (Siège-Kilomètre Offert) ainsi que les ratios de couverture financière d'Air France-KLM pour le premier semestre 2025, comparés aux périodes précédentes. On y observe une légère baisse du coût net au SKO (-0,5% en brut) et une amélioration du ratio d'endettement (Dette nette/EBITDA courant passant de 1,73 à 1,50).

||Ref.|30 juin 2025|30 juin 2024|
|-|-|-|-|
|Coût d'exploitation total (en M€)|A|15 199|14 579|
|Quotas de CO2 (en M€)|B|151|125|
|Chiffre d'affaires autres (en M€)|C|1 665|1 525|
|Autres produits de l'activité (en M€)|D|1|1|
|Coût net excl ETS (en M€)|E=A-B-C-D|13 382|12 928|
|Capacités produites exprimées en SKO||160 297|154 092|
|Coût net/SKO (en cts d'€)||8,35|8,39|
|Variation brute|||-0,5%|
|Effet de change sur les coûts nets (en M€)|||4|
|Variation à change constant|||-0,5%|
|Effet prix du carburant (en M€)|||(369)|
|Variation à change et prix du carburant constants excl ETS|||2,4%|
|COÛT NET AU SKO À CHANGE ET PRIX DU CARBURANT CONSTANTS EXCL ETS||8,35|8,15|


## Ratios de couverture

Le premier tableau de cette section détaille le ratio de levier financier (Dette nette sur EBITDA), montrant une réduction de l'endettement relatif au 30 juin 2025 par rapport à la fin d'année 2024.

|Ratio dette nette/EBITDA courant|30 juin 2025|31 décembre 2024|
|-|-|-|
|Dette nette|7 134|7 332|
|EBITDA courant|4 764|4 244|
|DETTE NETTE/EBITDA Courant|1,50|1,73|


Le second tableau présente la capacité de l'entreprise à couvrir ses frais financiers par son excédent brut d'exploitation.

|Ratio EBITDA courant /coût de l'endettement financier net|30 juin 2025|30 juin 2024|
|-|-|-|
|EBITDA courant|4 764|3 940|
|Coût de l'endettement financier net|390|304|
|EBITDA courant /COÛT DE L'ENDETTEMENT FINANCIER NET|12,22|12,96|


## Retour sur capitaux employés (ROCE)

Le retour sur capitaux employés est un indicateur de rentabilité qui rapporte un résultat après impôt à la valeur des capitaux employés. La méthodologie de calcul est la suivante :

*   le calcul des capitaux employés s'appuie sur une méthode additive en identifiant les postes du bilan concerné. Les capitaux employés sur l'année sont obtenus en prenant la moyenne des capitaux employés sur chaque bilan trimestriel ;
*   le résultat ajusté après impôt correspond à la somme du résultat d'exploitation ajusté des dividendes reçus et de la part dans le résultat des entreprises mises en équivalence.
Ce document présente le calcul du **ROCE (Return on Capital Employed)** pour le groupe Air France-KLM sur deux périodes glissantes de 12 mois se terminant au 30 juin 2025 et au 30 juin 2024. Les tableaux détaillent les composantes des capitaux employés (immobilisations, droits d'utilisation, fonds de roulement, etc.) sur quatre trimestres consécutifs pour établir une moyenne, ainsi que le résultat ajusté après impôt. Au 30 juin 2025, le ROCE s'établit à **13,3%**, contre **11,6%** l'année précédente.

|(en millions d'euros)|30 juin 2025|31 mars 2025|31 décembre 2024|30 septembre 2024|
|-|-|-|-|-|
|Écart d'acquisition et immobilisations incorporelles|1 390|1 377|1 375|1 356|
|Immobilisations aéronautiques|13 392|12 835|12 347|12 607|
|Autres immobilisations corporelles|1 587|1 554|1 533|1 500|
|Droits d'utilisation|8 479|8 030|7 592|6 652|
|Titres mis en équivalence|205|212|216|240|
|Autres actifs financiers, hors titres disponibles à la vente, valeurs mobilières de placement et dépôts liés aux dettes financières|194|196|195|218|
|Provisions, hors retraites, litige cargo et restructuration|(5 167)|(5 246)|(5 224)|(4 553)|
|Fonds de roulement ⁽¹⁾|(8 749)|(8 984)|(7 468)|(7 422)|
|Capitaux employés sur le bilan|11 331|9 974|10 566|10 598|
|Capitaux employés moyens (A)||||10 617|
|Résultat d'exploitation courant||||1 985|
|Dividendes reçus||||(1)|
|Part dans les résultats des entreprises mises en équivalence||||(33)|
|(Charge)/produit d'impôt normatif||||(536)|
|Résultat ajusté après impôt (B)||||1 415|
|ROCE (B/A)||||13,3%|


*(1) A l'exclusion du report des charges sociales et fiscales accordé suite au Covid.*

|(en millions d'euros)|30 juin 2024|31 mars 2024|31 décembre 2023|30 septembre 2023|
|-|-|-|-|-|
|Écart d'acquisition et immobilisations incorporelles|1 354|1 349|1 352|1 331|
|Immobilisations aéronautiques|12 197|11 646|11 501|11 296|
|Autres immobilisations corporelles|1 456|1 438|1 431|1 379|
|Droits d'utilisation|6 479|5 902|5 956|5 596|
|Titres mis en équivalence|134|134|129|127|
|Autres actifs financiers, hors titres disponibles à la vente, valeurs mobilières de placement et dépôts liés aux dettes financières|211|214|219|191|
|Provisions, hors retraites, litige cargo et restructuration|(4 700)|(4 523)|(4 346)|(4 481)|
|Fonds de roulement ⁽¹⁾|(8 222)|(8 284)|(6 981)|(7 804)|
|Capitaux employés sur le bilan|8 909|7 876|9 261|7 635|
|Capitaux employés moyens (A)||||8 420|
|Résultat d'exploitation courant||||1 310|
|Dividendes reçus||||(1)|
|Part dans les résultats des entreprises mises en équivalence||||8|
|(Charge)/produit d'impôt normatif||||(340)|
|Résultat ajusté après impôt (B)||||977|
|ROCE (B/A)||||11,6%|


*(1) A l'exclusion du report des charges sociales et fiscales accordé suite au Covid.*
### 2.1.6 Capitaux propres consolidés au 30 juin 2025

Les capitaux propres consolidés totaux s’élèvent à 1 309 millions d’euros au 30 juin 2025, contre 799 millions d’euros au 31 décembre 2024. La hausse de 510 millions d’euros constatée sur la période s’explique principalement par :

* l’impact des titres subordonnés à durée indéterminée pour un montant total de (30) millions d’euros ;
* l’impact des coupons sur titres subordonnés à durée indéterminée pour un montant total de (27) millions d’euros (net de l’effet impôt) ;
* un profit sur les six premiers mois de l’année qui s’élève à 401 millions d’euros ;
* l’impact positif de 168 millions d’euros sur le résultat global principalement dû à la réévaluation des régimes de retraite à prestations définies et à la variation de la juste valeur des instruments financiers qualifiés de couverture.

### 2.1.7 Résultats sociaux de la société Air France-KLM

En qualité de société holding, la société Air France-KLM n’a pas d’activité opérationnelle. Ses produits proviennent des redevances perçues au titre de l’utilisation du logo Air France-KLM par les deux sociétés opérationnelles et des prestations de services facturées à Air France et KLM. Ses charges comprennent essentiellement les frais de communication financière, les honoraires des Commissaires aux comptes, les rémunérations des mandataires sociaux ainsi que le personnel mis à disposition par Air France et KLM.

Au 30 juin 2025, le résultat d’exploitation représente un bénéfice de 7 millions d’euros et le résultat net ressort positif à 145 millions d’euros essentiellement du fait des produits financiers sur les valeurs mobilières de placement, les gains de change latents et le boni d’intégration fiscale.
# 2.2 ÉTATS FINANCIERS CONSOLIDÉS

## 2.2.1 Compte de résultat consolidé

Ce tableau présente le compte de résultat consolidé d'Air France-KLM pour le premier semestre 2025 comparé au premier semestre 2024. Il détaille la formation du résultat net, passant du chiffre d'affaires (produits des activités ordinaires) au résultat net part du groupe, en intégrant les charges opérationnelles, financières et fiscales. On note un redressement significatif du résultat d'exploitation courant qui passe de 24 millions d'euros en 2024 à 409 millions d'euros en 2025.

|Période du 1ᵉʳ janvier au 30 juin(en millions d’euros)|Notes|2025|2024|
|-|-|-|-|
|Produits des activités ordinaires||15 608|14 603|
|Charges externes|6|(9 608)|(9 385)|
|Frais de personnel|7|(4 867)|(4 596)|
|Impôts et taxes hors impôt sur le résultat||(102)|(96)|
|Autres produits et charges d'exploitation courants|8|835|819|
|Amortissements, dépréciations et provisions|9|(1 457)|(1 321)|
|Charges opérationnelles||(15 199)|(14 579)|
|Résultat d’exploitation courant||409|24|
|Cessions de matériels aéronautiques|10|(2)|15|
|Autres produits et charges non courants|10|(10)|(118)|
|Résultat des activités opérationnelles||397|(79)|
|Charges d'intérêts|11|(309)|(314)|
|Produits de la trésorerie et équivalents de trésorerie|11|102|170|
|Coût de l’endettement financier net|11|(207)|(144)|
|Autres produits et charges financiers|11|398|(213)|
|Résultat avant impôts des entreprises intégrées||588|(436)|
|Impôt sur le résultat|12|(176)|119|
|Résultat net des entreprises intégrées||412|(317)|
|Part dans le résultat des entreprises mises en équivalence||(11)|3|
|Résultat net||401|(314)|
|Résultat net des participations ne donnant pas le contrôle||87|86|
|Résultat net : Propriétaires de la société mère||314|(400)|
|Résultat net – Propriétaires de la société mère par action (en euros)||||
|■ De base|13|1,09|(1,63)|
|■ Dilué|13|1,04|(1,63)|


Les Notes annexes font partie intégrante de ces états financiers consolidés.
## 2.2.2 État du résultat global consolidé

Ce tableau présente le passage du résultat net au résultat global pour les périodes closes au 30 juin 2025 et 2024. Il détaille les autres éléments du résultat global, distinguant ceux recyclables en résultat (couvertures, change) de ceux non recyclables (réévaluations de régimes de retraites). On note un redressement significatif du résultat global, passant d'une perte de 133 millions d'euros en 2024 à un profit de 569 millions d'euros en 2025.

**Période du 1ᵉʳ janvier au 30 juin**

|(en millions d'euros)|Notes|2025|2024|
|-|-|-|-|
|Résultat net||401|(314)|
|Partie efficace de la variation de juste valeur des couvertures et coût de couverture portée en autres éléments du résultat global||99|207|
|Variation de la juste valeur et coût de couverture transférée en résultat||75|(64)|
|Écart de change résultant de la conversion||(25)|7|
|Impôts différés sur les éléments recyclables du résultat global||(43)|(41)|
|Éléments recyclables du résultat global des sociétés mises en équivalence, nets d'impôts||21|–|
|Total des autres éléments recyclables du résultat global||127|109|
|Réévaluation des engagements nets sur les régimes à prestations définies|19|40|81|
|Juste valeur des instruments de capitaux propres réévalués par le résultat global||1|(4)|
|Impôts différés sur les éléments non recyclables du résultat global||3|(5)|
|Éléments non recyclables du résultat global des sociétés mises en équivalence, nets d'impôts||(3)|–|
|Total des autres éléments non recyclables du résultat global||41|72|
|Total des autres éléments du résultat global, après impôt||168|181|
|RÉSULTAT GLOBAL||569|(133)|
|■ Propriétaires de la société mère||482|(219)|
|■ Participations ne donnant pas le contrôle||87|86|


Les Notes annexes font partie intégrante de ces états financiers consolidés.
## 2.2.3 Bilan consolidé

### ACTIF

Ce tableau présente l'actif du bilan consolidé d'Air France-KLM au 30 juin 2025 comparé au 31 décembre 2024. Il détaille les actifs non courants (immobilisations, droits d'utilisation, impôts différés) et les actifs courants (stocks, créances, trésorerie). On note une augmentation du total de l'actif, passant de 36 155 millions d'euros à 38 347 millions d'euros, principalement portée par la hausse des immobilisations aéronautiques et des droits d'utilisation.

|ACTIF(en millions d'euros)|Notes|30 juin 2025|31 décembre 2024|
|-|-|-|-|
|Goodwill||223|226|
|Immobilisations incorporelles||1 167|1 150|
|Immobilisations aéronautiques|14|13 392|12 347|
|Autres immobilisations corporelles|14|1 587|1 533|
|Droits d'utilisation|16|8 479|7 592|
|Titres mis en équivalence||205|216|
|Actifs de retraite|19|56|66|
|Autres actifs financiers non courants||1 066|1 369|
|Actifs financiers dérivés non courants||118|195|
|Impôts différés|12|518|662|
|Autres actifs non courants||448|214|
|Actif non courant||27 259|25 570|
|Autres actifs financiers courants||1 464|1 190|
|Actifs financiers dérivés courants||57|249|
|Stocks et en-cours||993|959|
|Créances clients||2 404|2 051|
|Autres actifs courants||1 271|1 260|
|Trésorerie et équivalents de trésorerie|17|4 850|4 829|
|Actifs détenus en vue de la vente||49|47|
|Actif courant||11 088|10 585|
|TOTAL ACTIF||38 347|36 155|


Les Notes annexes font partie intégrante de ces états financiers consolidés.
## Bilan consolidé (suite)

### PASSIF ET CAPITAUX PROPRES

Ce tableau présente le passif et les capitaux propres du groupe Air France-KLM au 30 juin 2025 comparativement au 31 décembre 2024. Il détaille la structure des capitaux propres (part du groupe et participations ne donnant pas le contrôle), ainsi que la décomposition du passif non courant (dettes à long terme, provisions retraite) et du passif courant (titres de transport émis, dettes fournisseurs, etc.). On note une amélioration des capitaux propres totaux, passant de 799 millions d'euros à 1 309 millions d'euros sur la période.

|(en millions d’euros)|Notes|30 juin 2025|31 décembre 2024|
|-|-|-|-|
|Capital|18.1|263|263|
|Primes d’émission et de fusion||7 560|7 560|
|Actions d’autocontrôle||(27)|(27)|
|Titres subordonnés à durée indéterminée|18.2|1 554|1 078|
|Réserves et résultat||(10 166)|(10 638)|
|*Capitaux propres – Part attribuable aux propriétaires de la société mère*||*(816)*|*(1 764)*|
|Titres subordonnés à durée indéterminée|18.2|2 088|2 530|
|Réserves et résultat||37|33|
|*Capitaux propres – Participations ne donnant pas le contrôle*||*2 125*|*2 563*|
|**CAPITAUX PROPRES**||**1 309**|**799**|
|Provisions retraite|19|1 681|1 686|
|Passifs et provisions de restitution pour avions loués et autres provisions non courants|20|4 513|4 493|
|Passifs financiers non courants|21|6 512|7 254|
|Dettes de loyers non courantes|16|4 864|4 714|
|Passifs financiers dérivés non courants||292|32|
|Impôts différés|12|2|2|
|Autres passifs non courants|23|807|904|
|**Passif non courant**||**18 671**|**19 085**|
|Passifs et provisions de restitution pour avions loués et autres provisions courants|20|1 096|1 181|
|Passifs financiers courants|21|1 952|1 692|
|Dettes de loyers courantes|16|922|982|
|Passifs financiers dérivés courants||324|137|
|Dettes fournisseurs||2 516|2 608|
|Titres de transport émis et non utilisés||5 606|4 097|
|Programme de fidélisation||906|906|
|Autres passifs courants|23|5 015|4 668|
|Concours bancaires|17|30|–|
|**Passif courant**||**18 367**|**16 271**|
|**TOTAL PASSIF**||**37 038**|**35 356**|
|**TOTAL CAPITAUX PROPRES ET PASSIF**||**38 347**|**36 155**|


Les Notes annexes font partie intégrante de ces états financiers consolidés.
## 2.2.4 Variation des capitaux propres consolidés

Ce tableau présente l'évolution des capitaux propres consolidés d'Air France-KLM entre le 31 décembre 2023 et le 30 juin 2025. Il détaille la part attribuable aux propriétaires de la société mère (capital, primes, actions d'auto-contrôle, titres subordonnés, réserves) et les participations ne donnant pas le contrôle. On y observe notamment l'impact du résultat global, des coupons sur titres subordonnés et des émissions/remboursements de titres subordonnés sur la période.

|Part attribuable aux propriétaires de la société mère<br/>(en millions d'euros)|Part attribuable aux propriétaires de la société mère<br/>Nombre d'actions|Part attribuable aux propriétaires de la société mère<br/>Capital|Part attribuable aux propriétaires de la société mère<br/>Primes d'émission et de fusion|Part attribuable aux propriétaires de la société mère<br/>Actions d'auto-contrôle|Part attribuable aux propriétaires de la société mère<br/>Titres subordonnés à durée indéterminée|Part attribuable aux propriétaires de la société mère<br/>Réserves et résultats|Part attribuable aux propriétaires de la société mère<br/>Sous total|Participations ne donnant pas le contrôle<br/>Titres subordonnés à durée indéterminée|Participations ne donnant pas le contrôle<br/>Réserves et résultats|Participations ne donnant pas le contrôle<br/>Sous Total|Total<br/>Capitaux propres|
|-|-|-|-|-|-|-|-|-|-|-|-|
|31 décembre 2023|262 769 869|263|7 560|(25)|1 076|(10 925)|(2 051)|2 524|27|2 551|500|
|Autres éléments du résultat global|–|–|–|–|–|181|181|–|–|–|181|
|Résultat de la période|–|–|–|–|–|(400)|(400)|–|86|86|(314)|
|*Résultat global*|–|–|–|–|–|(219)|(219)|–|86|86|(133)|
|Paiement fondé sur des actions|–|–|–|–|–|1|1|–|–|–|1|
|Coupons sur titres subordonnés à durée indéterminée|–|–|–|–|(25)|(37)|(62)|83|(83)|–|(62)|
|Impôts sur coupons sur titres subordonnés à durée indéterminée|–|–|–|–|–|31|31|–|–|–|31|
|*30 juin 2024*|262 769 869|263|7 560|(25)|1 051|(11 149)|(2 300)|2 607|30|2 637|337|
|31 décembre 2024|262 769 869|263|7 560|(27)|1 078|(10 638)|(1 764)|2 530|33|2 563|799|
|Autres éléments du résultat global|–|–|–|–|–|168|168|–|–|–|168|
|Résultat de la période|–|–|–|–|–|314|314|–|87|87|401|
|*Résultat global*|–|–|–|–|–|482|482|–|87|87|569|
|Achat d'actions propres|–|–|–|(1)|–|–|(1)|–|–|–|(1)|
|Paiement fondé sur des actions|–|–|–|1|–|1|2|–|–|–|2|
|Dividendes payés|–|–|–|–|–|–|–|–|(1)|(1)|(1)|
|Titres subordonnés à durée indéterminée|–|–|–|–|500|(6)|494|(524)|–|(524)|(30)|
|Coupons sur titres subordonnés à durée indéterminée|–|–|–|–|(24)|(41)|(65)|83|(83)|–|(65)|
|Impôts sur coupons sur titres subordonnés à durée indéterminée|–|–|–|–|–|38|38|–|–|–|38|
|Autres|–|–|–|–|–|(2)|(2)|(1)|1|–|(2)|
|*30 juin 2025*|262 769 869|263|7 560|(27)|1 554|(10 166)|(816)|2 088|37|2 125|1 309|


Les Notes annexes font partie intégrante de ces états financiers consolidés.
## 2.2.5 Tableau des flux de trésorerie consolidé

Ce tableau présente les flux de trésorerie consolidés d'Air France-KLM pour le premier semestre 2025 comparé au premier semestre 2024. Il détaille la génération de trésorerie par les activités d'exploitation (3 027 M€ en 2025 contre 1 650 M€ en 2024), les flux liés aux investissements (-1 642 M€) et les activités de financement (-1 364 M€). On note une forte amélioration de la capacité d'autofinancement et une variation positive du besoin en fonds de roulement.

|Période du 1ᵉʳ janvier au 30 juin(en millions d’euros)|Notes|2025|2024|
|-|-|-|-|
|Résultat net||401|(314)|
|Dotations aux amortissements et provisions d’exploitation|9|1 457|1 321|
|Dotations nettes aux provisions financières|11|150|141|
|Coût de la dette nette|11|206|144|
|Résultat sur cessions d’actifs corporels et incorporels||2|(21)|
|Résultat sur cessions de filiales et participations|10|–|(2)|
|Résultats non monétaires sur instruments financiers||(2)|6|
|Écart de change non réalisé||(616)|28|
|Résultats des sociétés mises en équivalence||11|(3)|
|Impôts différés|12|103|(153)|
|Autres éléments non monétaires||18|17|
|*Flux de trésorerie liés à l'exploitation avant variation du besoin en fond de roulement*||1 730|1 164|
|Variation du besoin en fonds de roulement|24|1 297|486|
|**FLUX DE TRÉSORERIE LIÉS À L'EXPLOITATION**||**3 027**|**1 650**|
|Acquisition de filiales et participations avec prise de contrôle, achats de parts dans les sociétés non contrôlées||(11)|(3)|
|Investissements corporels et incorporels|15|(2 315)|(2 067)|
|Produits liés à la perte de contrôle de filiales ou à la cession de titres de sociétés non contrôlées||–|8|
|Produits de cessions d'immobilisations corporelles ou incorporelles|10|573|373|
|Intérêts reçus||88|156|
|Dividendes reçus||9|1|
|Diminution (augmentation) nette des placements de plus de 3 mois||14|131|
|**FLUX DE TRÉSORERIE LIÉS AUX OPÉRATIONS D'INVESTISSEMENT**||**(1 642)**|**(1 401)**|
|Paiement pour acquérir des actions d'autocontrôle||(1)|–|
|Acquisition d'intérêts minoritaires sans changement de contrôle||–|(1)|
|Émission de titres subordonnés|18.2|494|–|
|Coupons sur titres subordonnés|18.2|(65)|(62)|
|Émission de nouveaux emprunts|21|314|936|
|Remboursement d’emprunts|21|(1 152)|(1 260)|
|Paiements de dettes de loyers|16|(487)|(442)|
|Nouveaux prêts||(146)|(11)|
|Remboursement des prêts||87|56|
|Intérêts payés||(407)|(386)|
|Dividendes distribués||(1)|–|
|**FLUX DE TRÉSORERIE LIÉS AUX ACTIVITÉS DE FINANCEMENT**||**(1 364)**|**(1 170)**|
|Effets des variations de change sur la trésorerie, équivalents de trésorerie et les concours bancaires courants (nets de la trésorerie acquise ou cédée)||(30)|18|
|**Variation de la trésorerie nette**||**(9)**|**(903)**|
|**Trésorerie, équivalents de trésorerie et concours bancaires à l’ouverture**|**17**|**4 829**|**6 181**|
|**Trésorerie, équivalents de trésorerie et concours bancaires à la clôture**|**17**|**4 820**|**5 278**|


Les Notes annexes font partie intégrante de ces états financiers consolidés.
# 2.3 NOTES AUX ÉTATS FINANCIERS CONSOLIDÉS

Ce tableau présente le sommaire détaillé des notes annexes aux états financiers consolidés, classées par numéro de note, intitulé et numéro de page correspondant.

|Note|Description|Page|
|-|-|-|
|Note 1|Description de l'activité|51|
|Note 2|Événements significatifs|51|
|Note 3|Règles et méthodes comptables|53|
|Note 4|Évolution du périmètre de consolidation|54|
|Note 5|Informations sectorielles|54|
|Note 6|Charges externes|57|
|Note 7|Frais de personnel et effectifs|58|
|Note 8|Autres produits et charges d'exploitation courants|58|
|Note 9|Amortissements, dépréciations et provisions|59|
|Note 10|Cessions de matériels aéronautiques et autres produits et charges non courants|59|
|Note 11|Coût de l'endettement financier et autres produits et charges financiers|60|
|Note 12|Impôts|61|
|Note 13|Résultat net – part du Groupe par action|62|
|Note 14|Immobilisations corporelles|63|
|Note 15|Investissements corporels et incorporels|63|
|Note 16|Droits d'utilisation et dettes de loyers|63|
|Note 17|Trésorerie, équivalents de trésorerie et concours bancaires|64|
|Note 18|Capitaux propres|65|
|Note 19|Actifs et provisions retraites|67|
|Note 20|Passifs et provisions de restitution pour avions loués et autres provisions|67|
|Note 21|Passifs financiers|71|
|Note 22|Indicateurs alternatifs de performance|73|
|Note 23|Autres passifs|75|
|Note 24|Variation du fonds de roulement|75|
|Note 25|Commande de matériels aéronautiques|76|
|Note 26|Parties liées|76|


# NOTE 1 DESCRIPTION DE L'ACTIVITÉ

Le terme « Air France-KLM » utilisé ci-après fait référence à la société holding Air France-KLM S.A. régie par le droit français. Le terme « Groupe » concerne l’ensemble économique composé d’Air France-KLM et de ses filiales. Le Groupe dont le siège social est situé en France, constitue un des plus grands groupes aériens mondiaux.

Son activité principale Réseau se compose du transport aérien de passagers sur vols réguliers et du cargo. Les activités du Groupe incluent également la maintenance aéronautique, le transport de passagers sur vols « Loisirs » (Transavia) et toute autre activité en relation avec le transport aérien.

La société anonyme Air France-KLM, domiciliée au 7, rue du cirque 75008 Paris – France, est l’entité consolidante du groupe Air France-KLM. Air France-KLM est coté à Paris (Euronext) et Amsterdam (Euronext).

La monnaie de présentation du Groupe, qui est également la monnaie fonctionnelle d’Air France-KLM, est l’euro.

# NOTE 2 ÉVÉNEMENTS SIGNIFICATIFS

## 2.1 Événements significatifs intervenus au cours de la période

### Mesures visant à améliorer la performance opérationnelle et financière de KLM
Le 29 janvier 2025, KLM a annoncé la suppression de 250 emplois portant exclusivement sur des postes non opérationnels. Toutes les parties prenantes travaillent à un plan détaillé qui sera discuté avec les instances représentatives du personnel. Sans pouvoir le garantir à ce stade, KLM a pour objectif d’éviter les départs forcés.

Cet évènement n’a pas eu d’impact sur les états financiers consolidés de 2024 et 2025.

### Renforcement du partenariat commercial avec WestJet et acquisition de participations minoritaires dans le capital
Delta Air Lines et Korean Air (partenaires de co-entreprise d’Air France-KLM) prévoient de renforcer leurs partenariats respectifs avec WestJet grâce à l'acquisition de participations minoritaires dans le capital de la compagnie aérienne canadienne auprès de Onex Partners.

Dans le cadre de ces accords, Delta et Korean procéderont à l’acquisition de participations indépendantes représentant au total 25 % du capital de WestJet. Delta investira 330 millions de dollars américains pour acquérir une participation de 15 %, tandis que Korean investira 220 millions de dollars américains pour acquérir une participation de 10 %.

A la clôture de l’opération, Delta aura le droit et a l'intention de vendre et de transférer 2,3 % du capital de WestJet à Air France-KLM, également partenaire commercial de WestJet, pour 50 millions de dollars américains.

Cette transaction distincte reste soumise à certaines validations et n’a donc pas eu d’impact sur les états financiers consolidés au 30 juin 2025.

### Emission d’obligations hybrides pour un montant de 500 millions d’euros
Air France-KLM a placé le 15 mai 2025 500 millions d’euros d’obligations hybrides, au coupon fixe annuel de 5,75% (yield de 5,875%) jusqu’à la première date de reset.

Elles sont comptabilisées en capitaux propres dans les états financiers consolidés (voir Note 18.2 ‘Titres subordonnées à durée indéterminée”).

### Extension des lignes de crédit renouvelables liées à l’ESG

<u>Air France-KLM et Air France:</u>

En avril 2023, Air France-KLM et Air France, co-emprunteurs, avaient signé une ligne de crédit renouvelable liée au développement durable d’un montant de 1,2 milliard d’euros. Cette ligne incluait une option d’augmentation en accordéon qui avait été exercée sur le premier trimestre 2024 pour un montant de 90 millions d’euros portant ainsi le montant disponible à environ 1,3 milliard d’euros.

Cette ligne de crédit avait par ailleurs une échéance initiale à 2026 et comprenait deux options d’extension d’un an. En avril 2024, une option d’extension d’un an avait été levée portant l’échéance à 2027.

Enfin et le 18 juillet 2024, un nouvel amendement avait été signé sur la ligne de crédit d’Air France-KLM et d’Air France qui prévoyait :
* l’extension de l’échéance à juillet 2028 associée à une option d’extension complémentaire d’un an ;
* l’augmentation de la ligne de crédit de 1 290 millions d’euros à 1 405 millions d’euros.

Cette dernière option d’extension a été exercée en juin 2025 portant l’échéance à 2029.

<u>KLM:</u>

En avril 2023, KLM avait signé une ligne de crédit renouvelable d’un montant de 1,0 milliard d’euros liée à des indicateurs de performance ESG avec une échéance initiale à 2027 et comprenant deux options d’extension d’un an. En avril 2024, une première option d’extension d’un an avait été levée portant l’échéance à 2028. La deuxième option d’extension a été exercée en avril 2025 portant l’échéance à 2029.
### Notification de remboursement des titres super subordonnés 2022 d'un montant de 497 millions d'euros

Air France a notifié à la société Apollo le 20 juin 2025 sa décision (irrévocable) de procéder au remboursement le 28 juillet 2025 des titres super subordonnés émis en juillet 2022 pour 497 millions d'euros.

Par conséquence, ces titres ont été reclassés des titres subordonnés à durée indéterminée en passifs financiers courants au 30 juin 2025 pour un montant total de 524 millions d'euros (le nominal pour 497 millions d'euros et les intérêts courus non échus pour 27 millions d'euros). Se référer aux Notes 18.2 "Titres subordonnés à durée indéterminée" et 21 "Passifs financiers".

## 2.2 Événements postérieurs à la clôture

### Processus en vue d'une prise de participation majoritaire dans le capital de SAS

Le groupe Air France-KLM va entamer un processus visant à prendre une participation majoritaire dans le capital de SAS. Le Groupe détient actuellement 19,9 % du capital de la compagnie scandinave et a initié à l'été 2024 une coopération commerciale entre Air France, KLM et SAS, reposant sur des accords élargis de partage de codes et de commercialisation interline. Cette coopération a été renforcée par l'entrée de SAS dans l'alliance SkyTeam.

Sous réserve de la satisfaction de l'ensemble des conditions requises, Air France-KLM procéderait à l'acquisition de l'ensemble des parts détenues par Castlelake et Lind Invest, portant ainsi sa propre participation dans SAS à 60,5 %. L'État danois conserverait sa participation de 26,4 % ainsi que ses sièges au sein du Conseil d'administration.

Sous réserve de l'obtention des autorisations réglementaires requises et de la levée des conditions suspensives, l'ambition du Groupe serait de finaliser l'opération au deuxième semestre 2026.

L'acquisition de cette participation majoritaire permettrait à Air France-KLM de prendre le contrôle de SAS, qui deviendrait alors une filiale du Groupe. Le Groupe détiendrait la majorité des sièges au sein du Conseil d'administration de la compagnie aérienne.

Air France-KLM et SAS demeurent à date des concurrents sur le plan commercial.
# NOTE 3 RÈGLES ET MÉTHODES COMPTABLES

En application du règlement européen n°1606 / 2002 du 19 juillet 2002, les états financiers consolidés du groupe Air France-KLM au 31 décembre 2024 ont été établis conformément aux normes IFRS (International Financial Reporting Standards) telles qu'adoptées par l'Union européenne à la date de clôture de ces états financiers consolidés et qui étaient d'application obligatoire à cette date.

Les états financiers consolidés semestriels résumés au 30 juin 2025 ont été préparés en conformité avec la norme IAS 34 « Information financière intermédiaire ».

Les principes comptables appliqués pour les états financiers consolidés semestriels résumés au 30 juin 2025 sont conformes à ceux retenus pour les états financiers consolidés de l'exercice 2024, à l'exception des normes et interprétations adoptées par l'Union européenne applicables à compter du 1er janvier 2025.

Les amendements ou normes IFRS applicables de façon obligatoire à compter du 1<sup>er</sup> janvier 2025 sont considérés comme non applicables ou sans impact significatif sur les états financiers du groupe Air France - KLM.

Les états financiers consolidés semestriels résumés au 30 juin 2025 ont été arrêtés par le Conseil d'administration le 30 juillet 2025.

## Évolution du référentiel comptable

### IFRS 18, Présentation et informations à fournir dans les états financiers

Le 9 avril 2025, l'International Accounting Standards Board (IASB) a publié sa nouvelle norme IFRS 18 "Présentation et informations à fournir dans les états financiers" visant à améliorer l'utilité des informations présentées dans les états financiers primaire et les notes annexes permettant aux investisseurs de disposer d'informations plus transparentes et comparables.

IFRS 18 vient remplacer la norme IAS 1 "Présentation des états financiers" et amender d'autres normes, principalement la norme IAS 7 "Tableau des flux de trésorerie".

Les principaux changements concernent :
* l'amélioration de la comparabilité de l'état du compte de résultat en introduisant trois catégories distinctes de produits et de charges (exploitation, investissement et financement) et en imposant de nouveaux sous-totaux dont le résultat d'exploitation ;
* l'amélioration de la transparence des indicateurs de performance définies par la direction ;
* l'introduction de règles et d'indications sur la manière d'agréger et de désagréger l'information financière tant dans les états financiers primaires que dans les notes annexes.

La norme IFRS 18 entre en vigueur pour les exercices ouverts à compter du 1<sup>er</sup> janvier 2027 avec une application anticipée possible au 1<sup>er</sup> janvier 2026. Elle n'est toutefois pas encore adoptée par l'Union européenne.

Le Groupe évalue actuellement les impacts de cette nouvelle norme notamment sur la structure du compte de résultat, sur le tableau des flux de trésorerie et sur les mesures de performance définies par la direction et communiquées.

A ce stade, les autres évolutions du référentiel comptable prévues sont suivies par le Groupe qui ne s'attend pas à des impacts significatifs.
# NOTE 4 ÉVOLUTION DU PÉRIMÈTRE DE CONSOLIDATION

Le 1<sup>er</sup> février 2024, KLM avait cédé sa filiale détenue à 100 % KLM Equipment Services B.V. à TCR International N.V. (TCR).
Il n’y avait pas eu d’acquisition significative au cours du premier semestre 2024.
Aucune acquisition ni cession significative n’a eu lieu au cours de la période close le 30 juin 2025.

# NOTE 5 INFORMATIONS SECTORIELLES

## Information par secteur d’activité (Note 5.1)

L’information sectorielle est établie sur la base des données de gestion interne communiquées au Comité exécutif, principal décideur opérationnel du Groupe.

Le Groupe est organisé autour des secteurs suivants :
* **Réseau** : Les revenus de ce secteur qui comprend le passage réseau et le cargo proviennent essentiellement des services de transport de passagers sur vols réguliers ayant un code des compagnies aériennes du Groupe hors Transavia, ce qui inclut les vols opérés par d’autres compagnies aériennes dans le cadre de contrats de partage de codes. Ils incluent également les revenus des partages de codes, les recettes d’excédent de bagages, les revenus de l’assistance aéroportuaire fournie par le Groupe aux compagnies aériennes tierces et des services liés aux systèmes d’information, ainsi que les opérations de transport de marchandises réalisées sous code des compagnies aériennes du Groupe, incluant le transport effectué par des partenaires dans le cadre de contrat de partage de codes. Les autres recettes du cargo correspondent essentiellement à la vente de capacité à d’autres transporteurs et aux transports de marchandises effectués pour le Groupe par des compagnies aériennes tierces ;
* **Maintenance** : Les revenus externes proviennent des services de maintenance fournis à d’autres compagnies aériennes et clients dans le monde ;
* **Transavia** : Les revenus de ce secteur proviennent de l’activité de transport « loisir » de passagers réalisée par Transavia ;
* **Autres** : Les revenus de ce secteur proviennent de diverses prestations fournies par le Groupe, non couvertes par les trois autres secteurs précités.

Les résultats alloués aux secteurs d’activité correspondent à ceux qui sont affectables de façon directe ou qui peuvent être alloués de façon raisonnable à ces segments d’activité. Les montants répartis dans les secteurs d’activité correspondent principalement à l’EBITDA courant, au résultat d’exploitation courant et au résultat des activités opérationnelles. Les autres éléments du compte de résultat sont regroupés dans la colonne « non répartis ».

Les transactions intersecteurs sont effectuées et valorisées à des conditions normales de marché.

## Information par secteur géographique (Note 5.2)

### Activité par zone de destination

Le chiffre d’affaires externe du transport aérien du Groupe par zone de destination est ventilé en sept secteurs géographiques :
* France métropolitaine ;
* Europe (hors France) et Afrique du Nord ;
* Antilles, Caraïbes, Guyane et Océan indien ;
* Afrique (hors Afrique du Nord), Moyen-Orient ;
* Amérique du Nord, Mexique et Polynésie française ;
* Amérique du Sud (hors Mexique) ;
* Asie et Nouvelle Calédonie.
## 5.1 Informations par secteur d’activité

**Période close au 30 juin 2025**

Ce tableau présente la ventilation de la performance financière du groupe Air France-KLM par secteur d'activité (Réseau, Maintenance, Transavia, Autres) pour le premier semestre 2025. Il détaille la formation du chiffre d'affaires, l'EBITDA courant, le résultat d'exploitation et le résultat net consolidé après prise en compte des ajustements de consolidation et des éléments non répartis (frais financiers et impôts).

|(en millions d'euros)|Notes|Réseau|Maintenance|Transavia|Autres|Non-répartis|Ajustements de consolidation|Total|
|-|-|-|-|-|-|-|-|-|
|Chiffre d’affaires transport de passagers|5.2|11 441|–|1 507|–|–|–|12 948|
|Chiffre d'affaires transport de fret|5.2|994|–|–|–|–|–|994|
|Chiffre d'affaires autres ⁽¹⁾||530|1 153|(35)|17|–|–|1 665|
|Chiffre d'affaires intersecteurs||13|1 636|1|132|–|(1 782)|–|
|*Chiffre d'affaires du segment*||*12 978*|*2 789*|*1 473*|*149*|*–*|*(1 782)*|*15 607*|
|Autres produits de l’activité||–|–|–|1|–|–|1|
|*Produits des activités ordinaires*||*12 978*|*2 789*|*1 473*|*150*|*–*|*(1 782)*|*15 608*|
|Carburant avions et SAF||(2 833)|(1)|(358)|–|–|–|(3 192)|
|Frais de Personnel||(3 431)|(638)|(404)|(403)|–|9|(4 867)|
|Autres||(5 201)|(1 812)|(706)|263|–|1 773|(5 683)|
|*EBITDA Courant*|22|*1 513*|*338*|*5*|*10*|*–*|*–*|*1 866*|
|Amortissement, dépréciations et provisions||(1 041)|(203)|(199)|(14)|–|–|(1 457)|
|*Résultat d'exploitation courant*||*473*|*135*|*(193)*|*(6)*|*–*|*–*|*409*|
|Résultat des activités opérationnelles||463|134|(194)|(6)|–|–|397|
|Part dans les résultats des entreprises mises en équivalence||2|(8)|–|(5)|–|–|(11)|
|Coût de l’endettement financier net et autres produits et charges financiers||–|–|–|–|191|–|191|
|Impôts||–|–|–|–|(176)|–|(176)|
|*RÉSULTAT NET*||*465*|*126*|*(194)*|*(11)*|*15*|*–*|*401*|


(1) Cette ligne inclut les indemnisations versées aux clients conformément au règlement CE261.
**Période close au 30 juin 2024**

Ce tableau présente la ventilation sectorielle des résultats financiers d'Air France-KLM pour le premier semestre 2024. Il détaille le chiffre d'affaires, l'EBITDA courant et le résultat net par segment d'activité : Réseau, Maintenance, Transavia et Autres, tout en incluant les éléments non répartis et les ajustements de consolidation. On note un chiffre d'affaires total de 14 603 millions d'euros pour un résultat net global négatif de 314 millions d'euros.

|(en millions d'euros)|Notes|Réseau|Maintenance|Transavia|Autres|Non-répartis|Ajustements|Total|
|-|-|-|-|-|-|-|-|-|
|Chiffre d’affaires transport de passagers|5.2|10 856|–|1 318|–|–|–|12 174|
|Chiffre d'affaires transport de fret|5.2|903|–|–|–|–|–|903|
|Chiffre d'affaires autres ⁽¹⁾||524|1 001|(17)|17|–|–|1 525|
|Chiffre d'affaires intersecteurs||12|1 424|4|126|–|(1 566)|–|
|*Chiffre d'affaires du segment*||*12 295*|*2 425*|*1 305*|*143*|*–*|*(1 566)*|*14 602*|
|Autres produits de l’activité||–|–|–|1|–|–|1|
|*Produits des activités ordinaires*||*12 295*|*2 425*|*1 305*|*144*|*–*|*(1 566)*|*14 603*|
|Carburant avions et SAF||(3 114)|(2)|(370)|–|–|–|(3 486)|
|Frais de Personnel||(3 278)|(591)|(345)|(390)|–|8|(4 596)|
|Autres||(4 815)|(1 600)|(583)|268|–|1 557|(5 173)|
|*EBITDA Courant*|22|*1 088*|*232*|*6*|*19*|*–*|*–*|*1 345*|
|Amortissements, dépréciations et provisions||(998)|(166)|(145)|(12)|–|–|(1 321)|
|*Résultat d'exploitation courant*||*90*|*66*|*(139)*|*7*|*–*|*–*|*24*|
|Résultat des activités opérationnelles||(14)|65|(139)|9|–|–|(79)|
|Part dans les résultats des entreprises mises en équivalence||1|2|–|–|–|–|3|
|Coût de l’endettement financier net et autres produits et charges financiers||–|–|–|–|(357)|–|(357)|
|Impôts||–|–|–|–|119|–|119|
|*RÉSULTAT NET*||*(13)*|*67*|*(139)*|*9*|*(238)*|*–*|*(314)*|


(1) Cette ligne inclut les indemnisations versées aux clients conformément au règlement CE261.
## 5.2 Informations par secteur géographique

### Activité par zone de destination

**CHIFFRE D’AFFAIRES EXTERNE DU TRANSPORT AÉRIEN PAR DESTINATION**

Ce tableau présente la ventilation du chiffre d'affaires externe du transport aérien par zone géographique de destination pour les périodes closes au 30 juin 2025 et 2024. Les données sont réparties par activité (Passage, Cargo, Transavia) et par région (France, Europe, Antilles/Caraïbes, Afrique/Moyen-Orient, Amériques et Asie/Nouvelle-Calédonie).

**Période close au 30 juin 2025**

|(en millions d'euros)|Notes|France métropolitaine|Europe (hors France) Afrique du Nord|Antilles Caraïbes Guyane Océan Indien|Afrique (hors Afrique du Nord) Moyen-Orient|Amérique du Nord, Mexique Polynésie française|Amérique du Sud, hors Mexique|Asie Nouvelle-Calédonie|Total|
|-|-|-|-|-|-|-|-|-|-|
|Passage|5.1|545|2 735|899|1 365|3 001|1 185|1 711|11 441|
|Cargo|5.1|108|192|19|133|144|120|278|994|
|Transavia|5.1|61|1 336|–|110|–|–|–|1 507|
|TOTAL TRANSPORT||714|4 263|918|1 608|3 145|1 305|1 989|13 942|


**Période close au 30 juin 2024**

|(en millions d'euros)|Notes|France métropolitaine|Europe (hors France) Afrique du Nord|Antilles Caraïbes Guyane Océan Indien|Afrique (hors Afrique du Nord) Moyen-Orient|Amérique du Nord, Mexique Polynésie française|Amérique du Sud, hors Mexique|Asie Nouvelle-Calédonie|Total|
|-|-|-|-|-|-|-|-|-|-|
|Passage|5.1|543|2 592|895|1 383|2 733|1 095|1 615|10 856|
|Cargo|5.1|101|144|14|152|128|144|220|903|
|Transavia|5.1|62|1 154|–|102|–|–|–|1 318|
|TOTAL TRANSPORT||706|3 890|909|1 637|2 861|1 239|1 835|13 077|


## NOTE 6 CHARGES EXTERNES

Ce tableau détaille les charges externes du groupe pour le premier semestre 2025 comparé au premier semestre 2024. Les postes principaux incluent le carburant, l'entretien aéronautique et les redevances.

**Période du 1ᵉʳ janvier au 30 juin**

|(en millions d'euros)|2025|2024|
|-|-|-|
|Carburant avions|(3 073)|(3 380)|
|SAF|(118)|(106)|
|Quotas de CO₂|(151)|(125)|
|Affrètements aéronautiques|(232)|(247)|
|Redevances aéronautiques|(1 116)|(976)|
|Commissariat|(471)|(434)|
|Achat d'assistance en escale|(1 041)|(974)|
|Achats et consommations d'entretien aéronautique|(1 824)|(1 598)|
|Frais commerciaux et de distribution|(568)|(553)|
|Autres frais|(1 014)|(992)|
|TOTAL|(9 608)|(9 385)|


Une partie des charges externes (notamment les coûts de carburant avions, achats et consommations d’entretien aéronautique) est soumise à la variation du cours du dollar US. Les couvertures associées sont présentées en Note 8 « Autres produits et charges d'exploitation courants ».
# NOTE 7 FRAIS DE PERSONNEL ET EFFECTIFS

Ce tableau détaille les charges de personnel pour le premier semestre 2025 comparativement à 2024. On observe une augmentation globale des frais de personnel, portés principalement par les salaires et traitements, malgré une légère baisse des coûts du personnel intérimaire.

|Période du 1ᵉʳ janvier au 30 juin(en millions d'euros)|2025|2024|
|-|-|-|
|Salaires et traitements|(3 389)|(3 221)|
|Autres charges sociales|(631)|(582)|
|Charges de retraite à cotisations définies|(508)|(483)|
|Charges de retraite à prestations définies|(79)|(76)|
|Coûts du personnel intérimaire|(132)|(136)|
|Charge d'intéressement et de participation|(64)|(21)|
|Charges relatives aux paiements fondés sur des actions|(1)|(1)|
|Autres|(63)|(76)|
|TOTAL|(4 867)|(4 596)|


### Charges de retraite à cotisations définies
Le Groupe verse des cotisations pour un régime de retraite multi-employeurs en France, la CRPN (Caisse de retraite du personnel navigant). Ce plan multi-employeur étant assimilé à un plan d’État, il est comptabilisé en tant que régime à cotisations définies en « charges de retraite à cotisations définies ».

Tous les principaux régimes de retraite de KLM aux Pays-Bas sont qualifiés de régimes à cotisations définies.

### ÉQUIVALENT TEMPS PLEIN DE LA PÉRIODE <sup>(1)</sup>

Ce tableau présente l'évolution des effectifs du groupe par catégorie de personnel. L'effectif total est en progression, passant de 79 976 à 81 387 équivalents temps plein.

|Période du 1ᵉʳ janvier au 30 juin|2025|2024|
|-|-|-|
|Pilotes|9 347|8 906|
|Personnel navigant commercial|22 755|22 187|
|Personnel au sol|46 924|46 403|
|Personnel intérimaire|2 361|2 480|
|TOTAL|81 387|79 976|


<sup>(1) Les calculs sont effectués selon la méthode de la double pondération (temps de présence sur la période et temps de travail).</sup>

# NOTE 8 AUTRES PRODUITS ET CHARGES D'EXPLOITATION COURANTS

Ce tableau récapitule les autres éléments constitutifs du résultat d'exploitation courant. Le total est en légère hausse, soutenu par la production capitalisée et d'autres produits divers.

|Période du 1ᵉʳ janvier au 30 juin(en millions d'euros)|2025|2024|
|-|-|-|
|Production capitalisée|755|728|
|Exploitation conjointe de lignes passage et cargo|(19)|3|
|Couverture sur flux d’exploitation (change)|5|25|
|Autres|94|63|
|TOTAL|835|819|


# NOTE 9 AMORTISSEMENTS, DÉPRÉCIATIONS ET PROVISIONS

Ce tableau présente le détail des charges d'amortissement et des variations de dépréciations et provisions pour les périodes closes au 30 juin 2025 et 2024. On observe une augmentation globale des charges d'amortissement, notamment sur les immobilisations aéronautiques et les droits d'utilisation.

|Période du 1ᵉʳ janvier au 30 juin(en millions d'euros)<br/>AMORTISSEMENTS|2025<br/>AMORTISSEMENTS|2024<br/>AMORTISSEMENTS|
|-|-|-|
|Immobilisations incorporelles|(89)|(82)|
|Immobilisations aéronautiques|(657)|(577)|
|Autres immobilisations corporelles|(94)|(90)|
|Droits d'utilisation|(724)|(613)|
|Sous total|(1 564)|(1 362)|
|DÉPRÉCIATIONS ET PROVISIONS|||
|Créances|26|20|
|Provisions|81|21|
|Sous total|107|41|
|TOTAL|(1 457)|(1 321)|


En 2024 et en 2025, les variations des provisions s’expliquaient principalement par des reprises liées à des restitutions d’avions.

Les mouvements au bilan du poste « provisions » sont détaillés dans la Note 20.

# NOTE 10 CESSIONS DE MATÉRIELS AÉRONAUTIQUES ET AUTRES PRODUITS ET CHARGES NON COURANTS

Ce tableau récapitule les impacts en résultat des opérations de cession de matériel et des éléments non courants. Le poste "Autres produits et charges non courants" en 2024 était marqué par une charge significative de 118 millions d'euros.

|Période du 1ᵉʳ janvier au 30 juin(en millions d'euros)|2025|2024|
|-|-|-|
|Cession-bail|2|(2)|
|Autres cessions aéronautiques|(4)|17|
|Cessions de matériels aéronautiques|(2)|15|
|Autres produits et charges non courants|(10)|(118)|


## Période close au 30 juin 2025

### Cessions de matériels aéronautiques
L’impact des cessions-bail sur avions (« *sales and leaseback* ») s’est traduit par un produit de 2 millions d’euros en compte de résultat et un produit de cession en tableau des flux de trésorerie de 558 millions d’euros au 30 juin 2025.

## Période close au 30 juin 2024

### Cessions de matériel aéronautiques
L’impact des cessions-bail sur avions (« *sales and leaseback* ») s’était traduit par une charge de 2 millions d’euros en compte de résultat et un produit de cession en tableau des flux de trésorerie de 328 millions d’euros au 30 juin 2024.

### Autres cessions aéronautiques
L’impact des autres cessions aéronautiques s’expliquait essentiellement par une opération de refinancement réalisée sur un B777 chez KLM ayant généré un produit de 16 millions d’euros au 30 juin 2024.

### Autres produits et charges non courants
Le montant des autres produits et charges non courants incluait une indemnité de (115) millions d’euros à payer par Air France-KLM à Virgin dans le cadre de renégociation d’un contrat.
# NOTE 11 COÛT DE L'ENDETTEMENT FINANCIER ET AUTRES PRODUITS ET CHARGES FINANCIERS

Ce tableau présente le détail du coût de l'endettement financier net et des autres produits et charges financiers pour les semestres clos au 30 juin 2025 et 2024. Le coût de l'endettement financier net s'établit à (207) millions d'euros en 2025 contre (144) millions d'euros en 2024. Le résultat total financier est positif à 191 millions d'euros en 2025, porté par un résultat de change significatif de 555 millions d'euros.

|Période du 1ᵉʳ janvier au 30 juin (en millions d'euros)|2025|2024|
|-|-|-|
|Produits des valeurs mobilières de placement|46|78|
|Autres produits financiers|56|92|
|*Produits de la trésorerie et équivalents de trésorerie*|*102*|*170*|
|Intérêts sur passifs financiers|(151)|(163)|
|Intérêts sur dettes de loyers|(166)|(140)|
|Intérêts intercalaires capitalisés|32|25|
|Autres éléments non monétaires|(6)|(13)|
|Autres charges financières|(18)|(23)|
|*Charges d'intérêts*|*(309)*|*(314)*|
|*Coût de l'endettement financier net*|*(207)*|*(144)*|
|Résultat de change|555|(47)|
|Instruments financiers|2|(6)|
|Dotation nette aux provisions|(3)|(3)|
|Désactualisation des provisions|(147)|(138)|
|Autres|(9)|(19)|
|*Autres produits et charges financiers*|*398*|*(213)*|
|*TOTAL*|*191*|*(357)*|


### Coût de l’endettement financier net

Les produits de la trésorerie et équivalents de trésorerie sont principalement constitués des produits d’intérêts des valeurs mobilières de placement et autres actifs financiers ainsi que du résultat net sur cessions de valeurs mobilières de placement.

### Résultat de change

Au 30 juin 2025, le résultat de change inclut un gain de change latent de 601 millions d’euros composé principalement :
* d’un gain de 483 millions d’euros sur les passifs et provisions de restitution des avions loués en dollar US ;
* d’un gain, net des dérivés de change, de 79 millions d’euros sur la dette nette dont un gain de 63 millions d’euros au titre du dollar US et un gain de 19 millions d’euros au titre du yen japonais.
* d’un gain de 37 millions d’euros sur les autres actifs et passifs, principalement lié au dollar US sur des compte du besoin en fonds de roulement.

Au 30 juin 2024, le résultat de change incluait principalement une perte de change latent de (28) millions d’euros composée principalement :
* d’une perte de (108) millions d’euros sur les passifs et provisions de restitution des avions loués en dollar US ;
* et d’un gain de 85 millions d’euros sur les passifs financiers principalement composé d’une perte au titre de la dette nette en dollar US pour 12 millions d’euros et d’un gain de 61 millions d’euros au titre de la dette nette en yen japonais et un gain de 16 millions d’euros au titre du franc suisse.

### Désactualisation des provisions

Le taux utilisé pour désactualiser les passifs et provisions de restitution pour avions loués et autres provisions non courants s’élève à 6,8 % en 2025 contre 7,3 % en 2024 (voir Note 20.1.1 « Passifs et provisions de restitution pour avions loués »).
# NOTE 12 IMPÔTS

La charge d'impôt aux bornes du Groupe est la suivante :

**Résumé du tableau :** Ce tableau présente la charge ou le produit d'impôt total comptabilisé au compte de résultat ainsi que les impôts sur les éléments comptabilisés en capitaux propres pour les premiers semestres 2025 et 2024.

**Période du 1<sup>er</sup> janvier au 30 juin**

|(en millions d'euros)|2025|2024|
|-|-|-|
|(Charge)/Produit total d'impôt au compte de résultat|(176)|119|
|Impôts sur éléments comptabilisés en capitaux propres (1)|(2)|(15)|


<sup>(1) Incluant 38 millions d'euros d'impôts sur coupons sur titres subordonnés à durée indéterminée au 30 juin 2025 (contre 30 millions d'euros au 30 juin 2024).</sup>

**Résumé du tableau :** Ce tableau détaille le calcul du taux d'impôt effectif du Groupe, en mettant en rapport le résultat avant impôt et la charge d'impôt correspondante pour les périodes comparées.

**Période du 1<sup>er</sup> janvier au 30 juin**

|(en millions d'euros)|2025|2024|
|-|-|-|
|Résultat avant impôt des entreprises intégrées|588|(436)|
|(Charge)/Produit total d'impôt au compte de résultat|(176)|119|
|Taux d'impôt effectif|(30 %)|(27 %)|


## Taux d'impôt effectif au 30 juin 2025

Le taux d'impôt effectif du Groupe Air France-KLM s'établit à 30 % au 30 juin 2025 compte tenu de la contribution exceptionnelle sur les bénéfices des grandes entreprises pour le Groupe fiscal français, portant ainsi le taux d'impôt effectif théorique à 30,2 % dans cette juridiction. Le taux d'impôt effectif théorique pour le Groupe fiscal néerlandais demeure à 25,8 %.

## Législation du modèle Pilier 2 de l'OCDE

Le groupe Air France-KLM est soumis aux règles Pilier 2 de l'OCDE suite à leur transposition en droit français et son entrée en vigueur pour les exercices ouverts à compter du 31 décembre 2023. En vertu de cette législation, le Groupe est tenu de payer un impôt complémentaire pour la différence entre son taux d'imposition effectif GloBE (TEI GloBE) dans chaque juridiction et le taux minimum de 15 % (pour l'exercice 2025).

Des régimes de protection temporaire ont été instaurés pour trois exercices maximum et permettent de différer l'application des règles. Ces régimes consistent en des tests simplifiés par rapport aux règles Pilier 2, calculés par juridiction et à la fin de chaque exercice.

Au 31 décembre 2024, le Groupe a estimé qu'il pouvait bénéficier des régimes de protection temporaire dans presque toutes les juridictions où il opère, sauf dans la juridiction France en raison de la reconnaissance/dé-reconnaissance des impôts différés principalement et dans quelques juridictions non significatives, qui doivent ainsi effectuer un calcul complet de TEI GloBE selon la loi en vigueur et les recommandations OCDE disponibles.

Le calcul complet du TEI GloBE pour la juridiction France dépassait le taux minimum de 15 %. En conséquence, aucune charge d'impôt complémentaire n'avait été comptabilisée au 31 décembre 2024.

Le Groupe n'a pas identifié d'évènement remettant en cause les calculs effectués au 31 décembre 2024 et n'anticipe pas de charge d'impôt complémentaire à comptabiliser au titre de l'exercice 2025.

Par conséquent, le taux d'imposition effectif prévu pour l'exercice 2025 n'est pas affecté par la législation Pilier 2 au 30 juin 2025.
# NOTE 13 RÉSULTAT NET – PART DU GROUPE PAR ACTION

## RÉSULTATS RETENUS POUR LE CALCUL DU RÉSULTAT DE BASE PAR ACTION
**Période du 1ᵉʳ janvier au 30 juin**

Ce tableau présente le passage du résultat net part du groupe au résultat net de base après déduction des coupons sur titres subordonnés. En 2025, le groupe affiche un bénéfice de 286 millions d'euros contre une perte de 427 millions d'euros en 2024.

|(en millions d'euros)|2025|2024|
|-|-|-|
|Résultat net – part du groupe|314|(400)|
|Coupons sur titres subordonnés à durée indéterminée- net de l'effet impôt|(28)|(27)|
|Résultat net de base – part du groupe|286|(427)|


## RÉSULTATS RETENUS POUR LE CALCUL DU RÉSULTAT DILUÉ PAR ACTION
**Période du 1ᵉʳ janvier au 30 juin**

Ce tableau détaille l'ajustement du résultat net pour le calcul du résultat dilué, en intégrant l'effet des intérêts sur obligations convertibles. Le résultat dilué retenu pour 2025 s'élève à 293 millions d'euros.

|(en millions d'euros)|2025|2024|
|-|-|-|
|Résultat net de base – part du groupe|286|(427)|
|Effet des actions ordinaires potentielles sur le résultat : intérêts versés sur les obligations convertibles (net d'impôt)|7|–|
|Résultat net – part du groupe (retenu pour le calcul du résultat dilué par action)|293|(427)|


## RAPPROCHEMENT DU NOMBRE D’ACTIONS UTILISÉ POUR LE CALCUL DES RÉSULTATS PAR ACTION
**Période du 1ᵉʳ janvier au 30 juin**

Ce tableau récapitule l'évolution du nombre d'actions (moyenne pondérée) servant de base au calcul des ratios par action, incluant les actions propres et les instruments dilutifs.

|Période du 1ᵉʳ janvier au 30 juin|2025|2024|
|-|-|-|
|Nombre moyen pondéré :|||
|■ d'actions ordinaires émises|262 769 869|262 769 869|
|■ d'actions propres achetées dans le cadre des plans d'options d'achat et autres actions propres achetées|(157 547)|(137 068)|
|Nombre d'actions retenu pour le calcul du résultat de base par action|262 612 322|262 632 801|
|Nombre d'actions potentiellement dilutives|19 996 070|22 421 076|
|Nombre d'actions retenu pour le calcul du résultat dilué par action|282 608 392|285 053 877|


Pour rappel, suite au remboursement de 452 millions d’euros d’OCEANE réalisé le 25 mars 2024, le nombre d'actions potentielles dilutives liées à l'OCEANE 2026 en circulation a été réduit de 4 966 518 actions à 472 580 actions.

Les obligations restantes en circulation pour un montant de 48 millions d’euros dont l’échéance était le 25 mars 2026 ont été remboursées en numéraire le 10 mai 2024, suite à l’exercice de l’option de remboursement anticipé par l’émetteur de l’obligation dans les conditions prévues par le Règlement des OCEANE 2026. Ces obligations restantes ont été remboursées par anticipation. Ce montant résiduel de remboursement équivaut à 2 654 942 obligations.

À l’issue de ces deux opérations, il n’y avait plus d’obligations OCEANE en circulation.

Le nombre d’actions potentiellement dilutives lié aux obligations subordonnées de dernier rang à durée indéterminée, convertibles en actions existantes s’établit à 19 996 070 actions au 30 juin 2025.

La conversion potentielle de cet instrument et ces effets sur le résultat ont été pris en compte au 30 juin 2025 pour déterminer le résultat dilué par action.

Au 30 juin 2025, compte tenu des éléments présentés ci-dessus, le résultat net de base par action ressort à 1,09 euro et le résultat net dilué par action ressort à 1,04 euro.
## NOTE 14 IMMOBILISATIONS CORPORELLES

Ce tableau présente la ventilation des immobilisations corporelles du groupe entre les actifs aéronautiques (avions, actifs en cours) et les autres immobilisations (terrains, matériels). Il détaille les valeurs brutes, les amortissements et les valeurs nettes au 31 décembre 2024 et au 30 juin 2025. On observe une augmentation de la valeur nette totale, passant de 13 880 millions d'euros à 14 982 millions d'euros sur la période.

|(en millions d'euros)<br/>VALEUR BRUTE|Immobilisations aéronautiques<br/>Avions en pleine propriété<br/>VALEUR BRUTE|Immobilisations aéronautiques<br/>Actifs en cours de construction<br/>VALEUR BRUTE|Immobilisations aéronautiques<br/>Autres<br/>VALEUR BRUTE|Immobilisations aéronautiques<br/>Total<br/>VALEUR BRUTE|Autres immobilisations corporelles<br/>Terrains et constructions<br/>VALEUR BRUTE|Autres immobilisations corporelles<br/>Matériels et installations<br/>VALEUR BRUTE|Autres immobilisations corporelles<br/>Actifs en cours de construction<br/>VALEUR BRUTE|Autres immobilisations corporelles<br/>Autres<br/>VALEUR BRUTE|Autres immobilisations corporelles<br/>Total<br/>VALEUR BRUTE|TotalVALEUR BRUTE|
|-|-|-|-|-|-|-|-|-|-|-|
|31 décembre 2024|18 450|1 918|2 902|23 270|2 877|1 092|297|1 088|5 354|28 624|
|30 juin 2025|18 566|2 857|3 063|24 486|2 920|1 101|327|1 102|5 450|29 936|
|AMORTISSEMENTS|||||||||||
|31 décembre 2024|(10 071)|-|(852)|(10 923)|(2 087)|(853)|-|(881)|(3 821)|(14 744)|
|30 juin 2025|(10 208)|-|(884)|(11 092)|(2 124)|(851)|-|(887)|(3 862)|(14 954)|
|VALEUR NETTE|||||||||||
|31 décembre 2024|8 379|1 918|2 050|12 347|790|239|297|207|1 533|13 880|
|30 juin 2025|8 358|2 857|2 179|13 392|796|250|327|215|1 587|14 982|


## NOTE 15 INVESTISSEMENTS CORPORELS ET INCORPORELS

Les investissements corporels et incorporels figurant dans le tableau des flux de trésorerie consolidé se ventilent comme suit :

**Période du 1ᵉʳ janvier au 30 juin**

Ce tableau récapitule les flux d'acquisition d'actifs pour le premier semestre 2025 comparé au premier semestre 2024. Le total des investissements s'élève à 2 315 millions d'euros en 2025 contre 2 067 millions d'euros en 2024, portés principalement par les acquisitions aéronautiques.

|(en millions d'euros)|2025|2024|
|-|-|-|
|Acquisition d'immobilisations aéronautiques|2 059|1 801|
|Acquisition d'autres immobilisations corporelles|151|145|
|Acquisition d'immobilisations incorporelles|108|86|
|Variation des passifs sur immobilisations|(3)|35|
|TOTAL|2 315|2 067|


## NOTE 16 DROITS D'UTILISATION ET DETTES DE LOYERS

Le tableau ci-dessous présente les droits d’utilisation par catégorie :

Ce tableau détaille la valeur nette des droits d'utilisation (actifs en location/leasing) par catégorie d'actifs. On note une progression significative de la valeur nette totale des droits d'utilisation, qui passe de 7 592 millions d'euros à 8 479 millions d'euros, principalement due à la catégorie "Avion".

|(en millions d'euros)<br/>VALEUR NETTE|Avion<br/>VALEUR NETTE|Maintenance<br/>VALEUR NETTE|Terrains & constructions<br/>VALEUR NETTE|Autres<br/>VALEUR NETTE|Total<br/>VALEUR NETTE|
|-|-|-|-|-|-|
|31 décembre 2024|4 031|2 819|679|63|7 592|
|30 juin 2025|4 638|3 060|709|72|8 479|


Le tableau ci-dessous présente les dettes de loyers par catégories :

Ce tableau détaille la répartition des dettes de loyers entre les parts courantes et non courantes au 30 juin 2025 et au 31 décembre 2024. Les dettes concernent principalement les avions, les rechanges aéronautiques, l'immobilier et d'autres actifs. Au 30 juin 2025, le total des dettes de loyers s'élève à 5 786 millions d'euros.

|(en millions d'euros)|Au 30 juin 2025<br/>Non Courant|Au 30 juin 2025<br/>Courant|Au 30 juin 2025<br/>Total|Au 31 décembre 2024<br/>Non courant|Au 31 décembre 2024<br/>Courant|Au 31 décembre 2024<br/>Total|
|-|-|-|-|-|-|-|
|Dettes de loyers – Avions|3 963|726|4 689|3 834|780|4 614|
|Dettes de loyers – Rechanges aéronautiques|104|54|158|115|61|176|
|Dettes de loyers – Immobilier|737|107|844|712|103|815|
|Dettes de loyers – Autres|60|13|73|53|15|68|
|Intérêts courus non échus|–|22|22|–|23|23|
|TOTAL – DETTES DE LOYERS|4 864|922|5 786|4 714|982|5 696|


# NOTE 17 TRÉSORERIE, ÉQUIVALENTS DE TRÉSORERIE ET CONCOURS BANCAIRES

Ce tableau présente la composition de la trésorerie et des équivalents de trésorerie, ainsi que les concours bancaires. Il distingue les montants totaux des montants nantis ou bloqués. Au 30 juin 2025, la trésorerie nette des concours bancaires s'établit à 4 820 millions d'euros.

|(en millions d'euros)|Au 30 juin 2025<br/>Total|Au 30 juin 2025<br/>Dont nantisou bloqués|Au 31 décembre 2024<br/>Total|Au 31 décembre 2024<br/>Dont nantisou bloqués|
|-|-|-|-|-|
|SICAV (actifs – instruments de dettes)|1 790|7|1 442|7|
|Dépôts (actifs – instruments de dettes) et comptes à termes|1 645|–|1 543|–|
|Caisses et banques|1 415|–|1 844|–|
|Trésorerie et équivalents de trésorerie|4 850|7|4 829|7|
|Concours bancaires|(30)|–|–|–|
|TRÉSORERIE, ÉQUIVALENTS DE TRÉSORERIE<br/>ET CONCOURS BANCAIRES|4 820|7|4 829|7|


# NOTE 18 CAPITAUX PROPRES

## 18.1 Capital

Au 30 juin 2025, le capital social est composé de 262 769 869 actions, entièrement libérées, d’une valeur nominale de 1 euro et le capital social du groupe Air France-KLM s’élève à 263 millions d’euros. Chaque action confère un droit de vote.

Cependant, depuis le 3 avril 2016, tout porteur détenant des actions nominatives depuis au moins deux ans dispose d’un droit de vote double.

Le capital et les droits de vote se répartissent de la façon suivante :

Ce tableau présente la répartition de l'actionnariat et des droits de vote du groupe Air France-KLM au 30 juin 2025 comparativement au 31 décembre 2024. On y observe la part détenue par les États français et néerlandais, les partenaires stratégiques (CMA CGM, China Eastern, Delta), les salariés et le public.

|Actionnaire|En % du capital<br/>Au 30 juin 2025|En % du capital<br/>Au 31 décembre 2024|En % des droits de vote<br/>Au 30 juin 2025|En % des droits de vote<br/>Au 31 décembre 2024|
|-|-|-|-|-|
|État français|28,0|28,0|29,3|27,5|
|État néerlandais|9,1|9,1|13,0|13,3|
|CMA CGM|8,8|8,8|12,5|12,8|
|China Eastern Airlines|4,6|4,6|6,5|6,7|
|Salariés et anciens salariés|3,0|3,1|2,9|3,0|
|Delta Air Lines|2,8|2,8|4,0|4,1|
|SPAAK (1)|0,9|0,9|1,2|1,2|
|Public|42,8|42,7|30,7|31,4|
|TOTAL|100|100|100|100|


*(1) Stichting Piloten Aandelen Air France-KLM.*

La ligne « Salariés et anciens salariés » regroupe les titres détenus par le personnel et les anciens salariés dans des fonds communs de placement d’entreprise (FCPE).

Au 30 juin 2025, tous les titres ont été émis et payés.
## 18.2 Titres subordonnés à durée indéterminée

Ce tableau détaille l'évolution des titres subordonnés à durée indéterminée entre le 31 décembre 2024 et le 30 juin 2025. Il distingue la part attribuable aux propriétaires de la société mère et les participations ne donnant pas le contrôle. On y observe notamment l'émission de nouvelles obligations hybrides pour 500 millions d'euros et le reclassement des Titres Super Subordonnés 2022 suite à une notification de remboursement.

|(en millions d'euros)||31 décembre 2024|Remboursement Nominal|Émission Nominal|Variation monétaire – Coupons|Variation non monétaire|30 juin 2025||
|-|-|-|-|-|-|-|-|-|
|Titres Super Subordonnés 2023|Nominal|727|–|–|–|–|727||
||Coupons|44|–|–|(55)|27|16||
||Obligations subordonnées de dernier rang à durée indéterminée, convertibles en actions nouvelles et/ou échangeables en actions existantes|Nominal|305|–|–|–|–|305|
||Coupons|2|–|–|(10)|10|2||
||Obligations subordonnées de dernier rang à durée indéterminée|Nominal|–|–|494|–|6|500|
||Coupons|–|–|–|–|4|4||
||TOTAL TITRES SUBORDONNÉS À DURÉE INDÉTERMINÉE – PART ATTRIBUABLE AUX PROPRIÉTAIRES DE LA SOCIÉTÉ MÈRE||1 078|–|494|(65)|47|1 554|
|Titres Super Subordonnés 2022|Nominal|497|–|–|–|(497)|–||
||Coupons|13|–|–|–|(13)|–||
||Titres Super Subordonnés Juillet 2023|Nominal|498|–|–|–|–|498|
||Coupons|16|–|–|–|17|33||
||Titres Super Subordonnés Novembre 2023|Nominal|1 493|–|–|–|–|1 493|
||Coupons|13|–|–|–|51|64||
||TOTAL TITRES SUBORDONNÉS – PARTICIPATIONS NE DONNANT PAS LE CONTRÔLE||2 530|–|–|–|(442)|2 088|
|Total des flux de trésorerie|||–|494|(65)||||


**Notification de remboursement des titres super subordonnés 2022 d’un montant de 497 millions d’euros**

Air France a notifié à la société Apollo le 20 juin 2025 sa décision (irrévocable) de procéder au remboursement le 28 juillet 2025 des titres super subordonnés émis en juillet 2022 pour 497m€.

Par conséquence, ces titres ont été reclassés des titres subordonnés à durée indéterminée en passifs financiers courants au 30 juin 2025 pour un montant total de 524 millions d’euros (le nominal pour 497 millions d’euros et les intérêts courus non échus pour 27 millions d’euros). Se référer à la Note 21 “Passifs financiers”.

**Emission d’obligations hybrides pour un montant de 500 millions d’euros**

Air France-KLM a placé le 15 mai 2025 500 millions d’euros d’obligations hybrides (494 millions d’euros net des frais d’émission), au coupon fixe annuel de 5,75% (yield de 5,875%) jusqu’à la première date de reset.
# NOTE 19 ACTIFS ET PROVISIONS RETRAITES

Au 30 juin 2025, les taux d’actualisation utilisés par les sociétés du Groupe pour le calcul des engagements de retraite à prestations définies sont les suivants :

Ce tableau présente les taux d'actualisation pour les engagements de retraite par zone géographique (Zone euro et Royaume-Uni) au 30 juin 2025 comparés au 31 décembre 2024.

||Au 30 juin 2025|Au 31 décembre 2024|
|-|-|-|
|Zone euro – Duration 10 à 15 ans|3,58 %|3,30 %|
|Royaume-Uni – Duration 20 ans|5,55 %|5,45 %|


Les taux d’inflation utilisés sont les suivants :

Ce tableau présente les taux d'inflation prévisionnels par zone géographique au 30 juin 2025 comparés au 31 décembre 2024.

||Au 30 juin 2025|Au 31 décembre 2024|
|-|-|-|
|Zone euro – Duration 10 à 15 ans|2,00 %|2,00 %|
|Royaume-Uni – Duration 20 ans|2,90 %|3,25 %|


Le taux de duration dix à quinze ans concerne essentiellement les régimes situés en France.

Au 30 juin 2025, la réévaluation des engagements nets sur les régimes à prestations définis est composée de (avant impôt) :

Ce tableau détaille les composants de la réévaluation des engagements de retraite (taux d'actualisation, inflation, rendement des actifs) pour le premier semestre 2025 par rapport au premier semestre 2024.

|Au 30 juin|2025|2024|
|-|-|-|
|Impact du changement de taux d’actualisation|46|98|
|Impact du changement de taux d’inflation|11|(15)|
|Ecart entre le rendement attendu et réel des actifs|(17)|(2)|
|TOTAL|40|81|


Après impôt, la réévaluation des engagements nets sur les régimes à prestations définis s’élève à 43 millions d’euros.

L’impact de la variation des taux d’actualisation sur les engagements a été calculé en utilisant les analyses de sensibilité de l’engagement de retraite à prestations définies. Celles-ci sont mentionnées dans la note 29.2 aux états financiers consolidés de l’exercice clos le 31 décembre 2024.

# NOTE 20 PASSIFS ET PROVISIONS DE RESTITUTION POUR AVIONS LOUÉS ET AUTRES PROVISIONS

Ce tableau récapitule l'évolution des passifs de restitution, de maintenance, de restructuration, des litiges et des quotas de CO2 entre le 1er janvier 2024 et le 30 juin 2025, avec une ventilation entre parts courantes et non courantes.

|(en millions d'euros)<br/>Montant au 1ᵉʳ janvier 2024|Passifs de restitution sur avions loués<br/>3 802|Maintenance sur avions loués<br/>161|Restructuration<br/>82|Litiges<br/>516|Provision pour restitution de quotas de CO₂<br/>213|Autres<br/>110|Total<br/>4 884|
|-|-|-|-|-|-|-|-|
|Dont : non courant|3 532|148|–|36|–|89|3 805|
|courant|270|13|82|480|213|21|1 079|
|Montant au 31 décembre 2024|4 572|169|87|464|250|132|5 674|
|Dont : non courant|4 163|153|–|69|–|108|4 493|
|courant|409|16|87|395|250|24|1 181|
|Montant au 30 juin 2025|4 378|172|76|464|385|134|5 609|
|Dont : non courant|4 047|153|–|67|133|113|4 513|
|courant|331|19|76|397|252|21|1 096|


Les mouvements de provision pour litiges ainsi que des autres provisions pour risques et charges impactant le compte de résultat sont enregistrés, selon leur nature, dans les différentes rubriques correspondantes du compte de résultat.
# 20.1 Provisions

### 20.1.1 Passifs et provisions de restitution pour avions loués

Le taux d’actualisation utilisé pour le calcul de ces passifs et provisions est de 6,8 % au 30 juin 2025 contre 7,3 % au 30 juin 2024 (voir Note 11 « Coût de l'endettement financier et autres produits et charges financiers »).

### 20.1.2 Provisions pour restructuration

Les mouvements de provision pour restructuration impactant le compte de résultat sont enregistrés en « autres produits et charges non courants » lorsque les effets sont significatifs (voir Note 10 « Cessions de matériels aéronautiques et autres produits et charges non courants »).

### 20.1.3 Provisions pour litiges avec les tiers

Dans le cours normal de ses activités, le groupe Air France-KLM et ses filiales Air France et KLM (et leurs filiales) sont impliqués dans divers litiges dont certains peuvent avoir un caractère significatif.

Une évaluation des risques de litiges avec les tiers a été effectuée avec le concours des avocats du Groupe et des provisions ont été enregistrées lorsque les circonstances les rendaient nécessaires.

Les provisions pour litiges comprennent également des provisions pour risques fiscaux qui n’entrent dans le champ d’IAS 12. De telles provisions sont constituées lorsque le Groupe estime, dans le cadre de contrôles fiscaux, que l’administration fiscale pourrait être amenée à remettre en cause une position fiscale prise par le Groupe ou l’une de ses filiales.

### 20.1.4 Litiges en matière de droit de la concurrence dans les secteurs du fret aérien

Air France, KLM et Martinair, filiale entièrement détenue par KLM depuis le 1<sup>er</sup> janvier 2009, ont été impliqués depuis février 2006 avec vingt-cinq autres compagnies aériennes dans des enquêtes diligentées par certaines autorités de la concurrence dans le monde concernant des allégations d’entente ou de pratiques concertées dans le secteur du fret aérien.

Au 31 décembre 2021, la plupart des procédures ouvertes avaient donné lieu à des accords transactionnels conclus entre les trois sociétés du groupe et les autorités compétentes et au paiement d’amendes qui avaient mis fin à ces procédures, à l’exception de celle initiée par la Commission Européenne qui est toujours en cours.

En Europe, la décision de la Commission Européenne de 2010 à l’encontre de 11 opérateurs de fret aérien, incluant les compagnies du Groupe Air France, KLM et Martinair, a été annulée par le Tribunal de l’Union européenne le 16 décembre 2015 parce qu’elle contenait une contradiction concernant le périmètre exact des pratiques sanctionnées. La Commission Européenne a adopté le 17 mars 2017 une nouvelle décision à l’encontre des opérateurs susvisés, dont Air France, KLM et Martinair. Le montant total des amendes imposées au titre de cette décision au niveau de groupe Air France-KLM est de 339 millions d’euros. Ce montant a été légèrement réduit de 15,4 millions d’euros par rapport à la première décision en raison du niveau inférieur de l’amende de Martinair pour des raisons techniques. Les entités du Groupe ont formé un recours contre cette décision devant le Tribunal de l’Union Européenne les 29 et 30 mai 2017. Les audiences devant le Tribunal ont eu lieu en juin et juillet 2019.

La décision du Tribunal en mars 2022 a confirmé les amendes infligées aux sociétés du groupe Air France-KLM. Les sociétés du Groupe ont formé des pourvois en juin 2022 devant la Cour de justice de l'Union Européenne. Les audiences se sont tenues les 18 et 19 avril 2024. L’avocat général a rendu ses conclusions le 5 septembre 2024 et préconisé le rejet des pourvois. L’arrêt de la Cour de justice de l’Union Européenne devrait être rendu en 2025. Au 30 juin 2025, le Groupe a maintenu une provision des 366 millions d’euros pour le montant total des amendes (incluant les intérêts).

### 20.1.5 Litige engagé à l’encontre de KLM par (d’anciens) pilotes de fret de Martinair

En 2015, une plainte a été déposée contre KLM par 152 (anciens) pilotes de la compagnie aérienne Martinair, ci-après désigné « Vrachtvliegers ». En 2016 et 2018, le tribunal de première instance et la Cour d’appel ont statué en faveur de KLM et rejeté toutes les demandes des plaignants. Cependant, en novembre 2019, la Cour Suprême a jugé que le jugement de la cour d’appel n’était pas suffisamment motivé et a renvoyé l’affaire devant une autre cour d’appel. Le 8 juin 2021, cette Cour d'appel a rendu son arrêt en faveur des plaignants, les anciens pilotes de Martinair, jugeant que le transfert du département cargo est qualifié de transfert d'entreprise.

Selon cette décision les droits et obligations découlant des contrats de travail de 116 pilotes de Martinair sont automatiquement transférés à KLM à compter du 1<sup>er</sup> janvier 2014. En revanche, la Cour d'appel a rejeté la demande des plaignants de transférer également les droits relatifs à l'ancienneté accumulés chez Martinair.

Le 8 août 2021, les Vrachtvliegers ont déposé des plaintes auprès de la Cour Suprême, réclamant que les droits relatifs à l'ancienneté accumulés chez Martinair soient transférés à KLM. Le 24 juin 2022, l’avocat général a conseillé à la Cour Suprême de rejeter les plaintes. Le 20 janvier 2023, la Cour Suprême a rejeté cette demande. Les pilotes ont également déposé une nouvelle plainte au sujet de la mise en place par KLM de ce transfert. L’audience s’est tenue le 15 novembre 2023. Le tribunal a rendu une décision le 11 janvier 2024, dans laquelle toutes les demandes ont été rejetées, à l'exception du respect de l'ancienneté accumulée au sein de Martinair en cas de licenciement (ce qui est conforme à la législation en vigueur).

Au 30 juin 2025, la provision s’établit à 22 millions d’euros (sans changement par rapport au 31 décembre 2024).
### 20.1.6 Autres provisions

Les autres provisions comprennent principalement des provisions pour contrats déficitaires et des provisions pour démantèlement de bâtiments construits sur le sol d’autrui.

### 20.2 Passifs éventuels

Le Groupe est impliqué dans des procédures gouvernementales, judiciaires ou d’arbitrages pour lesquelles dans certains cas, il n’a pas été constitué de provisions dans ses états financiers, en conformité avec les règles comptables applicables.

En effet, à ce stade des procédures, le Groupe n’est pas en mesure d’apprécier de manière fiable les risques financiers liés à certains de ces litiges.
Par ailleurs le Groupe estime que toute information supplémentaire divulguée pourrait nuire à la position juridique dans les procédures.

#### 20.2.1 Litiges en matière de droit de la concurrence dans le secteur du fret aérien

À la suite de l’ouverture en février 2006 des enquêtes de plusieurs autorités de la concurrence et de la décision initiale de la Commission Européenne de 2010, plusieurs actions civiles individuelles ou collectives ont été engagées par des transitaires et des expéditeurs de fret aérien dans plusieurs pays à l’encontre d’Air France, de KLM et de Martinair ainsi que des autres opérateurs de fret devant différentes juridictions civiles.

Dans le cadre de ces actions civiles, les transitaires et expéditeurs de fret aérien sollicitent l’attribution de dommages et intérêts pour compenser un prétendu surcoût causé par les pratiques anti-concurrentielles alléguées.

Pour Air France, KLM et Martinair, certaines actions civiles sont toujours en cours aux Pays-Bas et en Norvège. Les sociétés du Groupe et les autres compagnies aériennes concernées continuent à s’opposer vigoureusement à ces procédures civiles.

#### 20.2.2 Autres litiges

**Vol AF447 Rio-Paris**
Air France a été mis en examen avec Airbus, le 28 mars 2011, pour homicides involontaires sur les 228 victimes décédées lors de l’accident du vol AF447 Rio-Paris du 1<sup>er</sup> juin 2009.

Une ordonnance de non-lieu en faveur d’Air France et d’Airbus a été rendue le 4 septembre 2019 par les juges d’instruction du tribunal de grande instance.

Le ministère public et la plupart des parties civiles (dont des associations et syndicats pilotes) ont fait appel de cette décision. La cour d’appel de Paris s’est prononcée le 12 mai 2021 en renvoyant Airbus et Air France devant le Tribunal Correctionnel. Le procès pénal a eu lieu du 10 octobre au 8 décembre 2022 devant le tribunal correctionnel de Paris. Après un réquisitoire de relaxe du Ministère public, le tribunal a rendu un jugement de relaxe le 17 avril 2023 fondée sur l’absence de lien de causalité entre les fautes retenues et l’accident. Le 27 avril 2023, le Parquet général fait appel de la relaxe du constructeur Airbus et de la compagnie Air France.

La procédure d’appel se déroulera devant la Cour d’appel de Paris du 29 septembre au 27 novembre 2025.

**Litiges sur les Aides d’État**
En 2020, la mise en œuvre des mesures visant à renforcer la liquidité du Groupe (à savoir (i) un prêt garanti par l’Etat français (PGE) d’un montant de 4 milliards d’euros et un prêt de 3 milliards d’euros de l’Etat français, ainsi que (ii) une facilité de crédit renouvelable de 2,4 milliards d’euros garantie par l’Etat néerlandais et un prêt de 1 milliard d’euros de l’État néerlandais a été approuvée par la Commission Européenne en vertu des règles relatives aux aides d’État Covid 19 (décisions respectivement du 4 mai 2020 et du 13 juillet 2020, cette dernière décision ayant été remplacée, après annulation pour défaut de motivation, par une décision du 16 juillet 2021).

Le 6 avril 2021, le Groupe a annoncé la première partie de son plan de recapitalisation global. Certaines mesures de ce plan contenaient une aide d’État (le programme dit de “recapitalisation Covid 19”), qui ont été notifiées par les autorités françaises à la Commission européenne, cette dernière les ayant approuvées dans sa décision du 5 avril 2021. Cette décision a subordonné l’approbation des mesures à un certain nombre d’engagements pris par l’Etat français, notamment à l’attribution par Air France de créneaux de décollage et d’atterrissage à une compagnie tierce désignée à l’aéroport d’Orly.

Comme pour la plupart des décisions concernant les compagnies aériennes bénéficiant d’une aide d’État dans le cadre de la crise de la Covid 19, les décisions de la Commission européenne accordant les mesures de soutien à Air France et à KLM ont fait l’objet de procédures d’annulation engagées par Ryanair. Le 20 décembre 2023 et le 7 février 2024, le Tribunal de l’Union européenne a annulé les décisions de la Commission européenne mentionnées ci-dessus. Ces annulations se sont faites sur l’unique motif d’une détermination erronée du bénéficiaire de ces aides, celui-ci devant être, d’après le Tribunal, le Groupe en lui-même. Air France-KLM, Air France, KLM et la Commission européenne ont formé des pourvois en annulation devant la Cour de justice de l’Union européenne contre les arrêts du Tribunal. La Cour de justice de l’Union européenne doit encore se prononcer sur ces pourvois.

L’incertitude persiste quant aux conséquences juridiques et financières de l’annulation des décisions d’approbation des aides d’Etat jusqu’à l’obtention d’un arrêt définitif des juridictions de l’Union.

Il est rappelé que le Groupe a procédé au cours des exercices 2022 et 2023 et en vertu du cadre juridique applicable, au remboursement de l’intégralité des aides d’État susmentionnées et qui étaient grevées des engagements et contraintes précitées (engagements, mesures comportementales, application des intérêts). En conséquence, Air France-KLM, Air France et KLM sont donc totalement libérées des engagements et contraintes précitées qui étaient liées à ces aides de recapitalisation Covid-19. Les conséquences indirectes potentielles de l’annulation de l’approbation des aides d’État (sous réserve d’un succès éventuel des pourvois précités) pourraient inclure une demande de récupération des avantages non remboursés de la part des autorités françaises, limitée dans certains cas aux seuls intérêts d’illégalité.

La Commission européenne a approuvé, à nouveau, le 10 juillet 2024, les aides au renforcement de la liquidité du Groupe dans une décision unique confirmant leur
compatibilité avec le droit de l’Union. Cette nouvelle décision n’a pas d’impact sur les pourvois précités. Cette décision a fait l’objet d’un nouveau recours en annulation devant le Tribunal de l’Union européenne de la part de Ryanair le 14 avril 2025. Air France-KLM, Air France et KLM interviendront, aux côtés des État français et néerlandais, au soutien de la défense de la Commission européenne.

Enfin, comme elle l'a fait dans des cas similaires, la Commission européenne peut également décider, le cas échéant, d'entamer une procédure d'examen formelle sur les mesures de recapitalisation au cours de laquelle le Groupe veillera à défendre au mieux ses intérêts.

En janvier 2025, Air France-KLM a été informée du dépôt par Ryanair d’un recours devant le Tribunal administratif de Paris contre l’État français à la suite des arrêts d’annulation précités du Tribunal de l’Union européenne. La demande de Ryanair vise à ce que l’État doive récupérer tout avantage accordé par l’État dont il est allégué qu’il n’aurait pas encore été remboursé auquel s’ajouteraient des intérêts d’illégalité. Air France KLM et Société Air France se sont constitués parties, le 3 juillet 2025, pour répondre en défense à ce recours aux côtés de l’État français. La procédure est en cours.

Par ailleurs, Air France-KLM a également été informée, en avril 2025, d’une procédure similaire (mais de nature administrative dans un premier temps) diligentée par Ryanair contre les autorités néerlandaises s’agissant de la décision relative à l’aide accordée à KLM en 2020. La procédure est en cours.

Compte tenu de la nouvelle approbation précitée de juillet 2024, ces recours pourraient donner lieu au paiement (i) d’intérêts dits « d’illégalité » pour la période entre l’octroi de ces aides et leur nouvelle approbation en juillet 2024 (le montant nominal des aides de liquidité ne pouvant plus faire l’objet d’une quelconque récupération). S’agissant des mesures de recapitalisation, une récupération d’un montant à déterminer pourrait s’ajouter aux montants ayant déjà fait l’objet d’un remboursement.

Si la Cour de justice de l’Union européenne annulait les arrêts précités du Tribunal de l’Union européenne, ces recours de Ryanair deviendraient sans objet.

Hormis les points indiqués aux paragraphes 20.1 et 20.2, la Société n'a pas connaissance de litige, de procédure gouvernementale, judiciaire ou d’arbitrage (y compris toute procédure dont l’émetteur a connaissance, qui est en suspens ou dont il est menacé) qui pourrait avoir ou a eu récemment des effets significatifs sur la situation financière, le résultat, le patrimoine ou la rentabilité de l’entreprise, pour une période couvrant au moins les douze derniers mois.
# NOTE 21 PASSIFS FINANCIERS

Ce premier tableau présente la décomposition des passifs financiers du groupe au 30 juin 2025 comparée au 31 décembre 2024, en distinguant les parts courantes et non courantes. On observe une baisse globale de la dette financière totale, passant de 8 946 millions d'euros à 8 464 millions d'euros.

|(en millions d'euros)|30 juin 2025<br/>Non courant|30 juin 2025<br/>Courant|30 juin 2025<br/>Total|31 décembre 2024<br/>Non courant|31 décembre 2024<br/>Courant|31 décembre 2024<br/>Total|
|-|-|-|-|-|-|-|
|Emprunt subordonné à durée indéterminée en yens|118|–|118|123|–|123|
|Emprunt subordonné à durée indéterminée en francs suisses|401|–|401|398|–|398|
|*Obligations liées au développement durable*|500|500|1 000|1 000|–|1 000|
|Autres emprunts obligataires|1 056|497|1 553|1 078|515|1 593|
|Dettes de location avec option d'achat avantageuse|3 300|612|3 912|3 527|642|4 169|
|Autres emprunts|1 137|276|1 413|1 127|421|1 548|
|Intérêts courus non échus|–|67|67|1|114|115|
|TOTAL – PASSIFS FINANCIERS|6 512|1 952|8 464|7 254|1 692|8 946|


## VARIATION DU PASSIF FINANCIER

Ce tableau détaille les flux ayant impacté le passif financier au cours du premier semestre 2025. Les remboursements d'emprunts s'élèvent à 1 152 millions d'euros, compensant largement les nouvelles émissions de 314 millions d'euros. La colonne "Autres" de 466 millions d'euros reflète principalement des reclassements internes.

|(en millions d'euros)|31 décembre 2024|Émission de nouveaux emprunts|Remboursement des emprunts|Variation de la conversion|Autres|30 juin 2025|
|-|-|-|-|-|-|-|
|Emprunts à durée indéterminée en yens et francs suisses|521|–|–|(2)|–|519|
|Obligations liées au développement durable|1 000|–|–|–|–|1 000|
|Autres emprunts obligataires|1 593|–|(515)|(22)|497|1 553|
|Dettes de location avec option d'achat avantageuse|4 169|188|(377)|(80)|12|3 912|
|Autres emprunts|1 548|126|(260)|(6)|5|1 413|
|Intérêts courus non échus|115|–|–|–|(48)|67|
|TOTAL|8 946|314|(1 152)|(110)|466|8 464|


### 21.1 Emprunts Obligataires

**Remboursement du solde de l’emprunt obligataire d’un montant de 515 millions d’euros émis en 2020.**

Le 16 janvier 2025, Air France-KLM a remboursé le montant restant de l’emprunt obligataire, arrivé à maturité, d’un montant de 515 millions d’euros, souscrit en 2020.

**Notification de remboursement des titres super subordonnés 2022 d’un montant de 497 millions d’euros**

Air France a notifié à la société Apollo le 20 juin 2025 sa décision (irrévocable) de procéder au remboursement le 28 juillet 2025 des titres super subordonnés émis en juillet 2022 pour 497m€.

Par conséquence, ces titres ont été reclassés des titres subordonnés à durée indéterminée en passifs financiers courants au 30 juin 2025 pour un montant total de 524 millions d’euros (le nominal pour 497 millions d’euros et les intérêts courus non échus pour 27 millions d’euros). Se référer à la Note 18.2 “Titres subordonnés à durée indéterminée”.
## Analyse par échéance

Les échéances des passifs financiers se décomposent comme suit :

Ce tableau présente la ventilation des passifs financiers du Groupe par échéance au 30 juin 2025 comparée au 31 décembre 2024. On observe une diminution globale de la dette totale, passant de 8 946 millions d'euros à 8 464 millions d'euros, avec une part significative (plus de 2,8 milliards d'euros) arrivant à échéance au-delà de 4 ans.

|(en millions d'euros)|30 juin 2025|31 décembre 2024|
|-|-|-|
|Échéances en|||
|Fin d'année N|1 043|–|
|N+1|1 528|1 692|
|N+2|660|1 595|
|N+3|1 046|611|
|N+4|1 368|963|
|Au delà de 4 ans|2 819|4 085|
|TOTAL|8 464|8 946|


Les emprunts subordonnés à durée indéterminée de KLM sont inclus dans la ligne « au-delà de 4 ans ».

## Lignes de crédit

Le 18 avril 2023, Air France-KLM, Air France et KLM ont signé deux lignes de crédit renouvelables liées au développement durable avec un regroupement d'institutions financières internationales, pour un montant total de 2,2 milliards d'euros.

Pour chaque ligne de crédit, un ensemble d’indicateurs de performance en matière de développement durable a été intégré au coût de financement. Ceux-ci sont conformes à l'engagement d'Air France-KLM et de ses compagnies aériennes en faveur du développement durable et d'une décarbonation progressive de leurs activités. Les deux lignes de crédit comprennent un mécanisme d'ajustement de la marge de crédit (à la hausse ou à la baisse) conditionné par l’atteinte de chacun de ces indicateurs de performance (la réduction des émissions unitaires de CO₂, l'augmentation de la part du carburant d'aviation durable, entre autres).

En outre, certains covenants financiers sont également applicables à ces deux lignes de crédit. Au 30 juin 2025, ces covenants financiers sont respectés.

Le montant total disponible pour le Groupe Air France-KLM au 30 juin 2025 s’élève à 2,5 milliards d’euros.

### Air France-KLM et Air France

Air France-KLM et Air France, en qualité de co-emprunteurs, ont signé une ligne de crédit liée au développement durable de 1,2 milliard d'euros. Cette ligne incluait une option d’augmentation en accordéon qui a été exercée sur le premier trimestre 2024 pour un montant de 90 millions d’euros portant ainsi le montant disponible à environ 1,3 milliard d’euros.

Cette nouvelle ligne de crédit avait par ailleurs une échéance initiale à 2026 et comprenait deux options d’extension d’un an. En avril 2024, une option d’extension a été levée portant l’échéance à 2027.

Le 18 juillet 2024, un nouvel amendement a été signé sur la ligne de crédit d’Air France-KLM et Air France, prévoyant l’extension de l’échéance à juillet 2028 associée à une option d’extension complémentaire d’un an et l’augmentation de la ligne de crédit de 1,3 à 1,4 milliard d’euros.

Le 27 juin 2025, un nouvel amendement a été signé sur la ligne de crédit d’Air France-KLM et Air France, prévoyant l’extension de l’échéance à juillet 2029.

### KLM

En 2023, KLM a signé une ligne de crédit de 1 milliard d'euros indexée sur des indicateurs de performance ESG (« Environmental, Social and Governance »).

Cette ligne de crédit, dont l'échéance initiale était fixée à 2027, est assortie de deux options d'extension d'un an. Une option d’extension d’un an a été levée en avril 2025 portant l’échéance à avril 2029.

Par ailleurs, KLM dispose de trois autres lignes de crédit pour un montant de 0,1 milliard d’euros.
# NOTE 22 INDICATEURS ALTERNATIFS DE PERFORMANCE

## 22.1 EBITDA courant

Ce tableau présente le calcul de l'EBITDA courant pour le premier semestre 2025 comparé au premier semestre 2024. Il détaille les produits des activités ordinaires ainsi que les différentes charges d'exploitation (externes, personnel, impôts et autres produits/charges courants). L'EBITDA courant s'établit à 1 866 millions d'euros en 2025 contre 1 345 millions d'euros en 2024.

|Période du 1ᵉʳ janvier au 30 juin(en millions d'euros)|2025|2024|
|-|-|-|
|Produits des activités ordinaires|15 608|14 603|
|Charges externes|(9 608)|(9 385)|
|Frais de personnel|(4 867)|(4 596)|
|Impôts et taxes hors impôt sur le résultat|(102)|(96)|
|Autres produits et charges d'exploitation courants|835|819|
|EBITDA courant|1 866|1 345|


La répartition par secteur d’activité est présentée en Note 5.1.

## 22.2 Flux de trésorerie libre d’exploitation

Le calcul des flux de trésorerie libre d’exploitation, construit à partir du tableau des flux de trésorerie, se décompose ainsi :

Ce tableau détaille le passage du flux net de trésorerie d'exploitation au flux de trésorerie récurrent libre d'exploitation ajusté. Il prend en compte les investissements, les cessions, les intérêts, les loyers et les paiements exceptionnels liés au Covid-19. On note une amélioration significative du flux de trésorerie libre d'exploitation ajusté, passant de -716 millions d'euros en 2024 à +479 millions d'euros en 2025.

|Période du 1ᵉʳ janvier au 30 juin(en millions d'euros)|Notes|2025|2024|
|-|-|-|-|
|Flux net de trésorerie provenant de l’exploitation||3 027|1 650|
|Investissements corporels et incorporels|15|(2 315)|(2 067)|
|Produits de cession d’immobilisations corporelles et incorporelles||573|373|
|Flux de trésorerie libre d'exploitation||1 285|(44)|
|Intérêts (payés) et reçus||(319)|(230)|
|Paiements de dettes de loyers||(487)|(442)|
|Flux de trésorerie libre d'exploitation ajusté||479|(716)|
|Paiements exceptionnels réalisés/(reçus) (1)||244|850|
|Flux de trésorerie récurrent libre d'exploitation ajusté||723|134|


(1) Les paiements exceptionnels réalisés/(reçus), retraités du flux de trésorerie libre d’exploitation pour le calcul du flux de trésorerie récurrent libre d’exploitation ajusté correspondent au remboursement des charges sociales, des cotisations retraites et des taxes sur les salaires différés pendant la période du Covid-19.
## 22.3 Dette nette

Ce premier tableau présente la décomposition de la dette nette du Groupe Air France-KLM, en distinguant les passifs financiers (dettes financières et de loyers) des liquidités nettes (trésorerie, placements et obligations). Au 30 juin 2025, la dette nette s'établit à 7 134 millions d'euros, en baisse par rapport aux 7 332 millions d'euros au 31 décembre 2024.

|(en millions d'euros)|Notes|Au 30 juin 2025|Au 31 décembre 2024|
|-|-|-|-|
|Passifs financiers courants et non courants|21|8 464|8 946|
|Dettes de loyers courantes et non courantes|16|5 786|5 696|
|Intérêts courus non échus|21 & 16|(90)|(138)|
|Dépôts relatifs aux passifs financiers||(90)|(97)|
|Dépôts relatifs aux dettes de loyers||(86)|(98)|
|Impact des dérivés devise/dettes||53|(45)|
|Passifs financiers (I)||14 037|14 264|
|Trésorerie et équivalent trésorerie|17|4 850|4 829|
|Valeurs mobilières de placement à plus de 3 mois||1 031|1 046|
|Obligations||1 052|1 057|
|Concours bancaires courant|17|(30)|–|
|Liquidités nettes (II)||6 903|6 932|
|DETTE NETTE (I-II)||7 134|7 332|


Au 30 juin 2025, les liquidités nettes comprennent 424 millions d’euros (contre 428 millions d’euros au 31 décembre 2024) nantis ou bloqués.

Par ailleurs, le Groupe s’est engagé à maintenir un niveau de trésorerie dans certaines filiales opérationnelles. Au 30 juin 2025, cela représente un montant total de 725 millions d’euros (sans changement par rapport au 31 décembre 2024).

Ce second tableau détaille la variation de la dette nette sur le premier semestre 2025. Il met en évidence l'impact positif du flux de trésorerie libre d'exploitation (-1 285 millions d'euros, réduisant la dette) et l'impact négatif des nouveaux contrats de location (+1 180 millions d'euros).

|(en millions d'euros)|Notes|Au 30 juin 2025|
|-|-|-|
|Dette nette à l'ouverture||7 332|
|Flux de trésorerie libre d'exploitation|22.2|(1 285)|
|Intérêts payés et reçus||319|
|Coupons payés sur titres subordonnés et sur obligations subordonnées à durée indéterminée convertibles en actions nouvelles et/ou échangeables contre des actions existantes|18.2|65|
|Dividendes reçus||(9)|
|Emission d’obligations subordonnées de dernier rang à durée indéterminée|18.2|(494)|
|Reclassement en dettes financières des titres super subordonnés 2022|18.2|497|
|Nouveaux/modifications contrats de location||1 180|
|Effet du change latent sur la dette de loyer avions enregistrée en résultat global||(409)|
|Effet des dérivés sur la dette nette||99|
|Variation de la conversion en résultat||(175)|
|Autres variations non monétaires de la dette nette||14|
|DETTE NETTE À LA CLÔTURE||7 134|


# NOTE 23 AUTRES PASSIFS

Ce tableau présente la ventilation des autres passifs entre les parts courantes et non courantes au 30 juin 2025 et au 31 décembre 2024. Il détaille les dettes fiscales, sociales, les taxes aériennes, ainsi que les produits constatés d'avance et acomptes.

|(en millions d'euros)|Au 30 juin 2025<br/>Courant|Au 30 juin 2025<br/>Non courant|Au 30 juin 2025<br/>Total|Au 31 décembre 2024<br/>Courant|Au 31 décembre 2024<br/>Non courant|Au 31 décembre 2024<br/>Total|
|-|-|-|-|-|-|-|
|Dettes fiscales (y compris impôt société)|380|299|679|469|413|882|
|Taxes aériennes|1 196|–|1 196|879|–|879|
|Dettes sociales|1 427|192|1 619|1 409|328|1 737|
|Passifs sur immobilisations|44|6|50|47|9|56|
|Produits constatés d’avance|1 150|225|1 375|982|29|1 011|
|Avances et acomptes reçus|536|–|536|576|–|576|
|Dettes diverses|282|85|367|306|125|431|
|TOTAL|5 015|807|5 822|4 668|904|5 572|


Les produits constatés d’avance sont principalement liés aux contrats de l’activité Maintenance.
La variation des autres passifs au 30 juin 2025 s’établit comme suit :

Ce tableau de passage explique l'évolution du total des autres passifs entre le 31 décembre 2024 et le 30 juin 2025, en isolant l'impact de la variation du besoin en fonds de roulement, les écarts de conversion et les autres mouvements.

|(en millions d'euros)|31 décembre 2024|Variation de besoin en fond de roulement|Écart de conversion|Autre|30 juin 2025|
|-|-|-|-|-|-|
|Dettes fiscales (y compris impôt société)|882|(165)|(1)|(38)|679|
|Taxes aériennes|879|317|–|–|1 196|
|Dettes sociales|1 737|(118)|–|–|1 619|
|Passifs sur immobilisations|56|–|–|(6)|50|
|Produits constatés d’avance|1 011|300|(1)|64|1 375|
|Avances et acomptes reçus|576|54|(29)|(66)|536|
|Dettes diverses|431|13|(36)|(39)|367|
|Total|5 572|401|(67)|(85)|5 822|


# NOTE 24 VARIATION DU FONDS DE ROULEMENT

Ce tableau détaille les composantes de la variation du besoin en fonds de roulement pour les périodes de six mois se terminant au 30 juin 2025 et 2024.

|Période du 1ᵉʳ janvier au 30 juin(en millions d'euros)|2025|2024|
|-|-|-|
|(Augmentation)/diminution des stocks|(39)|(62)|
|(Augmentation)/diminution des créances clients|(392)|(325)|
|Augmentation/(diminution) des dettes fournisseurs|(26)|124|
|Augmentation/(Diminution) des billets émis non utilisés|1 496|1 672|
|Augmentation/(Diminution) des miles du programme fidélité|–|(11)|
|Variation des autres actifs|(272)|(294)|
|Variation des autres passifs|401|(658)|
|Variation provision sur émission quotas de CO2|135|46|
|Variation créances/dettes couverture carburant|(4)|(6)|
|Variation du besoin en fonds de roulement|1 297|486|


## NOTE 25 COMMANDE DE MATÉRIELS AÉRONAUTIQUES

Les échéances des engagements de commandes fermes en vue d’achat de matériels aéronautiques s’analysent comme suit :

Ce premier tableau présente l'échéancier financier des engagements de commandes fermes au 30 juin 2025 comparé au 31 décembre 2024. On observe une légère baisse du total des engagements financiers (de 14 397 M€ à 14 211 M€).

|(en millions d'euros)|Au 30 Juin 2025|Au 31 Décembre 2024|
|-|-|-|
|2nd semestre année N (6 mois)|1 397||
|Année N+1|2 161|2 505|
|Année N+2|3 437|2 398|
|Année N+3|2 957|3 682|
|Année N+4|2 660|3 087|
|Au delà de l'année N+4|1 599|2 725|
|TOTAL|14 211|14 397|


Les engagements portent principalement sur des montants en dollar US, convertis en euros au cours de clôture de chaque période considérée. Ces montants font par ailleurs l’objet de couvertures.

Le nombre d’appareils en commande ferme en vue d’achat au 30 juin 2025 augmente de 20 unités par rapport au 31 décembre 2024 et s’élève à 211 appareils. Cette évolution s’explique par la livraison de 10 appareils, et le transfert de 30 options vers des commandes fermes.

### Calendrier de livraison au 30 juin 2025

Ce second tableau détaille le calendrier de livraison physique des 211 appareils en commande, ventilés par type de flotte et modèle d'avion sur les prochaines années.

|Type avionFLOTTE LONG COURRIER – PASSAGE|Année de livraison<br/>2ème Semestre N (6 mois)<br/>FLOTTE LONG COURRIER – PASSAGE|Année de livraison<br/>N+1<br/>FLOTTE LONG COURRIER – PASSAGE|Année de livraison<br/>N+2<br/>FLOTTE LONG COURRIER – PASSAGE|Année de livraison<br/>N+3<br/>FLOTTE LONG COURRIER – PASSAGE|Année de livraison<br/>N+4<br/>FLOTTE LONG COURRIER – PASSAGE|Année de livraison<br/>Au-delà de N+4<br/>FLOTTE LONG COURRIER – PASSAGE|TotalFLOTTE LONG COURRIER – PASSAGE|
|-|-|-|-|-|-|-|-|
|A350|3|7|14|12|12|5|53|
|B787|3|–|–|–|–|–|3|
|FLOTTE LONG COURRIER – CARGO||||||||
|A350F|–|–|1|6|1|–|8|
|FLOTTE MOYEN COURRIER||||||||
|A220|9|9|6|–|–|–|24|
|A320 Neo / A321 Neo|6|15|35|29|22|16|123|
|TOTAL|21|31|56|47|35|21|211|


## NOTE 26 PARTIES LIÉES

Il n’y a pas eu de mouvements significatifs sur les parties liées sur la période tant en termes de périmètre que de montants.

Une prise de participation minoritaire dans le capital de WestJet est envisagée tel que décrit dans la note 2.1 “Événements significatifs intervenus au cours de la période”. Cette transaction reste toutefois soumise à certaines validations et n’a donc pas eu d’impact dans les états financiers consolidés au 30 juin 2025.
# Sommaire

# 3.

Ce tableau présente le sommaire de la section "Information et Contrôle" du rapport financier, détaillant les pages relatives à l'attestation du responsable et au rapport des commissaires aux comptes.

|Section|Description|Page|
|-|-|-|
|3.|INFORMATION ET CONTRÔLE|77|
|3.1|Attestation du Responsable|78|
|3.2|Rapport des commissaires aux comptes sur l'information financière semestrielle 2025|78|


# 3.1 <u>Attestation du Responsable</u>

Le 31 juillet 2025,

J’atteste, à ma connaissance, que les comptes résumés pour le semestre écoulé sont établis conformément aux normes comptables applicables et donnent une image fidèle du patrimoine, de la situation financière et du résultat de la société et de l’ensemble des entreprises comprises dans la consolidation, et que le rapport semestriel d’activité ci-joint présente un tableau fidèle des événements importants survenus pendant les six premiers mois de l’exercice, de leur incidence sur les comptes, des principales transactions entre parties liées et qu’il décrit les principaux risques et les principales incertitudes pour les six mois restants de l’exercice.

**Benjamin Smith**
Directeur général Air France – KLM

# 3.2 <u>Rapport des commissaires aux comptes sur l'information financière semestrielle 2025</u>

**Air France-KLM S.A.**
7, rue du Cirque, 75008 Paris
**Rapport des commissaires aux comptes sur l’information financière semestrielle 2025**
Période du 1<sup>er</sup> janvier 2025 au 30 juin 2025

Mesdames, Messieurs les Actionnaires,

En exécution de la mission qui nous a été confiée par votre Assemblée générale et en application de l'article L.451-1-2 III du Code monétaire et financier, nous avons procédé à :

*   a l'examen limité des états financiers consolidés semestriels résumés de la société Air France-KLM S.A., relatifs à la période du 1<sup>er</sup> janvier 2025 au 30 juin 2025, tels qu'ils sont joints au présent rapport ;
*   b la vérification des informations données dans le rapport semestriel d'activité.

Ces états financiers consolidés semestriels résumés ont été établis sous la responsabilité du Conseil d’administration. Il nous appartient, sur la base de notre examen limité, d'exprimer notre conclusion sur ces comptes.

### I - Conclusion sur les comptes

Nous avons effectué notre examen limité selon les normes d'exercice professionnel applicables en France.

Un examen limité consiste essentiellement à s'entretenir avec les membres de la direction en charge des aspects comptables et financiers et à mettre en œuvre des procédures analytiques. Ces travaux sont moins étendus que ceux requis pour un audit effectué selon les normes d'exercice professionnel applicables en France. En conséquence, l’assurance que les comptes, pris dans leur ensemble, ne comportent pas d’anomalies significatives obtenue dans le cadre d’un examen limité est une assurance modérée, moins élevée que celle obtenue dans le cadre d’un audit.

Sur la base de notre examen limité, nous n'avons pas relevé d'anomalies significatives de nature à remettre en cause la conformité des états financiers consolidés semestriels résumés avec la norme IAS 34 - norme du référentiel IFRS tel qu'adopté dans l'Union européenne relative à l'information financière intermédiaire.

### II - Vérification spécifique

Nous avons également procédé à la vérification des informations données dans le rapport semestriel d'activité commentant les états financiers consolidés semestriels résumés sur lesquels a porté notre examen limité.
Nous n'avons pas d'observation à formuler sur leur sincérité et leur concordance avec les états financiers consolidés semestriels résumés.

Paris La Défense et Neuilly sur Seine, le 30 juillet 2025

**Les commissaires aux comptes**

|KPMG S.A.|PricewaterhouseCoopers Audit S.A.S.|PricewaterhouseCoopers Audit S.A.S.|PricewaterhouseCoopers Audit S.A.S.|
|-|-|-|-|
|\[Signatures manuscrites]|\[Signatures manuscrites]|\[Signatures manuscrites]|\[Signatures manuscrites]|
|Valérie Besson|Eric Dupré|Philippe Vincent|Amélie Jeudi de Grissac|
|Associée|Associé|Associé|Associée|


L'image fournie est une page de couverture ou une page de transition d'un document institutionnel. Elle présente un design graphique épuré avec un dégradé de bleu allant du bleu foncé au bleu clair, occupant la majeure partie de la page selon une forme géométrique biseautée.

En bas à droite, on retrouve les éléments d'identification suivants :

*   Une icône stylisée représentant un écran d'ordinateur avec une flèche pointant vers le bas.
*   L'adresse du site internet : airfranceklm.com
*   Le logo officiel du groupe : **AIRFRANCEKLM GROUP**


In [20]:
# text_content = langchain_docs[0].page_content
# Markdown(text_content)

In [38]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False # On garde le titre dans le texte !
)

md_chunks = markdown_splitter.split_text(langchain_docs[0].page_content)

In [39]:
md_chunks

[Document(metadata={'Header 1': 'RAPPORT FINANCIER SEMESTRIEL 2025'}, page_content='# RAPPORT FINANCIER SEMESTRIEL 2025  \n----  \n**GROUPE AIR FRANCE-KLM**'),
 Document(metadata={'Header 1': 'Sommaire'}, page_content="# Sommaire  \nCe tableau présente la structure du rapport financier semestriel d'Air France-KLM au 30 juin 2025, divisé en trois sections principales : l'activité semestrielle, les états financiers et les informations de contrôle.  \n|Section|Page|\n|-|-|\n|Rapport semestriel d’activité|3|\n|Rapport financier|34|\n|Information et contrôle|77|  \n# Sommaire"),
 Document(metadata={'Header 1': '1.'}, page_content='# 1.  \n----  \nCe tableau présente le sommaire détaillé de la première partie du rapport financier, intitulée "Rapport semestriel d\'activité", incluant les sections sur l\'activité, la flotte, les faits marquants et le gouvernement d\'entreprise, avec leurs numéros de page respectifs.  \n|Section|Libellé|Page|\n|-|-|-|\n|1.|RAPPORT SEMESTRIEL D\'ACTIVITÉ|3|\n|1.

In [49]:
len(md_chunks)

163

In [28]:
import os
import pickle
from pathlib import Path

RAW_MARKDOWN_FILE = "airfrance_brut.md"

full_markdown_text = ""
for idx, page in enumerate(parsed_pages):
    full_markdown_text += f"\n\n\n\n"
    full_markdown_text += page.text

with open(RAW_MARKDOWN_FILE, "w", encoding="utf-8") as f:
        f.write(full_markdown_text)            


In [ ]:
RAW_MARKDOWN_FILE = "C:/Users/user/Desktop/LLM-RAG/src/processing/airfrance_brut.md"

from langchain_community.document_loaders import TextLoader
if Path(RAW_MARKDOWN_FILE).exists():
    loader = TextLoader(RAW_MARKDOWN_FILE, encoding="utf-8")
    documents = loader.load()


In [55]:
documents

[Document(metadata={'source': 'C:/Users/user/Desktop/LLM-RAG/src/processing/airfrance_brut.md'}, page_content='\n\n\n\n# RAPPORT FINANCIER SEMESTRIEL 2025\n\n----\n\n**GROUPE AIR FRANCE-KLM**\n# Sommaire\n\nCe tableau présente la structure du rapport financier semestriel d\'Air France-KLM au 30 juin 2025, divisé en trois sections principales : l\'activité semestrielle, les états financiers et les informations de contrôle.\n\n|Section|Page|\n|-|-|\n|Rapport semestriel d’activité|3|\n|Rapport financier|34|\n|Information et contrôle|77|\n\n\n# Sommaire\n\n# 1.\n\n----\n\nCe tableau présente le sommaire détaillé de la première partie du rapport financier, intitulée "Rapport semestriel d\'activité", incluant les sections sur l\'activité, la flotte, les faits marquants et le gouvernement d\'entreprise, avec leurs numéros de page respectifs.\n\n|Section|Libellé|Page|\n|-|-|-|\n|1.|RAPPORT SEMESTRIEL D\'ACTIVITÉ|3|\n|1.1|ACTIVITÉ|4|\n|1.1.1|Stratégie|4|\n|1.1.2|Les activités|7|\n|1.1.3|La flot

## Unstructured

In [3]:
from unstructured.partition.pdf import partition_pdf

In [4]:
elements = partition_pdf(
    filename=file_path,
    infer_table_structure=True,           # Active la reconnaissance de structure
    strategy="hi_res",                    # Utilise le modèle de vision (plus lent mais précis)
    languages=["fra"],
    chunking_strategy=None,               # Regroupe le texte par section
)

The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


In [5]:
for el in elements:
    if el.category == "Table":
        print("=" * 80)
        print(el)

Rapport semestriel d’activité Rapport financier Information et contrôle 3 34 77
RAPPORT SEMESTRIEL D'ACTIVITÉ 3 1.1 ACTIVITÉ 4 1.1.1 Stratégie 4 1.1.2 Les activités 7 1.1.3 La flotte 11 1.1.4 Faits marquants 13 1.1.5 Perspectives et évènements post clôture 23 1.1.6 Facteurs de risques 26 1.1.7 Parties liées 28 1.2 GOUVERNEMENT D'ENTREPRISE 29 1.2.1 Conseil d'administration 29 1.2.2 CEO Committee 31 1.2.3 Comité exécutif Groupe 31
Croissance du PIB réel (%) 2024 Monde 3,3 % Zone euro 0,9 % Dont France 1,1 % Dont Pays-Bas 1,0 % Amérique latine et Caraïbes 2,4 % Afrique subsaharienne 4,0 % États-Unis 2,8 % Chine 5,0 % Japon 0,1 % 2025 2,8 % 0,8 % 0,6 % 1,4 % 2,0 % 3,8 % 1,8 % 4,0 % 0,6 %
baril) T1 2024 T2 2024 T3 2024 T4 2024 T1 2025 T2 2025 Cours moyen du Brent 83,0 84,6 79,8 75,6 75,8 68,0 Cours moyen du Kérosène 110,1 103,3 92,1 87,2 93,5 84,0
Pour un euro T1 2024 T2 2024 T3 2024 T4 2024 T1 2025 T2 2025 (moyenne) USD 1,09 1,08 1,10 1,07 1,05 1,13 GBP 0,86 0,85 0,85 0,83 0,84 0,85 CHF 0

In [6]:
for el in elements:
    if el.category == "Table":
        print(el.metadata.text_as_html)

<table><tbody><tr><td></td><td>semestriel</td><td>d'activité</td><td></td></tr><tr><td>Rapport</td><td></td><td></td><td></td></tr><tr><td>Rapport</td><td>financier</td><td></td><td>54</td></tr><tr><td></td><td></td><td></td><td></td></tr><tr><td>Information</td><td>et contrôle</td><td></td><td>77</td></tr></tbody></table>
<table><tbody><tr><td>1.1</td><td>RAPPORT SEMESTRIEL D'ACTIVITÉ ACTIVITÉ</td></tr><tr><td>1.1.1</td><td>Stratégie</td></tr><tr><td>1.1.2</td><td>Les activités</td></tr><tr><td>1.1.3</td><td>La flotte</td></tr><tr><td>1.1.4</td><td>Faits marquants</td></tr><tr><td>1.1.5</td><td>Perspectives et évènements post clôture</td></tr><tr><td>1.1.6</td><td>Facteurs de risques</td></tr><tr><td>1.1.7</td><td>Parties liées</td></tr><tr><td>1.2</td><td>GOUVERNEMENT D'ENTREPRISE</td></tr><tr><td>1.2.1</td><td>Conseil d'administration</td></tr><tr><td>1.2.2</td><td>CEO Committee</td></tr><tr><td>1.2.3</td><td>Comité exécutif Groupe</td></tr></tbody></table>
<table><thead><tr><th>Mon

In [ ]:
# data_to_store_bis = []

# for el in elements:
#     # On ne garde que les tableaux et les titres pour garder le contexte
#     if el.category in ["Text", "NarrativeText", "ListItem", "Header", "Footer", "FigureCaption"]:
#         data_to_store_bis.append({
#             "type": el.category,
#             "content": el.text,
#             "page": el.metadata.page_number
#         })

In [ ]:
# from langchain_core.documents import Document

# max_pages = elements[-1].metadata.page_number
# documents = []
# current_doc = None

# for el in elements:
#     if el.category in ["Header", "Footer"]:
#         continue

#     if el.category == "Title":
#         if not current_doc:
#             current_doc = Document(page_content="", metadata=el.metadata.to_dict())
        
#     content = el.metadata.text_as_html if el.category == "Table" else el.text
#     current_doc.page_content += content + "\n\n"

#     if el.category == "Table":
#         documents.append(current_doc)
#         current_doc = None        

# if current_doc:
#     documents.append(current_doc)


In [ ]:
data_to_store = []

for el in elements:
    # On ne garde que les tableaux et les titres pour garder le contexte
    if el.category in ["Table", "Title"]:
        data_to_store.append({
            "type": el.category,
            "content": el.metadata.text_as_html if el.category == "Table" else el.text,
            # "text_table": el.text if el.category == "Table" else "",
            "page": el.metadata.page_number
        })

In [8]:
# for el in elements:
#     if el.category == "Table":
#         print(el.metadata.text_as_html)

## pdfplumber

In [ ]:
import pdfplumber

tableaux = []

with pdfplumber.open(file_path) as pdf:
    for i in range
    page = pdf.pages[5]  # page 5 (0-indexed)
    tables = page.extract_tables()
    # tables[0] → liste de listes avec toutes les cellules

In [49]:
tables

[[['1.1.2', 'Les activités']],
 [['Deuxième Trimestre', None, None, 'Premier Semestre'],
  ['variation\n2025 variation change\nconstant', None, None, None],
  ['6 671', '+4,8%', '', None],
  ['6 197\n473', '+5,0 %\n+2,5 %', '', None],
  ['6 937', '+4,6 %', None, None],
  ['-1 738\n-1 395', '+3,9 %\n-12,3 %', '', None],
  ['-2 628', '+8,5 %', None, None],
  ['-510', '+1,8 %', '', None],
  ['666', '+221 +190', None, None],
  ['9,6 %', '+2,9 pt', '', None]],
 [['Le réseau Passage affiche une solide performance au deuxième trimestre, portée par le'],
  ['dynamisme des cabines Premium et la progression du yield']],
 [['Deuxième Trimestre', None, None, 'Premier Semestre'],
  ['2025', 'variation\nvariation change\nconstant', None, None],
  ['19 752\n70 511\n61 621', '+3,4%\n+2,8%\n+2,9%', None, None],
  ['87,4%', '+0,0pt', '', None],
  ['6 362\n6 197', '+4,6 % +5,2 %\n+5,0 % +5,7 %', None, None],
  ['8,79', '+2,1 % +2,8 %', None, None]]]

## camelot

In [9]:
import camelot

tables = camelot.read_pdf(file_path, pages="all", flavor="stream")

print("tables:", tables.n)

for i, table in enumerate(tables):
    print(f"--- Table {i} ---")
    print(table.df)
    print()

tables: 108
--- Table 0 ---
                       0
0                RAPPORT
1              FINANCIER
2             SEMESTRIEL
3                   2025
4  GROUPE AIR FRANCE-KLM

--- Table 1 ---
   0                                                  1   2
0                         Rapport semestriel d’activité   3
1                                     Rapport financier  34
2                               Information et contrôle  77
3                                                          
4  2  AIR FRANCE - KLM – RAPPORT FINANCIER SEMESTRIE...    

--- Table 2 ---
                                0                                        1   2
0   RAPPORT SEMESTRIEL D'ACTIVITÉ                                            3
1                             1.1                                 ACTIVITÉ   4
2                           1.1.1                                Stratégie   4
3                           1.1.2                            Les activités   7
4                           1.1.3

In [59]:
# tables[36].df

In [36]:
import json

with open("air_france_data_bis.json", "w", encoding="utf-8") as f:
    json.dump(data_to_store_bis, f, ensure_ascii=False, indent=4)

80

In [34]:
import json

with open("air_france_data.json", "w", encoding="utf-8") as f:
    json.dump(data_to_store, f, ensure_ascii=False, indent=4)

In [ ]:
from langchain_community.document_loaders import UnstructuredFileLoader

In [ ]:
# Add the environment variable for the GCP service account credentials 
# os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '/Path/to/your/keyfile.json'

command = [ "unstructured-ingest", "gcs", "--remote-url", "gs://<YOUR-BUCKET>/", 
           "--structured-output-dir", "/YOUR/OUTPUT/PATH", "--num-processes", "2", 
           "--api-key", "<UNSTRUCTURED-API-KEY>", "--partition-by-api"]

## Docling